In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Google Drive mount skipped: {e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install dependencies in a Python-syntax-safe way.
# In Colab/Jupyter this is equivalent to running pip install, but it also passes normal Python syntax checks.
import sys
import subprocess

# Dependencies are managed by requirements.txt for local and Codex Cloud runs.
# Avoid network-dependent package installation every time this notebook executes.


In [ ]:
# ============================================
# Step 1: Load & Normalize OHLCV Data — V3 robust MT5 import
# ============================================

import os
import pandas as pd


def load_mt5_csv(filepath, *, timeframe=None, verbose=True):
    """
    Robust loader for MT5-exported OHLCV CSV files.

    Supports common export quirks, including the format in the uploaded sample:
    - Header: comma-separated  -> Time,Open,High,Low,Close,Volume
    - Rows:   tab-separated    -> 2023.01.03 01:00\t1826.63\t...

    Output:
    - DatetimeIndex named Time
    - lowercase columns: open, high, low, close, volume
    - numeric OHLCV columns
    """
    if filepath is None or not os.path.exists(filepath):
        raise FileNotFoundError(f"MT5 CSV not found: {filepath}")

    # Regex separator handles comma, tab, semicolon, and pipe in one loader.
    # This is necessary for mixed delimiter files where header uses comma but rows use tab.
    raw = pd.read_csv(filepath, sep=r"[\t,;|]", engine="python")
    raw.columns = [str(c).strip().replace("<", "").replace(">", "") for c in raw.columns]

    # If pandas produced one collapsed column, retry with no header and manual split fallback.
    if len(raw.columns) == 1:
        raw = pd.read_csv(filepath, sep=r"[\t,;|]", engine="python", header=None)
        if raw.shape[1] >= 6:
            raw = raw.iloc[:, :6]
            raw.columns = ["Time", "Open", "High", "Low", "Close", "Volume"]
        else:
            raise ValueError(f"Cannot parse MT5 file into 6 OHLCV columns: {filepath}")

    colmap = {str(c).strip().lower(): c for c in raw.columns}
    aliases = {
        "time": ["time", "date", "datetime", "timestamp"],
        "open": ["open", "o"],
        "high": ["high", "h"],
        "low": ["low", "l"],
        "close": ["close", "c"],
        "volume": ["volume", "tickvol", "tick_volume", "vol", "real_volume"],
    }

    selected = {}
    for target, names in aliases.items():
        found = None
        for name in names:
            if name in colmap:
                found = colmap[name]
                break
        if found is not None:
            selected[target] = found

    required = ["time", "open", "high", "low", "close"]
    missing = [c for c in required if c not in selected]
    if missing:
        raise ValueError(f"Missing required columns {missing}. Actual columns: {list(raw.columns)}")

    df = raw[[selected[c] for c in selected]].copy()
    df.columns = list(selected.keys())
    if "volume" not in df.columns:
        df["volume"] = 0

    df["time"] = pd.to_datetime(df["time"], format="%Y.%m.%d %H:%M", errors="coerce")
    if df["time"].isna().any():
        df["time"] = pd.to_datetime(df["time"], errors="coerce")

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["time", "open", "high", "low", "close"]).copy()
    df = df.drop_duplicates(subset=["time"]).sort_values("time")
    df = df.set_index("time")
    df.index.name = "Time"

    # Basic sanity checks
    bad_hl = (df["high"] < df[["open", "close", "low"]].max(axis=1)) | (df["low"] > df[["open", "close", "high"]].min(axis=1))
    if bad_hl.any() and verbose:
        print(f"⚠️ {os.path.basename(filepath)}: {int(bad_hl.sum())} rows have unusual OHLC high/low relationships.")

    if verbose:
        tf_msg = f" [{timeframe}]" if timeframe else ""
        print(f"✅ Loaded{tf_msg}: {os.path.basename(filepath)} | rows={len(df):,} | from={df.index.min()} | to={df.index.max()}")
        print("   columns:", list(df.columns))
    return df


# Default file paths. Override by defining DATA_PATHS before running this cell.
DATA_PATHS = globals().get("DATA_PATHS", {
    "M15": "/content/drive/MyDrive/EA_XAUUSD/XAUUSD_M15.csv",
    "M30": "/content/drive/MyDrive/EA_XAUUSD/XAUUSD_M30.csv",
    "H1":  "/content/drive/MyDrive/EA_XAUUSD/XAUUSD_H1.csv",
    "H4":  "/content/drive/MyDrive/EA_XAUUSD/XAUUSD_H4.csv",
})

# Local fallback for this ChatGPT/session or when testing outside Colab.
LOCAL_FALLBACK_PATHS = {
    "M15": os.path.abspath("XAUUSD_M15.csv"),
    "M30": os.path.abspath("XAUUSD_M30.csv"),
    "H1":  os.path.abspath("XAUUSD_H1.csv"),
    "H4":  os.path.abspath("XAUUSD_H4.csv"),
}

for _tf, _fallback in LOCAL_FALLBACK_PATHS.items():
    if (not DATA_PATHS.get(_tf) or not os.path.exists(DATA_PATHS.get(_tf, ""))) and os.path.exists(_fallback):
        DATA_PATHS[_tf] = _fallback

DF_M15 = DF_M30 = DF_H1 = DF_H4 = None
for _tf, _path in DATA_PATHS.items():
    if _path and os.path.exists(_path):
        globals()[f"DF_{_tf}"] = load_mt5_csv(_path, timeframe=_tf)
    else:
        print(f"⚠️ Skip {_tf}: file not found -> {_path}")

print("\n📌 Available dataframes:", [k for k in ["DF_M15", "DF_M30", "DF_H1", "DF_H4"] if globals().get(k) is not None])
if globals().get("DF_M30") is not None:
    print("\nDF_M30 Preview:")
    display(DF_M30.head())


✅ Loaded [M15]: XAUUSD_M15.csv | rows=60,389 | from=2023-01-03 01:00:00 | to=2025-07-23 17:00:00
   columns: ['open', 'high', 'low', 'close', 'volume']
✅ Loaded [M30]: XAUUSD_M30.csv | rows=30,197 | from=2023-01-03 01:00:00 | to=2025-07-23 17:00:00
   columns: ['open', 'high', 'low', 'close', 'volume']
✅ Loaded [H1]: XAUUSD_H1.csv | rows=15,109 | from=2023-01-03 01:00:00 | to=2025-07-23 17:00:00
   columns: ['open', 'high', 'low', 'close', 'volume']
✅ Loaded [H4]: XAUUSD_H4.csv | rows=3,954 | from=2023-01-03 00:00:00 | to=2025-07-23 16:00:00
   columns: ['open', 'high', 'low', 'close', 'volume']

📌 Available dataframes: ['DF_M15', 'DF_M30', 'DF_H1', 'DF_H4']

DF_M30 Preview:


,open,high,low,close,volume
Time,,,,,
2023-01-03 01:00:00,1826.63,1830.00,1823.58,1827.73,1375
2023-01-03 01:30:00,1827.73,1829.23,1826.83,1829.20,1004
2023-01-03 02:00:00,1829.20,1832.08,1828.55,1831.15,1698
2023-01-03 02:30:00,1831.26,1831.91,1830.10,1830.50,552
2023-01-03 03:00:00,1830.50,1832.28,1826.65,1827.95,2506


In [ ]:
# ============================================
# Step 2: Define Typical and Optimization Parameters (with merge and EMA 200)
# ============================================

import numpy as np
import itertools

# ✅ Typical Parameters (used for fast indicator precomputation)
typical_params = {
    'EMA': {'fast': 12, 'slow': 26},
    'RSI': {'period': 14, 'oversold': 30, 'overbought': 70},
    'MACD': {'fast': 12, 'slow': 26, 'signal': 9},
    'BollingerBands': {'period': 20, 'stddev': 2.0},
    'ATR': {'period': 14, 'multiplier': 2.0},
    'Stochastic': {'k': 14, 'd': 3, 'oversold': 20, 'overbought': 80},
    'Fibonacci': {'levels': [23.6, 38.2, 50, 61.8, 78.6]}  # These are not optimized
}

# ✅ Optimization Parameter Ranges
indicator_params = {
    'EMA': {
        'fast': list(range(5, 26, 5)),       # 5 to 25 step 5
        'slow': list(range(30, 201, 20))     # 30 to 200 step 20
    },
    'RSI': {
        'period': list(range(7, 22, 2)),               # 7 to 21 step 2
        'oversold': list(range(20, 36, 5)),            # 20 to 35 step 5
        'overbought': list(range(65, 81, 5))           # 65 to 80 step 5
    },
    'MACD': {
        'fast': list(range(8, 17)),                    # 8 to 16
        'slow': list(range(20, 31)),                   # 20 to 30
        'signal': list(range(6, 13))                   # 6 to 12
    },
    'BollingerBands': {
        'period': list(range(10, 31, 5)),              # 10 to 30 step 5
        'stddev': [1.5, 2.0, 2.5, 3.0]
    },
    'ATR': {
        'period': list(range(10, 22, 2)),              # 10 to 21 step 2
        'multiplier': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
    },
    'Stochastic': {
        'k': list(range(10, 22, 2)),                   # 10 to 21 step 2
        'd': [3, 4, 5],
        'oversold': list(range(15, 26, 5)),            # 15 to 25
        'overbought': list(range(75, 86, 5))           # 75 to 85
    },
    'Fibonacci': {
        'levels': [23.6, 38.2, 50, 61.8, 78.6]         # Used in fixed rule-based logic
    }
}

# ✅ Merge typical_params with indicator_params for precomputation
combined_params = {}

for ind, opt_params in indicator_params.items():
    combined_params[ind] = {}
    for key, values in opt_params.items():
        merged = set(values)
        if ind in typical_params and key in typical_params[ind]:
            val = typical_params[ind][key]
            if isinstance(val, (list, tuple)):
                merged.update(val)
            else:
                merged.add(val)
        combined_params[ind][key] = sorted(merged)

# ✅ Explicitly ensure EMA 200 is present in slow list
if 'EMA' in combined_params and 'slow' in combined_params['EMA']:
    if 200 not in combined_params['EMA']['slow']:
        combined_params['EMA']['slow'].append(200)
        combined_params['EMA']['slow'] = sorted(combined_params['EMA']['slow'])

print("✅ Combined parameter ranges prepared:")
for k, v in combined_params.items():
    print(f"{k}: {v}")


✅ Combined parameter ranges prepared:
EMA: {'fast': [5, 10, 12, 15, 20, 25], 'slow': [26, 30, 50, 70, 90, 110, 130, 150, 170, 190, 200]}
RSI: {'period': [7, 9, 11, 13, 14, 15, 17, 19, 21], 'oversold': [20, 25, 30, 35], 'overbought': [65, 70, 75, 80]}
MACD: {'fast': [8, 9, 10, 11, 12, 13, 14, 15, 16], 'slow': [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30], 'signal': [6, 7, 8, 9, 10, 11, 12]}
BollingerBands: {'period': [10, 15, 20, 25, 30], 'stddev': [1.5, 2.0, 2.5, 3.0]}
ATR: {'period': [10, 12, 14, 16, 18, 20], 'multiplier': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]}
Stochastic: {'k': [10, 12, 14, 16, 18, 20], 'd': [3, 4, 5], 'oversold': [15, 20, 25], 'overbought': [75, 80, 85]}
Fibonacci: {'levels': [23.6, 38.2, 50, 61.8, 78.6]}


In [ ]:
# ============================================
# Step 3: Precompute Indicators (Dynamic from combined_params)
# ============================================

def precompute_indicators(df, combined_params, typical_params=None):
    """
    Precompute a wide range of indicator values based on combined_params
    (merged typical_params and indicator_params) so that any parameter
    combination defined in Step 2 can be used without KeyErrors.
    """
    df = df.copy()

    # === EMA: use both fast & slow sets from combined_params
    ema_spans = sorted(set(
        combined_params.get('EMA', {}).get('fast', []) +
        combined_params.get('EMA', {}).get('slow', [])
    ))
    for span in ema_spans:
        if span > 0:  # safety check
            df[f"ema_{span}"] = df["close"].ewm(span=span, adjust=False).mean()

    # === RSI: use all periods in combined_params
    rsi_periods = combined_params.get('RSI', {}).get('period', [])
    for period in rsi_periods:
        if period > 0:
            delta = df["close"].diff()
            gain = delta.where(delta > 0, 0)
            loss = -delta.where(delta < 0, 0)
            avg_gain = gain.rolling(window=period).mean()
            avg_loss = loss.rolling(window=period).mean()
            rs = avg_gain / (avg_loss + 1e-6)
            df[f"rsi_{period}"] = 100 - (100 / (1 + rs))

    # === MACD: still use a default triple
    if typical_params is not None:
        macd_fast = typical_params['MACD']['fast']
        macd_slow = typical_params['MACD']['slow']
        macd_signal = typical_params['MACD']['signal']
    else:
        # fallback to standard MACD values
        macd_fast, macd_slow, macd_signal = 12, 26, 9

    ema_fast_series = df["close"].ewm(span=macd_fast, adjust=False).mean()
    ema_slow_series = df["close"].ewm(span=macd_slow, adjust=False).mean()
    macd_line = ema_fast_series - ema_slow_series
    macd_signal_line = macd_line.ewm(span=macd_signal, adjust=False).mean()
    df["macd_histogram"] = macd_line - macd_signal_line

    # === Bollinger Bands: use all periods
    bb_periods = combined_params.get('BollingerBands', {}).get('period', [])
    for period in bb_periods:
        if period > 1:  # window size must be >1
            mean = df["close"].rolling(window=period).mean()
            std = df["close"].rolling(window=period).std()
            df[f"bb_upper_{period}"] = mean + (2 * std)
            df[f"bb_lower_{period}"] = mean - (2 * std)
    # default 20 for backward compatibility
    df["bb_upper"] = df["close"].rolling(window=20).mean() + 2 * df["close"].rolling(window=20).std()
    df["bb_lower"] = df["close"].rolling(window=20).mean() - 2 * df["close"].rolling(window=20).std()

    # === ATR: use all periods
    atr_periods = combined_params.get('ATR', {}).get('period', [])
    high = df["high"]
    low = df["low"]
    close = df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    for period in atr_periods:
        if period > 0:
            df[f"atr_{period}"] = tr.rolling(window=period).mean()

    # === Stochastic: use all k and d
    stoch_ks = combined_params.get('Stochastic', {}).get('k', [])
    stoch_ds = combined_params.get('Stochastic', {}).get('d', [])
    for k in stoch_ks:
        if k > 0:
            lowest_low = df["low"].rolling(window=k).min()
            highest_high = df["high"].rolling(window=k).max()
            k_val = 100 * ((df["close"] - lowest_low) / (highest_high - lowest_low + 1e-6))
            df[f"stoch_k_{k}"] = k_val
            for d in stoch_ds:
                if d > 0:
                    df[f"stoch_d_{d}"] = k_val.rolling(window=d).mean()

    # === Fibonacci: support/resistance as before
    df["fib_support"] = df["low"].rolling(window=50).min()
    df["fib_resistance"] = df["high"].rolling(window=50).max()

    return df


In [ ]:
# ============================================
# Step 4: Entry Logic (Long/Short Functions)
# ============================================

import pandas as pd

# === LONG Entry Conditions ===

def ema_entry(df, params):
    return (df[f"ema_{params['fast']}"] > df[f"ema_{params['slow']}"]) & \
           (df[f"ema_{params['fast']}"] > df[f"ema_{params['fast']}"].shift(1))

def rsi_entry(df, params):
    return df[f"rsi_{params['period']}"] < params['oversold']

def macd_entry(df, params):
    return (df['macd_histogram'] > 0) & (df['macd_histogram'] > df['macd_histogram'].shift(1))

def bb_entry(df, params):
    return (df['close'].shift(1) < df['bb_lower']) & (df['close'] > df['bb_lower'])

def fib_entry(df, params=None):
    return df['close'] > df['fib_support']  # assumes fib_support is precomputed

def stoch_entry(df, params):
    return (df[f"stoch_k_{params['k']}"] > df[f"stoch_d_{params['d']}"]) & \
           (df[f"stoch_k_{params['k']}"] < params.get('oversold', 30))


# === SHORT Entry Conditions ===

def ema_short_entry(df, params):
    return (df[f"ema_{params['fast']}"] < df[f"ema_{params['slow']}"]) & \
           (df[f"ema_{params['fast']}"] < df[f"ema_{params['fast']}"].shift(1))

def rsi_short_entry(df, params):
    return df[f"rsi_{params['period']}"] > params['overbought']

def macd_short_entry(df, params):
    return (df['macd_histogram'] < 0) & (df['macd_histogram'] < df['macd_histogram'].shift(1))

def bb_short_entry(df, params):
    return (df['close'].shift(1) > df['bb_upper']) & (df['close'] < df['bb_upper'])

def fib_short_entry(df, params=None):
    return df['close'] < df['fib_resistance']

def stoch_short_entry(df, params):
    return (df[f"stoch_k_{params['k']}"] < df[f"stoch_d_{params['d']}"]) & \
           (df[f"stoch_k_{params['k']}"] > params.get('overbought', 70))


# === Function Dictionaries ===

entry_functions = {
    'EMA': ema_entry,
    'RSI': rsi_entry,
    'MACD': macd_entry,
    'BollingerBands': bb_entry,
    'Fibonacci': fib_entry,
    'Stochastic': stoch_entry
}

short_entry_functions = {
    'EMA': ema_short_entry,
    'RSI': rsi_short_entry,
    'MACD': macd_short_entry,
    'BollingerBands': bb_short_entry,
    'Fibonacci': fib_short_entry,
    'Stochastic': stoch_short_entry
}


# === Composite Entry Condition Builder ===

def generate_entry_condition(indicators, params, direction='long', mode='AND'):
    """
    Build a composite long/short entry condition.
    - indicators: list of indicators (e.g., ['EMA', 'RSI'])
    - params: dict of indicator-specific params
    - direction: 'long' or 'short'
    - mode: 'AND' or 'OR'
    Returns: function that takes df and returns signal series
    """
    funcs = entry_functions if direction == 'long' else short_entry_functions
    mode = mode.upper()

    def combined_entry(df):
        conditions = []
        for ind in indicators:
            func = funcs.get(ind)
            if func:
                cond = func(df, params.get(ind, {}))
                conditions.append(cond)

        if not conditions:
            return pd.Series([False] * len(df), index=df.index)

        return pd.concat(conditions, axis=1).all(axis=1) if mode == 'AND' else pd.concat(conditions, axis=1).any(axis=1)

    return combined_entry


In [ ]:
# ============================================
# Step 5: Exit Logic Functions (Final)
# ============================================

import numpy as np

# === SL/TP Generators ===

def calculate_atr_stop(entry_price, atr, atr_multiplier, direction='long'):
    """Calculate stop loss using ATR."""
    sl_distance = atr * atr_multiplier
    return entry_price - sl_distance if direction == 'long' else entry_price + sl_distance

def calculate_tp_by_rr(entry_price, sl_price, rr_ratio, direction='long'):
    """Calculate take profit using Risk:Reward ratio."""
    sl_distance = abs(entry_price - sl_price)
    return entry_price + sl_distance * rr_ratio if direction == 'long' else entry_price - sl_distance * rr_ratio

def calculate_structure_stop(df, idx, direction='long', window=20):
    """Calculate stop loss using swing low/high (structure)."""
    if direction == 'long':
        return df['low'].iloc[max(0, idx-window):idx].min()
    else:
        return df['high'].iloc[max(0, idx-window):idx].max()

def calculate_fib_target(entry_price, fib_levels, direction='long'):
    """Calculate potential targets using Fibonacci extension levels."""
    multipliers = {161.8: 1.618, 200.0: 2.0, 261.8: 2.618}
    return [
        entry_price * (1 + m/100) if direction == 'long' else entry_price * (1 - m/100)
        for m in multipliers.values()
        if m in fib_levels or True
    ]

def generate_exit_levels(entry_price, atr, df, idx, sl_type, tp_type, config, direction='long'):
    """Return both stop-loss and take-profit based on configuration."""
    if sl_type == 'atr':
        sl_price = calculate_atr_stop(entry_price, atr, config.get('atr_multiplier', 2.0), direction)
    elif sl_type == 'structure':
        sl_price = calculate_structure_stop(df, idx, direction)
    else:
        sl_price = None

    if tp_type == 'rr':
        rr_ratio = config.get('risk_reward_ratio', 2.0)
        tp_price = calculate_tp_by_rr(entry_price, sl_price, rr_ratio, direction)
    elif tp_type == 'fib':
        fib_levels = config.get('fib_levels', [161.8, 200.0])
        tp_price = calculate_fib_target(entry_price, fib_levels, direction)[-1]
    else:
        tp_price = None

    return sl_price, tp_price

# === Trailing Stop ===

def calculate_trailing_stop(current_price, atr, trail_type, params, direction='long'):
    """Return trailing stop level based on type."""
    if not params.get('trailing_enabled', False):
        return None  # disabled

    if trail_type == 'atr':
        multiplier = params.get('trail_multiplier', 1.5)
        distance = atr * multiplier
    elif trail_type == 'percent':
        percent = params.get('trail_percent', 1.0) / 100.0
        distance = current_price * percent
    elif trail_type == 'step':
        distance = params.get('trail_step_pips', 20) * 0.1  # convert to price
    else:
        return None

    return current_price - distance if direction == 'long' else current_price + distance

# === Time-Based Exit ===

def check_max_holding(entry_idx, current_idx, max_bars):
    """Return True if position exceeded maximum holding bars."""
    return (current_idx - entry_idx) >= max_bars

# === Indicator Reversal Exit ===

def indicator_reversal(entry_condition_fn):
    """Return opposite signal of entry function as exit trigger."""
    def exit_condition(df):
        return ~entry_condition_fn(df)
    return exit_condition

# === Max Drawdown Stop (strategy-level) ===

def check_drawdown_breach(equity_curve, threshold_pct=0.20):
    """Return True if drawdown exceeds threshold."""
    peak = np.maximum.accumulate(equity_curve)
    drawdown = (peak - equity_curve) / (peak + 1e-9)
    return (drawdown > threshold_pct).any()


In [ ]:
# ============================================
# Step 6: Position Sizing Methods (Final)
# ============================================

def calculate_position_size(
    capital: float,
    entry_price: float,
    stop_loss_price: float,
    sizing_method: str = 'fixed',
    config: dict = None,
    **kwargs
) -> float:
    """
    Calculate lot size based on strategy sizing method.
    Supports:
      • fixed lot size
      • risk_percent (risk % of capital based on SL distance)
      • atr_based (adjusted by volatility)
    Allows dynamic injection via config or kwargs.
    """

    runtime_spec = globals().get("XAUUSD_SPEC", {})

    # --- Load configuration values ---
    if config:
        fixed_lot = config.get("fixed_lot", 1.0)
        risk_percent = config.get("risk_percent", 1.0)
        atr = config.get("atr")
        atr_multiplier = config.get("atr_multiplier", 1.0)
        base_lot_size = config.get("base_lot_size", 0.1)
        volatility_divider = config.get("volatility_divider")
        contract_size = config.get("contract_size", runtime_spec.get("contract_size", 1.0))
        min_lot = config.get("min_lot", runtime_spec.get("min_lot", 0.1))
        max_lot = config.get("max_lot", runtime_spec.get("max_lot", 100.0))
        lot_step = config.get("lot_step", runtime_spec.get("lot_step", 0.01))
        precision = config.get("precision", runtime_spec.get("lot_precision", 2))
        verbose = config.get("verbose", False)
    else:
        fixed_lot = kwargs.get("fixed_lot", 1.0)
        risk_percent = kwargs.get("risk_percent", 1.0)
        atr = kwargs.get("atr")
        atr_multiplier = kwargs.get("atr_multiplier", 1.0)
        base_lot_size = kwargs.get("base_lot_size", 0.1)
        volatility_divider = kwargs.get("volatility_divider")
        contract_size = kwargs.get("contract_size", runtime_spec.get("contract_size", 1.0))
        min_lot = kwargs.get("min_lot", runtime_spec.get("min_lot", 0.1))
        max_lot = kwargs.get("max_lot", runtime_spec.get("max_lot", 100.0))
        lot_step = kwargs.get("lot_step", runtime_spec.get("lot_step", 0.01))
        precision = kwargs.get("precision", runtime_spec.get("lot_precision", 2))
        verbose = kwargs.get("verbose", False)

    lot_size = 0.0

    # --- Fixed lot ---
    if sizing_method == 'fixed':
        lot_size = fixed_lot
        if verbose:
            print(f"[Sizing: fixed] Lot Size = {lot_size:.{precision}f}")

    # --- Risk-based lot ---
    elif sizing_method == 'risk_percent':
        sl_distance = abs(entry_price - stop_loss_price)
        if sl_distance == 0:
            if verbose:
                print("[Warning] Stop-loss distance is 0. Cannot calculate risk-based size.")
            return 0.0
        risk_amount = (risk_percent / 100.0) * capital
        lot_size = risk_amount / (sl_distance * contract_size)
        if verbose:
            print(
                f"[Sizing: risk_percent] Risk = {risk_percent}%, "
                f"Risk Amount = {risk_amount}, SL = {sl_distance}, "
                f"Lot Size = {lot_size:.{precision}f}"
            )

    # --- ATR-based lot ---
    elif sizing_method == 'atr_based':
        if volatility_divider is not None:
            volatility_factor = volatility_divider
        elif atr is not None:
            volatility_factor = atr * atr_multiplier
        else:
            if verbose:
                print("[Warning] ATR or volatility_divider not provided.")
            return 0.0
        if volatility_factor == 0:
            if verbose:
                print("[Warning] Volatility factor is 0. Cannot size by ATR.")
            return 0.0
        lot_size = base_lot_size / volatility_factor
        if verbose:
            print(
                f"[Sizing: atr_based] ATR = {atr}, Multiplier = {atr_multiplier}, "
                f"Factor = {volatility_factor}, Lot Size = {lot_size:.{precision}f}"
            )

    else:
        raise ValueError(f"Invalid sizing method: {sizing_method}")

    if lot_step <= 0:
        raise ValueError("lot_step must be positive")

    try:
        lot_size = float(lot_size)
    except Exception:
        return 0.0

    if not np.isfinite(lot_size) or lot_size <= 0:
        return 0.0
    if lot_size < min_lot:
        if verbose:
            print("[Warning] Calculated lot is below broker minimum. Skipping trade.")
        return 0.0

    lot_size = min(lot_size, max_lot)
    steps = int((lot_size - min_lot) / lot_step + 1e-12)
    lot_size = min_lot + steps * lot_step
    lot_size = round(lot_size, precision)
    return min(max_lot, lot_size)


In [ ]:
# ============================================
# Step 7: Trading Frictions & Execution Cost Models (Final)
# ============================================

import numpy as np
import pandas as pd

# === Slippage Models ===
def apply_slippage(price, config, direction='buy', timestamp=None, verbose=False):
    """
    Apply slippage based on chosen mode.
    Supports: fixed, random_normal, random_uniform, time_sensitive.
    Returns: (price_with_slip, slip_value)
    """
    mode = config.get('slippage_mode', 'fixed')
    value = config.get('slippage_value', 1.0)

    if mode == 'fixed':
        slip = value
    elif mode == 'random_normal':
        mu = config.get('slippage_mu', 1.0)
        sigma = config.get('slippage_sigma', 0.5)
        slip = np.random.normal(mu, sigma)
    elif mode == 'random_uniform':
        low = config.get('slippage_low', 0.0)
        high = config.get('slippage_high', 2.0)
        slip = np.random.uniform(low, high)
    elif mode == 'time_sensitive':
        if timestamp is not None:
            hour = pd.to_datetime(timestamp).hour
            slip = value * 2 if 8 <= hour <= 17 else value
        else:
            slip = value
    else:
        slip = 0.0

    final_price = price + slip if direction == 'buy' else price - slip
    if verbose:
        print(f"[Slippage] Mode: {mode}, Slippage: {slip:.2f}, Final Price: {final_price:.2f}")
    return final_price, slip

# === Commission Model ===
def apply_commission(lot, config, verbose=False):
    """
    Apply commission cost.
    Returns total commission cost (round-trip by default: lot * rate * 2).
    """
    runtime_spec = globals().get("XAUUSD_SPEC", {})
    commission_per_lot = config.get('commission_per_lot')
    if commission_per_lot is None:
        rt_commission = config.get('commission_per_lot_round_turn', runtime_spec.get('commission_per_lot_round_turn', 0.0))
        commission_per_lot = float(rt_commission) / 2.0
    commission = lot * float(commission_per_lot) * 2
    if verbose:
        print(f"[Commission] Lot: {lot}, Rate: {commission_per_lot}, Total: {commission:.2f}")
    return commission

# === Spread Model ===
def apply_spread(price, config, direction='buy', verbose=False):
    """
    Apply spread to entry or exit price.
    Returns price adjusted by spread.
    """
    runtime_spec = globals().get("XAUUSD_SPEC", {})
    spread = config.get('spread_points', runtime_spec.get('spread_points', 0.0))
    point = config.get('point', runtime_spec.get('point', 0.01))
    cost_value_mode = str(config.get('cost_value_mode', runtime_spec.get('cost_value_mode', 'price'))).lower()
    spread_price = float(spread) * float(point) if cost_value_mode == 'points' else float(spread)
    adjusted_price = price + spread_price if direction == 'buy' else price - spread_price
    if verbose:
        print(f"[Spread] Price: {spread_price}, Adjusted Price: {adjusted_price:.2f}")
    return adjusted_price

# === Swap (Overnight Holding Fee) ===
def apply_swap_cost(lot, bars_held, config, verbose=False):
    """
    Apply swap cost based on bars held.
    Returns total swap cost (can be negative or positive depending on broker).
    """
    runtime_spec = globals().get("XAUUSD_SPEC", {})
    daily_swap = config.get('swap_per_lot', runtime_spec.get('swap_per_lot', 0.0))
    bars_per_day = config.get('bars_per_day', runtime_spec.get('bars_per_day', 96))
    days_held = bars_held / bars_per_day
    swap_cost = lot * daily_swap * days_held
    if verbose:
        print(f"[Swap] Bars Held: {bars_held}, Days: {days_held:.2f}, Swap Cost: {swap_cost:.2f}")
    return swap_cost

# === Combined Execution Cost Function ===
def compute_trade_costs(price, lot, bars_held, config, direction='buy', timestamp=None, verbose=False):
    """
    Combine slippage, spread, commission, and swap into one execution cost model.
    Returns:
      executed_price (price adjusted by slip + spread),
      total_cost (commission + swap),
      slippage_value,
      commission
    """
    # Apply slippage
    slipped_price, slippage_value = apply_slippage(price, config, direction, timestamp, verbose)

    # Apply spread
    executed_price = apply_spread(slipped_price, config, direction, verbose)

    # Commission and swap
    commission = apply_commission(lot, config, verbose)
    swap = apply_swap_cost(lot, bars_held, config, verbose)

    total_cost = commission + swap

    if verbose:
        print(f"[Costs] Executed Price: {executed_price:.2f}, Total Cost: {total_cost:.2f}")

    return executed_price, total_cost, slippage_value, commission


In [ ]:
# ============================================
# Step 8: Trade Filters (Modular Entry Filters)
# ============================================

# --- Individual Filters ---

def trend_filter(df, entry_idx, params):
    """
    Pass only if price is above chosen EMA column.
    """
    ema_col = params.get("ema_col", "ema_200")
    close_val = df.at[df.index[entry_idx], "close"]
    ema_val = df.at[df.index[entry_idx], ema_col]
    return close_val > ema_val

def volatility_filter(df, entry_idx, params):
    """
    Pass only if ATR at index is above its rolling average.
    """
    atr_col = f"atr_{params.get('atr_period', 14)}"
    atr_val = df.at[df.index[entry_idx], atr_col]
    atr_mean = df[atr_col].rolling(window=50).mean().iloc[entry_idx]
    return atr_val > atr_mean

def volume_filter(df, entry_idx, params):
    """
    Pass only if volume at index is above its 20-period average.
    """
    vol = df.at[df.index[entry_idx], "volume"]
    avg_vol = df["volume"].rolling(window=20).mean().iloc[entry_idx]
    return vol > avg_vol

def session_filter(df, entry_idx, params):
    """
    Pass only if time is within a defined session window.
    """
    hour = df.index[entry_idx].hour
    start = params.get("start_hour", 8)
    end = params.get("end_hour", 17)
    return start <= hour < end

def time_filter(df, entry_idx, params):
    """
    Exclude restricted hours (default: overnight illiquid times).
    """
    hour = df.index[entry_idx].hour
    restricted = params.get("restricted_hours", [22, 23, 0, 1])
    return hour not in restricted

# --- Filter Registry ---
entry_filters = {
    'trend_filter': trend_filter,
    'volatility_filter': volatility_filter,
    'volume_filter': volume_filter,
    'session_filter': session_filter,
    'time_filter': time_filter
}

# --- Apply All Active Filters ---
def passes_all_filters(df, index, filters, filter_params=None, verbose=False, return_dict=False):
    """
    Apply all selected entry filters to a given row.

    Parameters:
      df            : DataFrame with indicators
      index         : row index to check
      filters       : dict like {'use_trend_filter': True, 'use_time_filter': False}
      filter_params : dict with per-filter parameter dicts
      verbose       : print debug info
      return_dict   : if True, return dict of filter_name->bool instead of combined result

    Returns:
      bool (all pass) or dict (individual results)
    """
    results = {}
    all_pass = True
    filter_params = filter_params or {}

    for filter_name, is_enabled in filters.items():
        if not is_enabled:
            continue

        base_name = filter_name.replace("use_", "")
        func = entry_filters.get(base_name)

        if not func:
            if verbose:
                print(f"[Filter] ❌ {base_name}: function not found.")
            results[base_name] = False
            all_pass = False
            continue

        param_set = filter_params.get(base_name, {})
        passed = func(df, index, param_set)

        results[base_name] = passed
        if verbose:
            symbol = "✅" if passed else "❌"
            print(f"[Filter] {symbol} {base_name}: {passed}")

        if not passed:
            all_pass = False

    return results if return_dict else all_pass


In [ ]:
# ============================================
# Step 9: Strategy Config Generator (Enhanced)
# ============================================

import itertools
import hashlib
from tqdm import tqdm  # ✅ progress bar

def generate_all_strategy_configs(
    indicator_params,
    entry_functions,
    short_entry_functions,
    sl_tp_settings,
    trailing_stop_settings,
    time_based_settings,
    sizing_methods,
    friction_settings,
    max_filters=5,
    strategy_batch_id="RUN001",
    verbose=True
):
    """
    Generate all valid strategy configurations by combining:
    - Indicator sets (2 to 7)
    - All parameter permutations
    - Entry/exit logic
    - Trailing stops, time exits, indicator exits
    - Position sizing methods
    - Execution frictions
    - 0–N entry filters
    Includes progress bars for visibility.
    """

    all_configs = []
    seen_hashes = set()  # for deduplication
    indicator_names = list(indicator_params.keys())
    filter_names = list(entry_filters.keys())

    # Build list of all indicator combinations of size 2 to N
    combo_iter = list(itertools.chain.from_iterable(
        itertools.combinations(indicator_names, r) for r in range(2, len(indicator_names) + 1)
    ))

    if verbose:
        print(f"📊 Preparing to generate configs from {len(combo_iter)} indicator combos...")

    # Outer loop over indicator combinations with progress bar
    for combo in tqdm(combo_iter, desc="🔄 Indicator Combos"):
        # Build param space for this combo
        param_space = {}
        for ind in combo:
            keys = list(indicator_params[ind].keys())
            values = list(indicator_params[ind].values())
            param_space[ind] = list(itertools.product(*values))

        # All param permutations for this combo
        product_sets = list(itertools.product(*param_space.values()))

        # Inner loop over parameter combinations with nested progress bar
        for prod in tqdm(product_sets, desc=f"⚙️ Params for {combo}", leave=False):
            params = {
                ind: dict(zip(list(indicator_params[ind].keys()), prod[i]))
                for i, ind in enumerate(combo)
            }

            # Combine with SL/TP, trailing, time-based exit, sizing, friction, and filters
            for sl_tp in sl_tp_settings:
                for trail in trailing_stop_settings:
                    for time_stop in time_based_settings:
                        for sizing in sizing_methods:
                            for friction in friction_settings:
                                for k in range(0, max_filters + 1):
                                    for selected_filters in itertools.combinations(filter_names, k):
                                        filters_dict = {f"use_{f}": True for f in selected_filters}

                                        # Deduplication hash
                                        hash_input = str((combo, params, sl_tp, trail, time_stop, sizing, friction, filters_dict))
                                        config_hash = hashlib.md5(hash_input.encode()).hexdigest()
                                        if config_hash in seen_hashes:
                                            continue
                                        seen_hashes.add(config_hash)

                                        # Build entry logic
                                        entry_long_fn = generate_entry_condition(combo, params, direction='long')
                                        entry_short_fn = generate_entry_condition(combo, params, direction='short')

                                        config = {
                                            "strategy_batch_id": strategy_batch_id,
                                            "id": f"{'_'.join(combo)}",
                                            "uid": f"{strategy_batch_id}_{config_hash[:10]}",
                                            "indicators": combo,
                                            "params": params,
                                            "entry": {
                                                "long_condition": entry_long_fn,
                                                "short_condition": entry_short_fn,
                                                "type": "composite"
                                            },
                                            "exit": {
                                                **sl_tp,
                                                **trail,
                                                **time_stop
                                            },
                                            "sizing": sizing,
                                            "friction": friction,
                                            "filters": filters_dict,
                                            "metrics": {},
                                            "score": None
                                        }

                                        all_configs.append(config)

    if verbose:
        print(f"✅ Finished generating {len(all_configs):,} unique strategy configs.")
    return all_configs


In [ ]:

# ============================================
# Strategy Parameter Grids (Global)
# ============================================

# A. SL/TP Variants
atr_multipliers = [1.0, 1.5, 2.0, 2.5, 3.0]
rr_ratios = [round(x, 1) for x in np.arange(1.0, 3.1, 0.5)]
fib_levels = [1.618, 2.0, 2.618]
sl_tp_settings = []

for sl_type in ['atr', 'structure']:
    for tp_type in ['rr', 'fib']:
        for atr in atr_multipliers:
            for rr in rr_ratios:
                sl_tp_settings.append({
                    'sl_type': sl_type,
                    'tp_type': tp_type,
                    'atr_multiplier': atr,
                    'risk_reward_ratio': rr,
                    'fib_levels': fib_levels,
                    'exit_on_reversal': True,
                    'max_holding_bars': 50
                })

# B. Trailing Stop Variants
trailing_stop_settings = [
    {'trailing_type': 'none'},
    *[{'trailing_type': 'atr', 'trail_multiplier': x} for x in [1.0, 1.5, 2.0, 2.5]],
    *[{'trailing_type': 'percent', 'trail_percent': x} for x in [0.3, 0.5, 1.0, 1.5]],
    *[{'trailing_type': 'step', 'trail_step_pips': x} for x in [10, 20, 50, 100]]
]

# C. Time-Based Exit
time_based_settings = [{'max_holding_bars': x} for x in range(10, 110, 20)]

# D. Indicator-Based Exit Toggle
indicator_exit_settings = [{'use_indicator_exit': True}, {'use_indicator_exit': False}]

# E. Strategy-Level Drawdown Limits
drawdown_limits = [0.10, 0.15, 0.20]

# F. Sizing Methods
sizing_methods = [
    {'sizing_method': 'fixed', 'fixed_lot': 0.1},
    {'sizing_method': 'risk_percent', 'risk_percent': 1.0},
    {'sizing_method': 'atr_based', 'base_lot_size': 0.1, 'atr_multiplier': 1.5}
]

# G. Friction Settings
from pathlib import Path
from xauusd_ea.baseline import (
    assert_runtime_broker_spec_matches_profile,
    load_broker_profile,
)

_PARAM_GRID_BROKER_PROFILE_PATH = Path(globals().get("BROKER_PROFILE_PATH", "config/xm_micro_gold.json"))
if not _PARAM_GRID_BROKER_PROFILE_PATH.exists():
    _PARAM_GRID_BROKER_PROFILE_PATH = Path.cwd() / "config" / "xm_micro_gold.json"
_PARAM_GRID_BROKER = load_broker_profile(_PARAM_GRID_BROKER_PROFILE_PATH)
_PARAM_GRID_RUNTIME_SPEC = _PARAM_GRID_BROKER.to_runtime_spec()
_PARAM_GRID_CHECKED_SPEC = assert_runtime_broker_spec_matches_profile({
    **_PARAM_GRID_RUNTIME_SPEC,
    'lot_precision': 2,
    'spread_application': 'full',
    'cost_value_mode': 'points',
    'spread_points': float(_PARAM_GRID_RUNTIME_SPEC['spread_baseline_price']) / float(_PARAM_GRID_RUNTIME_SPEC['point']),
    'commission_per_lot_round_turn': float(_PARAM_GRID_RUNTIME_SPEC['commission_per_lot_round_turn_usd']),
    'fee_per_lot_round_turn': float(_PARAM_GRID_RUNTIME_SPEC['fee_per_lot_round_turn_usd']),
    'swap_per_lot': float(_PARAM_GRID_RUNTIME_SPEC['swap_long_points']) * float(_PARAM_GRID_RUNTIME_SPEC['point']) * float(_PARAM_GRID_RUNTIME_SPEC['contract_size']),
    'swap_long_per_lot': float(_PARAM_GRID_RUNTIME_SPEC['swap_long_points']) * float(_PARAM_GRID_RUNTIME_SPEC['point']) * float(_PARAM_GRID_RUNTIME_SPEC['contract_size']),
    'swap_short_per_lot': float(_PARAM_GRID_RUNTIME_SPEC['swap_short_points']) * float(_PARAM_GRID_RUNTIME_SPEC['point']) * float(_PARAM_GRID_RUNTIME_SPEC['contract_size']),
}, _PARAM_GRID_BROKER)
friction_settings = globals().get('CUSTOM_FRICTION_SETTINGS', [{
    'cost_value_mode': _PARAM_GRID_CHECKED_SPEC['cost_value_mode'],
    'slippage_mode': 'fixed',
    'slippage_value': 10,
    'spread_points': _PARAM_GRID_CHECKED_SPEC['spread_points'],
    'commission_per_lot_round_turn': _PARAM_GRID_CHECKED_SPEC['commission_per_lot_round_turn'],
    'fee_per_lot_round_turn': _PARAM_GRID_CHECKED_SPEC['fee_per_lot_round_turn'],
    'swap_per_lot': _PARAM_GRID_CHECKED_SPEC['swap_per_lot'],
    'swap_long_per_lot': _PARAM_GRID_CHECKED_SPEC['swap_long_per_lot'],
    'swap_short_per_lot': _PARAM_GRID_CHECKED_SPEC['swap_short_per_lot'],
    'bars_per_day': 96
}])


In [ ]:
# ============================================
# Step 10: Core Backtest Engine (Corrected & Hardened, supports combined_params)
# ============================================

def run_backtest(config, df, initial_capital=10000.0, verbose=False, filter_params=None):
    capital = initial_capital
    open_position = None
    trades = []
    equity_curve = []
    bars_held = 0

    # Get entry functions
    entry_long_fn = config["entry"]["long_condition"]
    entry_short_fn = config["entry"]["short_condition"]
    filters = config.get("filters", {})

    # Dynamically determine ATR period to read from df
    atr_period = config.get("exit", {}).get("atr_period", 14)
    atr_col = f"atr_{atr_period}" if f"atr_{atr_period}" in df.columns else "atr_14"

    for i in range(1, len(df)):
        row = df.iloc[i]
        timestamp = row.name
        price = row["close"]
        current_atr = row.get(atr_col, 10)  # fallback if missing

        # === Apply Trade Filters ===
        if not passes_all_filters(df, i, filters, filter_params=filter_params):
            equity_curve.append(capital)
            continue

        # === Entry Logic ===
        if open_position is None:
            signal_long = entry_long_fn(df).iloc[i]
            signal_short = entry_short_fn(df).iloc[i]

            if signal_long:
                direction = 'long'
            elif signal_short:
                direction = 'short'
            else:
                equity_curve.append(capital)
                continue

            entry_price = price
            atr_multiplier = config["exit"].get("atr_multiplier", 2.0)

            # Stop Loss
            if config["exit"]["sl_type"] == 'atr':
                stop_loss = calculate_atr_stop(entry_price, current_atr, atr_multiplier, direction)
            else:
                stop_loss = calculate_structure_stop(df, i, direction)

            # Take Profit
            if config["exit"]["tp_type"] == 'rr':
                tp_price = calculate_tp_by_rr(
                    entry_price,
                    stop_loss,
                    config["exit"].get("risk_reward_ratio", 2.0),
                    direction
                )
            else:
                tp_price = calculate_fib_target(
                    entry_price,
                    config["params"].get("Fibonacci", {}),
                    direction
                )[0]

            # Position Sizing
            lot = calculate_position_size(
                capital,
                entry_price,
                stop_loss,
                sizing_method=config["sizing"]["sizing_method"],
                config=config["sizing"]
            )

            # Slippage + Spread Adjustment
            executed_price, total_cost, *_ = compute_trade_costs(
                entry_price,
                lot,
                0,
                config["friction"],
                direction=direction,
                timestamp=timestamp,
                verbose=verbose
            )

            # Open Position
            open_position = {
                "entry_idx": i,
                "entry_price": executed_price,
                "lot": lot,
                "direction": direction,
                "stop_loss": stop_loss,
                "take_profit": tp_price
            }
            bars_held = 0

        else:
            # === Manage Open Position ===
            bars_held += 1
            current_price = price
            trail_type = config["exit"].get("trailing_type", "none")

            # Trailing stop logic
            if trail_type != "none":
                new_sl = open_position["stop_loss"]

                if trail_type == "atr":
                    multiplier = config["exit"].get("trail_multiplier", 1.5)
                    distance = current_atr * multiplier
                    new_sl = current_price - distance if open_position["direction"] == "long" else current_price + distance

                elif trail_type == "percent":
                    percent = config["exit"].get("trail_percent", 0.5) / 100.0
                    distance = current_price * percent
                    new_sl = current_price - distance if open_position["direction"] == "long" else current_price + distance

                elif trail_type == "step":
                    pip_value = 0.1
                    step_pips = config["exit"].get("trail_step_pips", 20)
                    gain = (current_price - open_position["entry_price"]) if open_position["direction"] == "long" else (open_position["entry_price"] - current_price)
                    if gain >= step_pips * pip_value:
                        new_sl = current_price - step_pips * pip_value if open_position["direction"] == "long" else current_price + step_pips * pip_value

                # Update stop loss if trailing improved
                if open_position["direction"] == "long" and new_sl > open_position["stop_loss"]:
                    open_position["stop_loss"] = new_sl
                elif open_position["direction"] == "short" and new_sl < open_position["stop_loss"]:
                    open_position["stop_loss"] = new_sl

            # Exit conditions
            hit_tp = current_price >= open_position["take_profit"] if open_position["direction"] == "long" else current_price <= open_position["take_profit"]
            hit_sl = current_price <= open_position["stop_loss"] if open_position["direction"] == "long" else current_price >= open_position["stop_loss"]

            exit_price = None
            exit_reason = None

            if hit_tp:
                exit_price = open_position["take_profit"]
                exit_reason = "TP"
            elif hit_sl:
                exit_price = open_position["stop_loss"]
                exit_reason = "SL"
            elif config["exit"].get("max_holding_bars", 0) > 0 and check_max_holding(open_position["entry_idx"], i, config["exit"]["max_holding_bars"]):
                exit_price = current_price
                exit_reason = "Time"
            elif config["exit"].get("exit_on_reversal"):
                reversal_fn = indicator_reversal(
                    entry_long_fn if open_position["direction"] == 'long' else entry_short_fn
                )
                if reversal_fn(df).iloc[i]:
                    exit_price = current_price
                    exit_reason = "Reversal"

            # Execute exit
            if exit_price is not None:
                executed_price, total_cost, *_ = compute_trade_costs(
                    exit_price,
                    open_position["lot"],
                    bars_held,
                    config["friction"],
                    direction=open_position["direction"],
                    timestamp=timestamp,
                    verbose=verbose
                )

                pnl = (executed_price - open_position["entry_price"]) * open_position["lot"] * 100
                pnl *= 1 if open_position["direction"] == "long" else -1
                pnl -= total_cost
                capital += pnl

                trades.append({
                    "entry": open_position["entry_price"],
                    "exit": executed_price,
                    "entry_time": df.index[open_position["entry_idx"]],
                    "exit_time": timestamp,
                    "entry_idx": open_position["entry_idx"],
                    "exit_idx": i,
                    "lot": open_position["lot"],
                    "direction": open_position["direction"],
                    "pnl": pnl,
                    "bars": bars_held,
                    "reason": exit_reason
                })

                open_position = None

        equity_curve.append(capital)

    if not equity_curve:
        equity_curve = [initial_capital]

    return trades, capital, equity_curve


In [ ]:
# ============================================
# Step 10.5: Strategy Evaluation, Scoring, Filtering
# ============================================

import numpy as np
import pandas as pd

def evaluate_strategy(trades, equity_curve, config):
    """
    Evaluate a single strategy's performance metrics and store in config['metrics'].
    """
    metrics = {}
    if not trades:
        config["metrics"] = {
            'net_profit': 0,
            'win_rate': 0,
            'profit_factor': 0,
            'expectancy': 0,
            'sharpe': 0,
            'max_drawdown': 0,
            'trade_count': 0
        }
        config["score"] = -np.inf
        return config

    trade_df = pd.DataFrame(trades)

    pnl = trade_df['pnl']
    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    metrics['net_profit'] = pnl.sum()
    metrics['trade_count'] = len(trade_df)
    metrics['win_rate'] = (len(wins) / len(trade_df)) if len(trade_df) else 0
    metrics['profit_factor'] = wins.sum() / abs(losses.sum()) if abs(losses.sum()) > 0 else np.inf
    metrics['expectancy'] = pnl.mean()
    metrics['sharpe'] = pnl.mean() / (pnl.std() + 1e-9) if len(pnl) > 1 else 0

    # Drawdown
    equity = np.array(equity_curve)
    peak = np.maximum.accumulate(equity)
    drawdown = (peak - equity) / (peak + 1e-9)
    metrics['max_drawdown'] = drawdown.max()

    config["metrics"] = metrics

    # === Scoring Formula ===
    score = (
        metrics['net_profit'] -
        metrics['max_drawdown'] * 1000 +  # penalize drawdown
        metrics['sharpe'] * 100 +
        metrics['profit_factor'] * 50
    )
    config["score"] = round(score, 4)

    return config


def filter_strategies(all_configs, min_trade_count=30, max_drawdown=0.20, min_sharpe=1.0):
    """
    Filter strategies based on performance criteria.
    Returns only configs that pass.
    """
    selected = []
    for config in all_configs:
        m = config.get("metrics", {})
        if (
            m.get('net_profit', 0) > 0 and
            m.get('trade_count', 0) >= min_trade_count and
            m.get('max_drawdown', 1) <= max_drawdown and
            m.get('sharpe', 0) >= min_sharpe
        ):
            selected.append(config)
    return selected


def rank_strategies(configs, top_n=50):
    """
    Return the top-N strategies by score.
    """
    return sorted(configs, key=lambda x: x.get("score", -np.inf), reverse=True)[:top_n]

In [ ]:
# ============================================
# Step 11A: Walk-Forward Splitter
# ============================================

def split_walk_forward_windows(df, train_window=3000, test_window=500, step_size=500):
    windows = []
    max_start = len(df) - train_window - test_window
    for start in range(0, max_start + 1, step_size):
        train_start = start
        train_end = start + train_window
        test_start = train_end
        test_end = train_end + test_window
        if test_end > len(df):
            break
        windows.append(((train_start, train_end), (test_start, test_end)))
    return windows


# ============================================
# Step 11B: Run All Strategy Configs on Training Set
# ============================================

def run_training_loop(df, window, strategy_configs):
    (train_start, train_end), _ = window
    train_df = df.iloc[train_start:train_end].copy()
    results = []

    for config in strategy_configs:
        trades, final_capital, equity_curve = run_backtest(config, train_df)
        evaluate_strategy(trades, equity_curve, config)  # use Step 10.5 eval
        results.append({
            'strategy': config,
            'final_capital': final_capital,
            'trades': trades,
            'equity_curve': equity_curve,
            **config['metrics'],
            'score': config['score']
        })

    return results


# ============================================
# Step 11C: Select Top Strategies by Score
# ============================================

def select_top_strategies(results, top_n=3, metric='score'):
    sorted_results = sorted(results, key=lambda x: x.get(metric, -np.inf), reverse=True)
    return sorted_results[:top_n]


# ============================================
# Step 11D: Run Top Strategies on Test Set
# ============================================

def validate_on_test_set(df, window, top_strategies):
    _, (test_start, test_end) = window
    test_df = df.iloc[test_start:test_end].copy()
    test_results = []

    for strategy_result in top_strategies:
        strategy_config = strategy_result['strategy']
        trades, final_capital, equity_curve = run_backtest(strategy_config, test_df)
        evaluate_strategy(trades, equity_curve, strategy_config)
        test_results.append({
            'strategy': strategy_config,
            'final_capital': final_capital,
            'trades': trades,
            'equity_curve': equity_curve,
            **strategy_config['metrics'],
            'score': strategy_config['score']
        })

    return test_results


# ============================================
# Step 11E: Collect Results by Window
# ============================================

def aggregate_results(window_index, window, test_results):
    aggregated = []
    for result in test_results:
        record = result.copy()
        record['window_index'] = window_index
        record['window'] = window
        aggregated.append(record)
    return aggregated


# ============================================
# Step 11F: Full Walk-Forward Execution
# ============================================

def run_full_walk_forward(df, strategy_configs, wf_config):
    """
    wf_config = {
        'train_window': 3000,
        'test_window': 500,
        'step_size': 500,
        'top_n_strategies': 3
    }
    """
    wf_results = []
    windows = split_walk_forward_windows(df,
                                         wf_config.get("train_window", 3000),
                                         wf_config.get("test_window", 500),
                                         wf_config.get("step_size", 500))

    for idx, window in enumerate(windows):
        print(f"▶ Window {idx+1}/{len(windows)}: Train {window[0]}, Test {window[1]}")
        train_results = run_training_loop(df, window, strategy_configs)
        top_strategies = select_top_strategies(train_results,
                                               top_n=wf_config.get("top_n_strategies", 3),
                                               metric="score")
        test_results = validate_on_test_set(df, window, top_strategies)
        wf_results.extend(aggregate_results(idx, window, test_results))

    return wf_results


In [ ]:
# ============================================
# Step 12A: Compute Strategy Performance Metrics (CAGR fix)
# ============================================

import numpy as np
import pandas as pd

def compute_strategy_metrics(trades, initial_capital=10000.0, risk_free_rate=0.0):
    if not trades:
        return {k: 0 for k in [
            'Net Profit', 'CAGR', 'Max Drawdown', 'Volatility', 'Sharpe Ratio',
            'Sortino Ratio', 'Win Rate', 'Profit Factor', '# Trades',
            'Avg Trade Duration', 'Expectancy', 'Recovery Factor',
            'Max Consecutive Losses', 'Strategy Score'
        ]}

    df = pd.DataFrame(trades)
    df['cum_pnl'] = df['pnl'].cumsum()
    df['equity'] = initial_capital + df['cum_pnl']
    df['returns'] = df['pnl'] / initial_capital

    net_profit = df['pnl'].sum()

    # ✅ Fix: guard against invalid duration values
    bars_per_year = 6048  # for M15
    duration_years = len(df) / bars_per_year
    if duration_years <= 0:
        cagr = 0.0
    else:
        ratio = (initial_capital + net_profit) / initial_capital
        if ratio <= 0:
            cagr = 0.0
        else:
            cagr = ratio ** (1 / duration_years) - 1

    # Drawdown
    peak = df['equity'].cummax()
    drawdown = peak - df['equity']
    max_drawdown = drawdown.max()

    # Volatility & Sharpe
    volatility = df['returns'].std() * np.sqrt(len(df))
    downside = df['returns'][df['returns'] < 0]
    sortino_denom = downside.std() * np.sqrt(len(df)) if len(downside) > 0 else 1e-6

    sharpe = (df['returns'].mean() - risk_free_rate) / (volatility + 1e-9)
    sortino = (df['returns'].mean() - risk_free_rate) / sortino_denom

    wins = df[df['pnl'] > 0]
    losses = df[df['pnl'] < 0]
    win_rate = len(wins) / len(df) if len(df) else 0
    profit_factor = wins['pnl'].sum() / abs(losses['pnl'].sum()) if not losses.empty else np.inf
    expectancy = df['pnl'].mean()
    avg_duration = df['bars'].mean() if 'bars' in df.columns else 1

    # Max consecutive losses
    loss_flags = (df['pnl'] < 0).astype(int)
    groups = (loss_flags != loss_flags.shift()).cumsum()
    max_consec_losses = loss_flags.groupby(groups).sum().max()

    recovery_factor = net_profit / max_drawdown if max_drawdown > 0 else np.inf

    score = (
        sharpe * 0.4 +
        profit_factor * 0.2 +
        win_rate * 0.2 +
        expectancy * 0.1 -
        (max_drawdown / initial_capital) * 0.1
    )

    return {
        'Net Profit': net_profit,
        'CAGR': cagr,
        'Max Drawdown': max_drawdown,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Win Rate': win_rate,
        'Profit Factor': profit_factor,
        '# Trades': len(df),
        'Avg Trade Duration': avg_duration,
        'Expectancy': expectancy,
        'Recovery Factor': recovery_factor,
        'Max Consecutive Losses': max_consec_losses,
        'Strategy Score': score
    }


# ============================================
# Step 12B: Backtest Evaluation Loop (Safe)
# ============================================

def evaluate_backtest_configs(configs, df, initial_capital=10000):
    records = []

    for config in configs:
        trades, final_capital, equity_curve = run_backtest(
            config,
            df,
            initial_capital=initial_capital
        )

        # Always ensure equity_curve is array-like
        if equity_curve and len(equity_curve) > 0:
            equity_curve_arr = np.array(equity_curve, dtype=float)
        else:
            # fallback to a single-point curve if no bars
            equity_curve_arr = np.array([initial_capital], dtype=float)

        metrics = compute_strategy_metrics(trades, initial_capital=initial_capital)
        metrics.update({
            'Strategy ID': config.get('id', 'N/A'),
            'Sizing': config.get('sizing', {}).get('sizing_method', 'N/A'),
            'Final Capital': final_capital,
            'equity_curve': equity_curve_arr,
            'trades': trades
        })

        records.append(metrics)

    df_results = pd.DataFrame(records)

    # Deduplicate & filter
    return clean_strategies(df_results)



# ============================================
# Step 12C: Walk-Forward Result Enrichment
# ============================================

def enrich_wf_results_with_metrics(wf_results, initial_capital=10000):
    enriched = []
    for result in wf_results:
        metrics = compute_strategy_metrics(result['trades'], initial_capital=initial_capital)
        enriched.append({
            'Window': result['window_index'],
            'Strategy ID': result['strategy'].get('id', 'N/A'),
            'Sizing': result['strategy'].get('sizing', {}).get('sizing_method', 'N/A'),
            'Final Capital': result['final_capital'],
            **metrics
        })

    df_enriched = pd.DataFrame(enriched)
    return clean_strategies(df_enriched)  # ✅ Deduplication + Filtering



In [ ]:
# ============================================
# Step 13A: Remove Duplicate Strategies (Safe)
# ============================================

def deduplicate_strategies(df, threshold=0.99, dedup_metric='Strategy Score', verbose=False):
    """
    Drops strategies with highly correlated equity curves.
    Keeps only the top-scoring from each cluster.
    Safely skips if equity_curve data is missing or empty.
    """
    if "equity_curve" not in df.columns or df.empty:
        if verbose:
            print("⚠️ No equity_curve column or empty DataFrame. Skipping deduplication.")
        return df

    # Filter valid equity curves
    valid_indices = []
    for idx, curve in enumerate(df["equity_curve"].values):
        if isinstance(curve, (list, np.ndarray)) and len(curve) > 0:
            valid_indices.append(idx)

    if not valid_indices:
        if verbose:
            print("⚠️ No valid equity curves found. Skipping deduplication.")
        return df

    # Build matrix only from valid indices
    equity_matrix = np.vstack([df["equity_curve"].iloc[i] for i in valid_indices])
    corr_matrix = np.corrcoef(equity_matrix)

    keep = []
    seen = set()

    for i_pos, i in enumerate(valid_indices):
        if i in seen:
            continue
        group = [i]
        for j_pos, j in enumerate(valid_indices):
            if j_pos <= i_pos:
                continue
            if corr_matrix[i_pos][j_pos] >= threshold:
                group.append(j)
        best_idx = max(group, key=lambda idx: df.loc[idx, dedup_metric])
        keep.append(best_idx)
        seen.update(group)

    result = df.loc[keep].reset_index(drop=True)
    if verbose:
        print(f"✅ Deduplicated from {len(df)} to {len(result)} strategies.")
    return result




# ============================================
# Step 13B: Apply Robustness Filters
# ============================================

def filter_valid_strategies(df, min_trades=30, max_mdd=0.3, win_rate_cap=0.95, capital=10000, verbose=False):
    """
    Filters strategies based on trade count, drawdown, and win rate sanity.
    """
    df = df.copy()

    if verbose:
        print(f"📊 Applying filters: min_trades={min_trades}, max_drawdown={max_mdd}, win_rate_cap={win_rate_cap}")

    before = len(df)

    df = df[df["# Trades"] >= min_trades]
    df = df[df["Max Drawdown"] <= max_mdd * capital]
    df = df[df["Win Rate"] <= win_rate_cap]

    after = len(df)
    if verbose:
        print(f"✅ Filtered {before - after} strategies. Remaining: {after}")
    return df.reset_index(drop=True)



# ============================================
# Step 13C: Combined Filter and Deduplicate (Safe)
# ============================================

def clean_strategies(
    df,
    use_dedup=True,
    capital=10000,
    min_trades=30,
    max_mdd=0.3,
    win_rate_cap=0.95,
    dedup_threshold=0.99,
    dedup_metric='Strategy Score',
    verbose=False
):
    """
    Filters strategies by basic validity criteria and optionally deduplicates them.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing strategy metrics and equity_curves.
    use_dedup : bool
        Whether to run correlation-based deduplication.
    capital : float
        Initial capital (used for interpreting drawdowns).
    min_trades : int
        Minimum trade count for validity.
    max_mdd : float
        Maximum allowed drawdown as a fraction of capital (e.g., 0.3 = 30%).
    win_rate_cap : float
        Upper sanity cap on win rate (e.g., 0.95 = 95%).
    dedup_threshold : float
        Correlation threshold to consider strategies duplicates.
    dedup_metric : str
        Metric to decide which strategy to keep in a duplicate group.
    verbose : bool
        If True, prints logs.

    Returns
    -------
    pd.DataFrame
        Cleaned (and optionally deduplicated) DataFrame of strategies.
    """

    if df.empty:
        if verbose:
            print("⚠️ Input DataFrame is empty. Nothing to clean.")
        return df

    # --- Step 1: Filter by metrics
    filtered = df.copy()

    # Minimum trade count
    filtered = filtered[filtered["# Trades"] >= min_trades]

    # Max drawdown (convert to raw currency if needed)
    # In your metrics, 'Max Drawdown' is absolute, so compare directly:
    filtered = filtered[filtered["Max Drawdown"] <= (max_mdd * capital)]

    # Win rate sanity cap
    filtered = filtered[filtered["Win Rate"] <= win_rate_cap]

    filtered = filtered.reset_index(drop=True)

    if verbose:
        print(f"✅ Filtered strategies: {len(filtered)} remain after basic criteria.")

    # --- Step 2: Deduplicate
    if use_dedup:
        filtered = deduplicate_strategies(
            filtered,
            threshold=dedup_threshold,
            dedup_metric=dedup_metric,
            verbose=verbose
        )
    else:
        if verbose:
            print("ℹ️ Deduplication skipped by user request.")

    if verbose:
        print(f"✅ Final cleaned strategies: {len(filtered)} remain.")

    return filtered



In [ ]:
# ============================================
# Step 14A: Export & Rank Top Strategies
# ============================================

def export_top_strategies(df, top_n=10, output_path="top_strategies.csv"):
    """
    Save top N strategies based on Strategy Score to CSV.
    """
    top_df = df.sort_values("Strategy Score", ascending=False).head(top_n)
    top_df.to_csv(output_path, index=False)
    return top_df



# ============================================
# Step 14B: Plot Equity Curve of a Strategy
# ============================================

import matplotlib.pyplot as plt

def plot_equity_curve(trades, initial_capital=10000):
    """
    Plot the cumulative equity curve of a trade list.
    """
    equity = [initial_capital]
    for trade in trades:
        equity.append(equity[-1] + trade['pnl'])

    plt.figure(figsize=(10, 5))
    plt.plot(equity, label="Equity Curve", linewidth=2)
    plt.title("Strategy Equity Curve")
    plt.xlabel("Trade Number")
    plt.ylabel("Equity")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================
# Step 14C: Export Strategy Config as JSON
# ============================================

import json

def export_strategy_config(strategy_config, filepath="best_strategy.json"):
    """
    Export selected strategy config to a .json file for EA conversion or reuse.
    """
    export_dict = {
        "indicators": strategy_config.get("indicators"),
        "params": strategy_config.get("params"),
        "entry": strategy_config.get("entry", {}).get("type", "composite"),
        "exit": strategy_config.get("exit"),
        "sizing": strategy_config.get("sizing"),
        "filters": strategy_config.get("filters"),
        "friction": strategy_config.get("friction")
    }
    with open(filepath, 'w') as f:
        json.dump(export_dict, f, indent=2)



# ============================================
# Step 14D: Unified Evaluation Mode (Fixed for unhashable columns)
# ============================================

def unified_strategy_comparison(
    backtest_df,
    walkforward_df,
    top_n=1,
    score_col="Strategy Score",
    output_json=True,
    json_path_prefix="best_strategy",
    output_csv=False
):
    """
    Compare top strategies from backtest and walk-forward and export configs.
    Drops unhashable columns (equity_curve, trades) before deduplication.
    """
    results = {}

    # 1. Top from Walk-Forward
    top_wf = walkforward_df.sort_values(score_col, ascending=False).head(top_n)
    results["walkforward"] = top_wf

    # 2. Top from Backtest
    top_bt = backtest_df.sort_values(score_col, ascending=False).head(top_n)
    results["backtest"] = top_bt

    # 3. Drop unhashable columns for deduplication
    drop_cols = ['equity_curve', 'trades']
    top_bt_clean = top_bt.drop(columns=[c for c in drop_cols if c in top_bt.columns], errors='ignore')
    top_wf_clean = top_wf.drop(columns=[c for c in drop_cols if c in top_wf.columns], errors='ignore')

    # 4. Combine and deduplicate
    combined = pd.concat([top_bt_clean, top_wf_clean], ignore_index=True)
    combined = combined.drop_duplicates()
    top_combined = combined.sort_values(score_col, ascending=False).head(top_n)
    results["combined"] = top_combined

    # 5. Export configs
    for label, df in results.items():
        if output_json:
            for i, row in df.iterrows():
                config = row.get("strategy") if isinstance(row.get("strategy"), dict) else None
                if config:
                    export_strategy_config(config, filepath=f"{json_path_prefix}_{label}.json")
        if output_csv:
            df.to_csv(f"{json_path_prefix}_{label}.csv", index=False)

    return results



In [ ]:
# ============================================================
# FINAL MULTI-TIMEFRAME / WALK-FORWARD PIPELINE — V3.9 HOLDOUT-SAFE EXACT CONFIG FORWARD
# Fixes applied:
#   1) Explicit RUN_MODE instead of mislabeled single-timeframe full test
#   2) XAUUSD broker spec + lot normalization + contract-size PnL
#   3) One final run_backtest() engine with next-bar execution option
#   4) SL/TP evaluated with high/low, not close only
#   5) ATR/percent/step trailing support
#   6) indicator-exit + max-drawdown cutoff integrated
#   7) filter combinations 0..MAX_FILTERS integrated
#   8) multi-timeframe and optional walk-forward runner
#   9) V2 fixes: explicit cost units, no ATR look-ahead fallback, mark-to-market equity, entry-bar SL/TP, OOS aggregation, dedup, MT5-style smoothing, stronger Fib zones
#   10) V3 fixes: robust MT5 CSV import compatibility, ATR entry option, TF-specific WFA/swap, OOS gate, CLEAN dedup, early-stop metric fix, bid/ask-aware execution
#   11) V3.4 fixes: stricter SAMPLED trade gate, TF friction in Cell 19, optional skip risk trades below min lot, clearer trailing-stop reasons
#   12) V3.6 fixes: robust RUN_ID generation when RUN_ID_OVERRIDE is None/blank/"None" string
#   13) V3.7 fixes: AUTO_SAMPLE_THEN_WF samples all timeframes first, selects passing candidates, then runs focused Walk-Forward automatically
#   14) V3.8 fixes: AUTO_SAMPLE_THEN_WF forward-tests exact sampled configs loaded from CFGS.pkl, not regenerated indicator families
#   15) V3.9 fixes: chronological sample/forward holdout split, sample-completion guard, stronger OOS pass rule, direct Config Batch Path lookup, debug flag
# ============================================================
from datetime import datetime
import os, gc, pickle, random, copy, itertools, math, json, hashlib, time, glob
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

from xauusd_ea.validation import (
    UnsafeEvaluationError,
    assert_exact_forward_config_identity,
    plan_walk_forward_windows,
    split_sample_holdout,
)
from xauusd_ea.baseline import (
    assert_runtime_broker_spec_matches_profile,
    load_broker_profile,
)

# ============================================================
# 0) RUNTIME CONFIG
# ============================================================
# Edit RUN_MODE_DEFAULT to choose the normal run mode for a fresh runtime.
# RUN_MODE_OVERRIDE is the only external override; this intentionally ignores stale RUN_MODE values left in memory.
RUN_MODE_DEFAULT = "AUTO_SAMPLE_THEN_WF"
RUN_MODE = globals().get("RUN_MODE_OVERRIDE", RUN_MODE_DEFAULT) or RUN_MODE_DEFAULT

RUN_ID_OVERRIDE = "AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE"

_RUN_ID_OVERRIDE_RAW = globals().get("RUN_ID_OVERRIDE", None)
if _RUN_ID_OVERRIDE_RAW is None or str(_RUN_ID_OVERRIDE_RAW).strip().lower() in {"", "none", "nan"}:
    RUN_ID = f"{RUN_MODE}_{datetime.now().strftime('%Y%m%d_%H%M')}"
else:
    RUN_ID = str(_RUN_ID_OVERRIDE_RAW).strip()

_BROKER_PROFILE_PATH = Path(globals().get("BROKER_PROFILE_PATH", "config/xm_micro_gold.json"))
if not _BROKER_PROFILE_PATH.exists():
    _BROKER_PROFILE_PATH = Path.cwd() / "config" / "xm_micro_gold.json"
_BROKER_PROFILE = load_broker_profile(_BROKER_PROFILE_PATH)
INITIAL_CAPITAL = float(_BROKER_PROFILE.initial_capital_usd)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
PIPELINE_START_TIME = time.time()

OUTPUT_ROOT = "/content/drive/MyDrive/XAUUSD_fulltest_batches"
if not os.path.exists("/content"):
    OUTPUT_ROOT = os.path.abspath("./XAUUSD_fulltest_batches")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Use next-bar execution to reduce look-ahead bias.
# "next_bar_open" = signal from bar i-1, enter/exit at bar i open.
# "same_bar_close" = legacy-style, signal and execution at same bar close.
EXECUTION_MODE = "next_bar_open"
SAME_BAR_EXIT_POLICY = "SL_FIRST"  # conservative when TP and SL are both touched in the same candle
ENTRY_MODE = "AND"                 # "AND", "OR", or "VOTE"
MIN_CONFIRMATIONS = None            # used only for VOTE; None = majority

# Verified XM Micro GOLDmicro runtime constants. The checked profile is the single source of truth.
_XM_RUNTIME_SPEC = _BROKER_PROFILE.to_runtime_spec()
XAUUSD_SPEC = assert_runtime_broker_spec_matches_profile({
    **_XM_RUNTIME_SPEC,
    "lot_precision": 2,
    "spread_application": "full",  # used only when ohlc_price_source is not bid/ask
    "cost_value_mode": "points",   # "points" = MT5 points; set to "price" only if values are already price units
    "spread_points": float(_XM_RUNTIME_SPEC["spread_baseline_price"]) / float(_XM_RUNTIME_SPEC["point"]),
    "commission_per_lot_round_turn": float(_XM_RUNTIME_SPEC["commission_per_lot_round_turn_usd"]),
    "fee_per_lot_round_turn": float(_XM_RUNTIME_SPEC["fee_per_lot_round_turn_usd"]),
    "swap_per_lot": float(_XM_RUNTIME_SPEC["swap_long_points"]) * float(_XM_RUNTIME_SPEC["point"]) * float(_XM_RUNTIME_SPEC["contract_size"]),
    "swap_long_per_lot": float(_XM_RUNTIME_SPEC["swap_long_points"]) * float(_XM_RUNTIME_SPEC["point"]) * float(_XM_RUNTIME_SPEC["contract_size"]),
    "swap_short_per_lot": float(_XM_RUNTIME_SPEC["swap_short_points"]) * float(_XM_RUNTIME_SPEC["point"]) * float(_XM_RUNTIME_SPEC["contract_size"]),
}, _BROKER_PROFILE)

BARS_PER_YEAR_BY_TF = {
    "M15": 4 * 24 * 5 * 52,
    "M30": 2 * 24 * 5 * 52,
    "H1":  1 * 24 * 5 * 52,
    "H4":  6 * 5 * 52,
}

BARS_PER_DAY_BY_TF = {
    "M15": 96,
    "M30": 48,
    "H1": 24,
    "H4": 6,
}

# Calendar-normalized walk-forward windows. Bars differ by timeframe, so V2's single window size
# made M15/M30 OOS windows too short. These are still adjustable before final production runs.
WF_SETTINGS_BY_TF = {
    "M15": {"train_window": 12000, "test_window": 2000, "step_size": 1000, "top_n_train": 3},
    "M30": {"train_window": 6000,  "test_window": 1000, "step_size": 500,  "top_n_train": 3},
    "H1":  {"train_window": 3000,  "test_window": 500,  "step_size": 250,  "top_n_train": 3},
    "H4":  {"train_window": 750,   "test_window": 125,  "step_size": 60,   "top_n_train": 3},
}

MODE_SETTINGS = {
    "SMOKE": {
        "timeframes": ["M15"],
        "batch_size": 100,
        "max_params_per_combo": 10,
        "settings_samples": 1,
        "total_configs_cap": 200,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 3,   # smoke only validates pipeline; keep low but avoid 1-trade winners
        "drop_atr_from_entries": False,
        "max_data_rows": 12000,
        "max_runtime_minutes": 25,
        "max_batches_per_tf": 3,
    },
    # Runtime-safe default sampled mode for Colab. It runs one timeframe and fewer configs first.
    # Use this after SMOKE. Expand to SAMPLED_MULTI_TF or SAMPLED_DEEP only after the fast run works.
    "SAMPLED": {
        "timeframes": ["M30"],
        "batch_size": 50,
        "max_params_per_combo": 12,
        "settings_samples": 2,
        "total_configs_cap": 1500,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,  # stricter SAMPLED gate for reliability
        "drop_atr_from_entries": False,
        "max_data_rows": 12000,
        "max_runtime_minutes": 55,
        "max_batches_per_tf": 30,
    },
    # Same sampled profile but for H1 only. Use this when M30 candidates are weak.
    "SAMPLED_H1": {
        "timeframes": ["H1"],
        "batch_size": 50,
        "max_params_per_combo": 12,
        "settings_samples": 2,
        "total_configs_cap": 1500,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,
        "drop_atr_from_entries": False,
        "max_data_rows": 8000,
        "max_runtime_minutes": 55,
        "max_batches_per_tf": 30,
    },
    # Same sampled profile but for M15 only. It can be slower than M30/H1.
    "SAMPLED_M15": {
        "timeframes": ["M15"],
        "batch_size": 50,
        "max_params_per_combo": 12,
        "settings_samples": 2,
        "total_configs_cap": 1500,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,
        "drop_atr_from_entries": False,
        "max_data_rows": 12000,
        "max_runtime_minutes": 55,
        "max_batches_per_tf": 30,
    },
    # Runtime-safe all-timeframe sampled scan. Use this when you want candidates from every TF before WFA.
    # Ordered from lighter/stabler TFs first; M15 is last because it is usually the heaviest.
    "SAMPLED_ALL_TF": {
        "timeframes": ["M30", "H1", "H4", "M15"],
        "batch_size": 50,
        "max_params_per_combo": 12,
        "settings_samples": 2,
        "total_configs_cap": 1500,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,
        "drop_atr_from_entries": False,
        "max_data_rows_by_tf": {"M15": 12000, "M30": 12000, "H1": 8000, "H4": 5000},
        "max_runtime_minutes": 150,
        "max_batches_per_tf": 30,
    },
    # Two-stage orchestration:
    #   Phase 1: run SAMPLED_ALL_TF across M30/H1/H4/M15 and collect CLEAN candidates.
    #   Phase 2: automatically focus WALK_FORWARD on candidate timeframes + indicator families only.
    "AUTO_SAMPLE_THEN_WF": {
        "timeframes": ["M30", "H1", "H4", "M15"],
        "batch_size": 50,
        "max_params_per_combo": 12,
        "settings_samples": 2,
        "total_configs_cap": 1500,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,
        "drop_atr_from_entries": False,
        "max_data_rows_by_tf": {"M15": 12000, "M30": 12000, "H1": 8000, "H4": 5000},
        "max_runtime_minutes": 150,
        "max_batches_per_tf": 30,
        "candidate_top_n": 10,
        "candidate_max_per_tf_family": 2,
        # Use only the first part of each capped TF dataset for sample selection.
        # Exact Forward/WFA then uses the later holdout segment, reducing selection leakage.
        "sample_forward_split_ratio": 0.70,
        "require_sample_complete_before_forward": True,
        "focused_wf": {
            "timeframes": [],
            "batch_size": 20,
            "max_params_per_combo": 4,
            "settings_samples": 1,
            "total_configs_cap": 300,
            "max_filters": 0,
            "filter_samples": 1,
            "min_trades": 30,
            "drop_atr_from_entries": False,
            "max_data_rows_by_tf": {"M15": 8000, "M30": 8000, "H1": 7000, "H4": 4000},
            "max_runtime_minutes": 60,
            "max_batches_per_tf": 8,
            "wf_settings_by_tf": {
                "M15": {"train_window": 3000, "test_window": 700, "step_size": 500, "top_n_train": 2},
                "M30": {"train_window": 2400, "test_window": 600, "step_size": 400, "top_n_train": 2},
                "H1":  {"train_window": 1800, "test_window": 400, "step_size": 300, "top_n_train": 2},
                "H4":  {"train_window": 650,  "test_window": 100, "step_size": 80,  "top_n_train": 2},
            },
        },
    },
    # Medium multi-timeframe sampled run. Use after SAMPLED finds plausible candidates.
    "SAMPLED_MULTI_TF": {
        "timeframes": ["M15", "M30", "H1", "H4"],
        "batch_size": 50,
        "max_params_per_combo": 8,
        "settings_samples": 1,
        "total_configs_cap": 400,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,  # stricter multi-TF sampled gate
        "drop_atr_from_entries": False,
        "max_data_rows_by_tf": {"M15": 12000, "M30": 12000, "H1": 8000, "H4": 5000},
        "max_runtime_minutes": 90,
        "max_batches_per_tf": 8,
    },
    # Previous heavier sampled profile retained for deeper runs only.
    "SAMPLED_DEEP": {
        "timeframes": ["M15", "M30", "H1", "H4"],
        "batch_size": 200,
        "max_params_per_combo": 30,
        "settings_samples": 2,
        "total_configs_cap": 3000,
        "max_filters": 2,
        "filter_samples": 4,
        "min_trades": 30,
        "drop_atr_from_entries": False,
        "max_data_rows_by_tf": {"M15": 24000, "M30": 18000, "H1": 12000, "H4": 8000},
        "max_runtime_minutes": 150,
        "max_batches_per_tf": None,
    },
    "FULL": {
        "timeframes": ["M15", "M30", "H1", "H4"],
        "batch_size": 300,
        "max_params_per_combo": None,  # full Cartesian per combo; can be very large
        "settings_samples": None,      # full exit/sizing/filter grid; can be very large
        "total_configs_cap": None,
        "max_filters": 5,
        "filter_samples": None,
        "min_trades": 50,  # stricter gate for final full-sample ranking
        "drop_atr_from_entries": False,
        "max_data_rows": None,
        "max_runtime_minutes": None,
        "max_batches_per_tf": None,
    },
    "WALK_FORWARD": {
        "timeframes": ["M15", "M30", "H1", "H4"],
        "batch_size": 50,
        "max_params_per_combo": 8,
        "settings_samples": 1,
        "total_configs_cap": 800,
        "max_filters": 1,
        "filter_samples": 2,
        "min_trades": 30,  # OOS aggregate should have enough total trades
        "drop_atr_from_entries": False,
        "wf_train_window": 3000,
        "wf_test_window": 500,
        "wf_step_size": 500,
        "wf_top_n_train": 3,
        "max_data_rows_by_tf": {"M15": 24000, "M30": 18000, "H1": 12000, "H4": 8000},
        "max_runtime_minutes": 120,
        "max_batches_per_tf": 10,
    },
}

if RUN_MODE not in MODE_SETTINGS:
    raise ValueError(f"Unsupported RUN_MODE={RUN_MODE}. Choose one of: {list(MODE_SETTINGS)}")

CFG = MODE_SETTINGS[RUN_MODE]
ACTIVE_TIMEFRAMES = globals().get("ACTIVE_TIMEFRAMES_OVERRIDE", CFG["timeframes"])
BATCH_SIZE = CFG["batch_size"]
MAX_PARAMS_PER_COMBO = CFG["max_params_per_combo"]
SETTINGS_SAMPLES = CFG["settings_samples"]
TOTAL_CONFIGS_CAP = CFG["total_configs_cap"]
MAX_FILTERS = CFG["max_filters"]
FILTER_SAMPLES = CFG["filter_samples"]
MIN_TRADES = CFG["min_trades"]
DROP_ATR_FROM_ENTRIES = CFG["drop_atr_from_entries"]
MAX_DATA_ROWS = CFG.get("max_data_rows")
MAX_DATA_ROWS_BY_TF = CFG.get("max_data_rows_by_tf", {})
MAX_RUNTIME_MINUTES = CFG.get("max_runtime_minutes")
MAX_BATCHES_PER_TF = CFG.get("max_batches_per_tf")
RESUME_EXISTING_BATCHES = globals().get("RESUME_EXISTING_BATCHES", True)
# In SMOKE mode, stopped-early rows are allowed so the engine can be inspected.
# In SAMPLED/FULL/WALK_FORWARD, stopped-early rows are filtered out from CLEAN results.
ALLOW_STOPPED_EARLY_IN_CLEAN = (RUN_MODE == "SMOKE")
MIN_RELIABLE_TRADES_FOR_REVIEW = 30
# When risk-percent sizing produces a raw lot below broker min_lot, skip the trade instead of forcing min_lot.
# This prevents a 1% risk setup from accidentally risking more than planned on small accounts.
SKIP_RISK_LOT_BELOW_MIN = globals().get("SKIP_RISK_LOT_BELOW_MIN", True)
SOFT_KEEP_TOP_N = 5
SKIP_INDICATORS = set()
FOCUSED_INDICATOR_COMBOS = None  # Optional list of indicator-family tuples used by focused WFA.
DEBUG_SIGNAL_COUNTS = bool(globals().get("DEBUG_SIGNAL_COUNTS", False))

# V3.9 holdout / runtime-state globals.
# In AUTO_SAMPLE_THEN_WF, Phase 1 samples only the earlier segment of each capped TF dataset.
# Exact Forward/WFA uses the later holdout segment to reduce selection leakage.
AUTO_HOLDOUT_ENABLED = False
AUTO_FORWARD_RAW_BY_TF = {}
AUTO_SAMPLE_FORWARD_RATIO = float(CFG.get("sample_forward_split_ratio", 0.70))
PROFILE_RUNTIME_GUARD_HIT = False
LAST_PROFILE_COMPLETED = True

# Quality gates are applied after the basic CLEAN filter. They keep weak sampled/full candidates
# from being promoted just because they are the least-bad rows in a batch.
QUALITY_GATES = {
    "SMOKE":            {"min_net_profit": -np.inf, "min_profit_factor": 0.00, "max_dd_pct": 0.30, "min_sharpe": -999.0},
    "SAMPLED":          {"min_net_profit": 0.00,    "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "SAMPLED_H1":       {"min_net_profit": 0.00,    "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "SAMPLED_M15":      {"min_net_profit": 0.00,    "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "SAMPLED_MULTI_TF": {"min_net_profit": 0.00,    "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "SAMPLED_ALL_TF":   {"min_net_profit": 0.00,    "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "AUTO_SAMPLE_THEN_WF": {"min_net_profit": 0.00, "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "SAMPLED_DEEP":     {"min_net_profit": 0.00,    "min_profit_factor": 1.20, "max_dd_pct": 0.15, "min_sharpe": 0.50},
    "FULL":             {"min_net_profit": 0.00,    "min_profit_factor": 1.25, "max_dd_pct": 0.15, "min_sharpe": 0.60},
    "WALK_FORWARD":     {"min_net_profit": 0.00,    "min_profit_factor": 1.15, "max_dd_pct": 0.20, "min_sharpe": 0.30},
}


def _apply_quality_gate(df: pd.DataFrame, run_mode: str, verbose: bool = False, label: str = "") -> pd.DataFrame:
    """Apply post-CLEAN quality gates for non-smoke runs."""
    if df is None or df.empty:
        return pd.DataFrame()
    mode = str(run_mode).upper()
    if mode == "SMOKE":
        return df.reset_index(drop=True)
    gate = QUALITY_GATES.get(mode, QUALITY_GATES["SAMPLED"])
    out = df.copy()
    before = len(out)

    if "Net Profit" in out.columns:
        out = out[out["Net Profit"] >= float(gate["min_net_profit"])]
    if "Profit Factor" in out.columns:
        pf = pd.to_numeric(out["Profit Factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out = out[pf >= float(gate["min_profit_factor"])]
    if "Max Drawdown %" in out.columns:
        dd = pd.to_numeric(out["Max Drawdown %"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out = out[dd <= float(gate["max_dd_pct"])]
    if "Sharpe Ratio" in out.columns:
        sh = pd.to_numeric(out["Sharpe Ratio"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out = out[sh >= float(gate["min_sharpe"])]
    if "Stopped Early" in out.columns:
        out = out[out["Stopped Early"] == False]
    if mode == "WALK_FORWARD" and "OOS Pass Rule" in out.columns:
        out = out[out["OOS Pass Rule"] == True]

    out = out.reset_index(drop=True)
    if verbose and before != len(out):
        print(
            f"🧹 Quality gate{(' '+label) if label else ''}: {before} → {len(out)} | "
            f"PF>={gate['min_profit_factor']}, DD<={gate['max_dd_pct']:.0%}, "
            f"Sharpe>={gate['min_sharpe']}, NetProfit>={gate['min_net_profit']}"
        )
    return out


# Explicit friction units for MT5-style backtests.
# Default spread/commission/swap are sourced from config/xm_micro_gold.json.
# To override without editing this cell, define CUSTOM_FRICTION_SETTINGS before running Cell 18.
ACTIVE_FRICTION_SETTINGS = globals().get("CUSTOM_FRICTION_SETTINGS", [{
    "cost_value_mode": XAUUSD_SPEC["cost_value_mode"],
    "slippage_mode": "fixed",
    "slippage_value": 10,
    "spread_points": XAUUSD_SPEC["spread_points"],
    "commission_per_lot_round_turn": XAUUSD_SPEC["commission_per_lot_round_turn"],
    "swap_per_lot": XAUUSD_SPEC["swap_per_lot"],
    "swap_long_per_lot": XAUUSD_SPEC["swap_long_per_lot"],
    "swap_short_per_lot": XAUUSD_SPEC["swap_short_per_lot"],
    # This is overwritten per timeframe by _with_timeframe_friction().
    "bars_per_day": 96,
}])


print(f"🚀 Starting XAUUSD pipeline | run_mode={RUN_MODE} | run_id={RUN_ID}")
print(f"📌 Active timeframes: {ACTIVE_TIMEFRAMES}")
print(f"📌 Runtime guard: max_runtime_minutes={MAX_RUNTIME_MINUTES} | max_batches_per_tf={MAX_BATCHES_PER_TF} | resume={RESUME_EXISTING_BATCHES}")
print(f"📌 Batch/config: batch_size={BATCH_SIZE} | cap={TOTAL_CONFIGS_CAP} | max_params_per_combo={MAX_PARAMS_PER_COMBO} | settings_samples={SETTINGS_SAMPLES} | max_filters={MAX_FILTERS}")
print(f"📌 Data row cap: max_data_rows={MAX_DATA_ROWS} | by_tf={MAX_DATA_ROWS_BY_TF}")
print(f"📌 Execution mode: {EXECUTION_MODE} | exit policy: {SAME_BAR_EXIT_POLICY}")

# ============================================================
# 1) PARAM ADAPTER + INDICATOR HELPERS
# ============================================================
REQUIRED = {
    "EMA":            ["fast", "slow"],
    "MACD":           ["fast", "slow", "signal"],
    "RSI":            ["period", "overbought", "oversold"],
    "BollingerBands": ["period", "mult"],
    "ATR":            ["period"],
    "Stochastic":     ["k", "d"],
    "Fibonacci":      [],
}
ALIASES = {
    "fast":   ["fast", "fast_period", "short", "short_period"],
    "slow":   ["slow", "slow_period", "long", "long_period"],
    "signal": ["signal", "signal_period"],
    "period": ["period", "length", "window", "n"],
    "mult":   ["mult", "std", "stddev", "dev", "multiplier"],
    "k":      ["k", "k_period", "%k", "k_len"],
    "d":      ["d", "d_period", "%d", "d_len"],
    "smooth": ["smooth", "smooth_k", "smoothing"],
    "overbought": ["overbought", "upper", "upper_band", "ob", "rsi_overbought", "rsi_upper"],
    "oversold":   ["oversold", "lower", "lower_band", "os", "rsi_oversold", "rsi_lower"],
    "levels":     ["levels", "fib_levels"],
}
DEFAULTS = {
    "EMA":             {"fast": 12, "slow": 26},
    "MACD":            {"fast": 12, "slow": 26, "signal": 9},
    "RSI":             {"period": 14, "overbought": 70, "oversold": 30},
    "BollingerBands":  {"period": 20, "mult": 2.0},
    "ATR":             {"period": 14, "multiplier": 2.0},
    "Stochastic":      {"k": 14, "d": 3, "smooth": 3, "overbought": 80, "oversold": 20},
    "Fibonacci":       {"levels": [23.6, 38.2, 50.0, 61.8, 78.6]},
}


def _adapt_params(params: Optional[dict]) -> dict:
    """Normalize mixed key names such as stddev -> mult, fast_period -> fast."""
    out = copy.deepcopy(params) if isinstance(params, dict) else {}
    for ind in list(REQUIRED.keys()) + ["ATR"]:
        out.setdefault(ind, {})
        if not isinstance(out[ind], dict):
            out[ind] = {}

    for ind, defaults in DEFAULTS.items():
        src = out.get(ind, {}) or {}
        dst = dict(src)
        needed_keys = set(REQUIRED.get(ind, [])) | set(defaults.keys())
        for key in needed_keys:
            if key in dst:
                continue
            found_key = None
            for alias in ALIASES.get(key, [key]):
                if alias in src:
                    found_key = alias
                    break
                for k in src.keys():
                    if str(k).lower().replace("-", "_") == str(alias).lower().replace("-", "_"):
                        found_key = k
                        break
                if found_key:
                    break
            value = src.get(found_key) if found_key else defaults.get(key)
            if value is not None:
                dst[key] = value
        out[ind] = dst

    return out


def _ema(series: pd.Series, span: int) -> pd.Series:
    return pd.Series(series, index=series.index).ewm(span=int(span), adjust=False).mean()


def _rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """RSI with Wilder-style smoothing, closer to MT5 iRSI than a simple rolling mean."""
    period = int(period)
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi = rsi.where(avg_loss != 0, 100.0)
    rsi = rsi.where(avg_gain != 0, 0.0)
    return rsi


def _macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    ema_fast = _ema(close, fast)
    ema_slow = _ema(close, slow)
    line = ema_fast - ema_slow
    sig = _ema(line, signal)
    hist = line - sig
    return line, sig, hist


def _bollinger(close: pd.Series, period: int = 20, mult: float = 2.0):
    ma = close.rolling(int(period)).mean()
    sd = close.rolling(int(period)).std(ddof=0)
    return ma, ma + float(mult) * sd, ma - float(mult) * sd


def _atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    """ATR with Wilder-style smoothing, closer to MT5 iATR than a simple rolling mean."""
    period = int(period)
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()


def _stochastic(high: pd.Series, low: pd.Series, close: pd.Series, k: int = 14, d: int = 3, smooth: Optional[int] = 3):
    k = int(k)
    d = int(d)
    ll = low.rolling(k).min()
    hh = high.rolling(k).max()
    raw_k = (close - ll) / (hh - ll).replace(0, np.nan) * 100
    if smooth not in (None, "", False) and int(smooth) > 1:
        raw_k = raw_k.rolling(int(smooth)).mean()
    dline = raw_k.rolling(d).mean()
    return raw_k, dline


def _standardize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    df2.columns = [str(c).strip().lower() for c in df2.columns]
    rename_map = {"tickvol": "volume", "vol": "volume"}
    df2 = df2.rename(columns=rename_map)
    required_cols = ["open", "high", "low", "close"]
    missing = [c for c in required_cols if c not in df2.columns]
    if missing:
        raise ValueError(f"Missing OHLC columns: {missing}")
    if "volume" not in df2.columns:
        df2["volume"] = 0.0
    if not isinstance(df2.index, pd.DatetimeIndex):
        df2.index = pd.to_datetime(df2.index, errors="coerce")
    df2 = df2[~df2.index.isna()].sort_index()
    return df2



def _params_cache_key(params: Optional[dict]) -> str:
    """Stable-ish key used to avoid recalculating indicators multiple times for the same config."""
    try:
        return json.dumps(_adapt_params(params or {}), sort_keys=True, default=str)
    except Exception:
        return repr(_adapt_params(params or {}))

def _ensure_indicator_columns(df: pd.DataFrame, params: Optional[dict]) -> pd.DataFrame:
    """
    Recalculate columns on a copy using this strategy's params.
    This avoids the old bug where default precomputed columns were reused for non-default params.
    Uses a DataFrame attr cache key to avoid repeated recalculation inside composite entries.
    """
    p = _adapt_params(params or {})
    cache_key = _params_cache_key(p)
    if isinstance(df, pd.DataFrame) and df.attrs.get("_xau_indicator_cache_key") == cache_key:
        return df
    df2 = _standardize_ohlcv(df)
    close = df2["close"]
    high = df2["high"]
    low = df2["low"]

    # EMA: create both dynamic names used by old entry funcs and generic names.
    ep = p.get("EMA", {})
    f = int(ep.get("fast", DEFAULTS["EMA"]["fast"]))
    s = int(ep.get("slow", DEFAULTS["EMA"]["slow"]))
    df2[f"ema_{f}"] = _ema(close, f)
    df2[f"ema_{s}"] = _ema(close, s)
    df2["ema_fast"] = df2[f"ema_{f}"]
    df2["ema_slow"] = df2[f"ema_{s}"]
    if "ema_200" not in df2.columns:
        df2["ema_200"] = _ema(close, 200)

    # RSI
    rp = p.get("RSI", {})
    r_period = int(rp.get("period", DEFAULTS["RSI"]["period"]))
    df2[f"rsi_{r_period}"] = _rsi(close, r_period)
    df2["rsi"] = df2[f"rsi_{r_period}"]

    # MACD
    mp = p.get("MACD", {})
    mf = int(mp.get("fast", DEFAULTS["MACD"]["fast"]))
    ms = int(mp.get("slow", DEFAULTS["MACD"]["slow"]))
    msg = int(mp.get("signal", DEFAULTS["MACD"]["signal"]))
    macd_line, macd_signal, macd_hist = _macd(close, mf, ms, msg)
    df2["macd_line"] = macd_line
    df2["macd_signal"] = macd_signal
    df2["macd_histogram"] = macd_hist

    # Bollinger Bands
    bp = p.get("BollingerBands", {})
    b_period = int(bp.get("period", DEFAULTS["BollingerBands"]["period"]))
    b_mult = float(bp.get("mult", DEFAULTS["BollingerBands"]["mult"]))
    bb_mid, bb_upper, bb_lower = _bollinger(close, b_period, b_mult)
    df2["bb_middle"] = bb_mid
    df2["bb_upper"] = bb_upper
    df2["bb_lower"] = bb_lower
    df2[f"bb_upper_{b_period}"] = bb_upper
    df2[f"bb_lower_{b_period}"] = bb_lower

    # ATR
    ap = p.get("ATR", {})
    atr_period = int(ap.get("period", DEFAULTS["ATR"]["period"]))
    df2[f"atr_{atr_period}"] = _atr(high, low, close, atr_period)
    df2["atr"] = df2[f"atr_{atr_period}"]
    atr_ma_period = int(ap.get("ma_period", 50) or 50)
    df2["atr_ma"] = df2["atr"].rolling(atr_ma_period, min_periods=max(5, atr_ma_period // 3)).mean()
    if "atr_14" not in df2.columns:
        df2["atr_14"] = _atr(high, low, close, 14)

    # Stochastic
    sp = p.get("Stochastic", {})
    kk = int(sp.get("k", DEFAULTS["Stochastic"]["k"]))
    dd = int(sp.get("d", DEFAULTS["Stochastic"]["d"]))
    sm = sp.get("smooth", DEFAULTS["Stochastic"].get("smooth", 3))
    st_k, st_d = _stochastic(high, low, close, kk, dd, sm)
    df2[f"stoch_k_{kk}"] = st_k
    df2[f"stoch_d_{dd}"] = st_d  # backward-compatible with old entry funcs
    df2["stoch_k"] = st_k
    df2["stoch_d"] = st_d

    # Fibonacci swing-zone proxy using prior rolling swing high/low.
    # This is stricter than the old rolling min/max proxy and avoids using the current bar as the swing anchor.
    fib_lookback = int(p.get("Fibonacci", {}).get("lookback", 50) or 50)
    swing_high = high.rolling(window=fib_lookback, min_periods=max(10, fib_lookback // 3)).max().shift(1)
    swing_low = low.rolling(window=fib_lookback, min_periods=max(10, fib_lookback // 3)).min().shift(1)
    fib_range = (swing_high - swing_low).replace(0, np.nan)
    df2["fib_swing_high"] = swing_high
    df2["fib_swing_low"] = swing_low
    df2["fib_382"] = swing_high - fib_range * 0.382
    df2["fib_500"] = swing_high - fib_range * 0.500
    df2["fib_618"] = swing_high - fib_range * 0.618
    df2["fib_786"] = swing_high - fib_range * 0.786
    df2["fib_support"] = df2["fib_618"]
    df2["fib_resistance"] = swing_low + fib_range * 0.618
    df2.attrs["_xau_indicator_cache_key"] = cache_key
    return df2




def fibonacci_zone_entry(df: pd.DataFrame, params: Optional[dict] = None) -> pd.Series:
    """Long Fib signal: price reclaims the 61.8% retracement zone with a small tolerance."""
    params = params or {}
    tol = float(params.get("tolerance_pct", 0.003))
    support = df.get("fib_support")
    if support is None:
        return pd.Series(False, index=df.index)
    near_zone = df["low"] <= support * (1 + tol)
    reclaim = (df["close"] > support) & (df["close"] > df["open"])
    return (near_zone & reclaim).fillna(False)


def fibonacci_zone_short_entry(df: pd.DataFrame, params: Optional[dict] = None) -> pd.Series:
    """Short Fib signal: price rejects the 61.8% retracement zone with a small tolerance."""
    params = params or {}
    tol = float(params.get("tolerance_pct", 0.003))
    resistance = df.get("fib_resistance")
    if resistance is None:
        return pd.Series(False, index=df.index)
    near_zone = df["high"] >= resistance * (1 - tol)
    reject = (df["close"] < resistance) & (df["close"] < df["open"])
    return (near_zone & reject).fillna(False)



def atr_volatility_entry(df: pd.DataFrame, params: Optional[dict] = None) -> pd.Series:
    """
    ATR is non-directional, so this acts as a volatility-regime confirmation.
    It can participate in 2–7 indicator combinations without pretending to predict direction.
    """
    params = params or {}
    atr = df.get("atr")
    atr_ma = df.get("atr_ma")
    if atr is None or atr_ma is None:
        return pd.Series(False, index=df.index)
    min_ratio = float(params.get("min_ratio", 0.85))
    max_ratio = float(params.get("max_ratio", 2.50))
    ratio = atr / atr_ma.replace(0, np.nan)
    return ((ratio >= min_ratio) & (ratio <= max_ratio)).fillna(False)


def atr_volatility_short_entry(df: pd.DataFrame, params: Optional[dict] = None) -> pd.Series:
    # Same volatility confirmation for shorts; direction comes from the other indicators in the combo.
    return atr_volatility_entry(df, params=params)

# Override the loose legacy Fibonacci functions from Step 4 and add ATR as an entry-capable volatility filter.
if "entry_functions" in globals():
    entry_functions["Fibonacci"] = fibonacci_zone_entry
    entry_functions["ATR"] = atr_volatility_entry
if "short_entry_functions" in globals():
    short_entry_functions["Fibonacci"] = fibonacci_zone_short_entry
    short_entry_functions["ATR"] = atr_volatility_short_entry

# ============================================================
# 2) ENTRY BINDING
# ============================================================
def call_entry_fn(fn, ind_key: Optional[str], df: pd.DataFrame, params: Optional[dict]):
    p_adapted = _adapt_params(params or {})
    cache_key = _params_cache_key(p_adapted)
    if isinstance(df, pd.DataFrame) and df.attrs.get("_xau_indicator_cache_key") == cache_key:
        df2 = df
    else:
        df2 = _ensure_indicator_columns(df, p_adapted)
    shapes: List[dict] = []
    if ind_key:
        shapes.append(p_adapted.get(ind_key, {}))
        shapes.append({ind_key: p_adapted.get(ind_key, {})})
        shapes.append(p_adapted)
    else:
        shapes.append(p_adapted)

    last_err = None
    for shp in shapes:
        try:
            sig = fn(df2, shp)
            return pd.Series(sig, index=df2.index).fillna(False).astype(bool)
        except Exception as e:
            last_err = e
    raise last_err if last_err else RuntimeError("Cannot call entry function")


def _norm_combo_key(combo: Sequence[str]) -> str:
    return "_".join(sorted(map(str, combo)))


def _compose_signals(pairs: List[Tuple[Any, str]], mode: str = "AND", min_confirmations: Optional[int] = None):
    mode = str(mode or "AND").upper()

    def _run(df: pd.DataFrame, params: Optional[dict] = None):
        signals = []
        for fn, ind_key in pairs:
            sig = call_entry_fn(fn, ind_key, df, params or {})
            signals.append(sig)
        if not signals:
            return pd.Series(False, index=df.index)
        mat = pd.concat(signals, axis=1).fillna(False).astype(bool)
        if mode == "OR":
            return mat.any(axis=1)
        if mode == "VOTE":
            required = min_confirmations
            if required is None:
                required = int(math.ceil(len(signals) / 2.0))
            return mat.sum(axis=1) >= int(required)
        return mat.all(axis=1)

    return _run


def _resolve_entry_pair(entry_functions, short_entry_functions, combo, entry_logic=None, skip_indicators=frozenset()):
    entry_logic = entry_logic or {}
    mode = entry_logic.get("mode", ENTRY_MODE)
    min_conf = entry_logic.get("min_confirmations", MIN_CONFIRMATIONS)

    if isinstance(entry_functions, dict):
        key_norm = _norm_combo_key(combo)
        for k, v in entry_functions.items():
            if isinstance(v, dict) and "long" in v and "short" in v:
                if set(str(k).replace("|", "_").upper().split("_")) == set(key_norm.upper().split("_")):
                    return v["long"], v["short"], str(k)

    long_pairs, short_pairs, names = [], [], []
    for ind in combo:
        if ind in skip_indicators:
            return None, None, None
        if ind not in entry_functions or ind not in short_entry_functions:
            return None, None, None
        long_pairs.append((entry_functions[ind], ind))
        short_pairs.append((short_entry_functions[ind], ind))
        names.append(ind)

    strat_name = f"{mode}_" + "_".join(sorted(names))
    return _compose_signals(long_pairs, mode, min_conf), _compose_signals(short_pairs, mode, min_conf), strat_name


def rebind_config_from_pickle(saved_cfg, entry_functions, short_entry_functions, skip_indicators=frozenset()):
    cfg = copy.deepcopy(saved_cfg)
    combo = tuple(cfg.get("entry_indicators", []) or cfg.get("indicators", []) or [])
    long_fn, short_fn, strat_name = _resolve_entry_pair(
        entry_functions,
        short_entry_functions,
        combo,
        entry_logic=cfg.get("entry_logic", {}),
        skip_indicators=skip_indicators,
    )
    if long_fn is None or short_fn is None:
        raise ValueError(f"Cannot rebind entry for combo={combo}")

    params = _adapt_params(cfg.get("params", {}))

    def _long_bound(df, fn=long_fn, p=params):
        return fn(df, p)

    def _short_bound(df, fn=short_fn, p=params):
        return fn(df, p)

    cfg["params"] = params
    cfg["entry"] = {
        "long_condition": _long_bound,
        "short_condition": _short_bound,
        "indicators": combo,
        "strategy_name": strat_name or cfg.get("entry_strategy_name") or "_".join(combo),
    }
    return cfg


# ============================================================
# 3) BROKER / COST / SIZING HELPERS
# ============================================================
def _to_price_units(value: float, spec: dict, friction: dict) -> float:
    if str(friction.get("cost_value_mode", spec.get("cost_value_mode", "price"))).lower() == "points":
        return float(value) * float(spec.get("point", 0.01))
    return float(value)


def _normalize_lot(lot: Any, spec: dict = XAUUSD_SPEC) -> float:
    min_lot = float(spec["min_lot"])
    max_lot = float(spec["max_lot"])
    step = float(spec["lot_step"])
    precision = int(spec.get("lot_precision", 2))
    if step <= 0:
        raise ValueError("lot_step must be positive")
    try:
        lot = float(lot)
    except Exception:
        return 0.0
    if not np.isfinite(lot) or lot < min_lot:
        return 0.0
    lot = min(lot, max_lot)
    steps = int((lot - min_lot) / step + 1e-12)
    quantized = min_lot + steps * step
    return round(min(max_lot, quantized), precision)


def _spread_price(friction: dict, spec: dict) -> float:
    return abs(_to_price_units(friction.get("spread_points", 0.0), spec, friction))


def _apply_execution_price(price: float, side: str, friction: dict, spec: dict, timestamp=None) -> Tuple[float, dict]:
    """
    Convert raw OHLC price into executable price.

    Default assumes MT5 OHLC is Bid:
    - Buy executes at Ask = Bid + spread + slippage
    - Sell executes at Bid - slippage

    If your exported OHLC is Mid, set XAUUSD_SPEC["ohlc_price_source"] = "mid".
    """
    side = str(side).lower()
    price = float(price)
    slip_mode = friction.get("slippage_mode", "fixed")
    slip_value = friction.get("slippage_value", 0.0)

    if slip_mode == "fixed":
        slip = _to_price_units(slip_value, spec, friction)
    elif slip_mode == "random_normal":
        slip = _to_price_units(np.random.normal(friction.get("slippage_mu", slip_value), friction.get("slippage_sigma", 0.0)), spec, friction)
    elif slip_mode == "random_uniform":
        slip = _to_price_units(np.random.uniform(friction.get("slippage_low", 0.0), friction.get("slippage_high", slip_value)), spec, friction)
    elif slip_mode == "time_sensitive":
        hour = pd.to_datetime(timestamp).hour if timestamp is not None else 12
        raw = slip_value * 2 if 8 <= hour <= 17 else slip_value
        slip = _to_price_units(raw, spec, friction)
    else:
        slip = 0.0
    slip = abs(float(slip))

    spread = _spread_price(friction, spec)
    source = str(spec.get("ohlc_price_source", friction.get("ohlc_price_source", "bid"))).lower()

    if source == "bid":
        if side == "buy":
            executed = price + spread + slip
            spread_component = spread
        elif side == "sell":
            executed = price - slip
            spread_component = 0.0
        else:
            raise ValueError("side must be 'buy' or 'sell'")
    elif source == "ask":
        if side == "buy":
            executed = price + slip
            spread_component = 0.0
        elif side == "sell":
            executed = price - spread - slip
            spread_component = spread
        else:
            raise ValueError("side must be 'buy' or 'sell'")
    else:
        spread_component = spread / 2.0 if spec.get("spread_application", "half") == "half" else spread
        if side == "buy":
            executed = price + spread_component + slip
        elif side == "sell":
            executed = price - spread_component - slip
        else:
            raise ValueError("side must be 'buy' or 'sell'")

    return float(executed), {"slippage": slip, "spread_component": spread_component, "spread_total": spread, "ohlc_price_source": source}


def _commission_per_side(lot: float, friction: dict) -> float:
    rt_rate = friction.get("commission_per_lot_round_turn", friction.get("commission_per_lot", 0.0))
    return float(lot) * float(rt_rate) / 2.0


def _swap_cash(lot: float, bars_held: int, friction: dict) -> float:
    # negative = cost, positive = credit
    daily_swap = float(friction.get("swap_per_lot", 0.0))
    bars_per_day = max(float(friction.get("bars_per_day", 96.0)), 1.0)
    days_held = float(bars_held) / bars_per_day
    return float(lot) * daily_swap * days_held


def _gross_pnl(entry_price: float, exit_price: float, lot: float, direction: str, spec: dict) -> float:
    contract_size = float(spec["contract_size"])
    raw = (float(exit_price) - float(entry_price)) * float(lot) * contract_size
    return raw if direction == "long" else -raw


def _calculate_lot(capital: float, entry_price: float, stop_loss: float, sizing_cfg: dict, atr_value: Optional[float] = None, spec: dict = XAUUSD_SPEC) -> float:
    """
    Calculate executable lot size.

    V3.4 change:
    - For risk_percent sizing, if the raw risk-based lot is below broker min_lot and
      SKIP_RISK_LOT_BELOW_MIN=True, return 0.0 so the trade is skipped instead of
      being forced to min_lot and exceeding the intended risk percentage.
    """
    cfg = {**spec, **(sizing_cfg or {})}
    if atr_value is not None:
        cfg["atr"] = atr_value

    method = str(cfg.get("sizing_method", "fixed") or "fixed").lower()
    min_lot = float(cfg.get("min_lot", spec["min_lot"]))
    contract_size = float(cfg.get("contract_size", spec["contract_size"]))

    if method == "risk_percent":
        sl_distance = abs(float(entry_price) - float(stop_loss))
        if sl_distance <= 0 or not np.isfinite(sl_distance):
            return 0.0
        risk_percent = float(cfg.get("risk_percent", 1.0))
        risk_amount = (risk_percent / 100.0) * float(capital)
        lot_raw = risk_amount / (sl_distance * contract_size)
        if not np.isfinite(lot_raw) or lot_raw <= 0:
            return 0.0
        skip_below_min = bool(cfg.get("skip_if_lot_below_min", SKIP_RISK_LOT_BELOW_MIN))
        if skip_below_min and lot_raw < min_lot:
            return 0.0
        return _normalize_lot(lot_raw, cfg)

    try:
        lot_raw = calculate_position_size(
            capital,
            entry_price,
            stop_loss,
            sizing_method=method,
            config=cfg,
        )
    except Exception:
        lot_raw = cfg.get("fixed_lot", cfg.get("base_lot_size", spec["min_lot"]))

    try:
        if not np.isfinite(float(lot_raw)) or float(lot_raw) <= 0:
            return 0.0
    except Exception:
        return 0.0
    return _normalize_lot(lot_raw, cfg)


# ============================================================
# 4) STOP / TARGET / FILTER HELPERS
# ============================================================
def _calculate_structure_stop_safe(df: pd.DataFrame, idx: int, direction: str, window: int = 20) -> float:
    start = max(0, idx - int(window))
    if direction == "long":
        return float(df["low"].iloc[start:idx + 1].min())
    return float(df["high"].iloc[start:idx + 1].max())


def _calculate_fib_target_safe(entry_price: float, df: pd.DataFrame, idx: int, direction: str, fib_levels=None) -> float:
    fib_levels = fib_levels or [1.618]
    level = float(fib_levels[0]) if isinstance(fib_levels, (list, tuple)) else float(fib_levels)
    lookback = 50
    start = max(0, idx - lookback)
    swing_high = float(df["high"].iloc[start:idx + 1].max())
    swing_low = float(df["low"].iloc[start:idx + 1].min())
    rng = max(abs(swing_high - swing_low), float(XAUUSD_SPEC.get("point", 0.01)))
    # If levels are 23.6/38.2 etc, use first extension proxy. If 1.618/2.0, use as extension.
    ext = level / 100.0 if level > 10 else level
    if ext < 1:
        ext = 1 + ext
    return float(entry_price + rng * ext) if direction == "long" else float(entry_price - rng * ext)


def _valid_stop_target(entry: float, sl: float, tp: float, direction: str) -> bool:
    if not all(np.isfinite([entry, sl, tp])):
        return False
    if direction == "long":
        return sl < entry < tp
    return tp < entry < sl


def _build_filter_combinations(max_filters: int = 0, samples: Optional[int] = None) -> List[dict]:
    names = ["trend_filter", "volatility_filter", "volume_filter", "session_filter", "time_filter"]
    combos = [{}]
    max_n = min(int(max_filters or 0), len(names))
    for r in range(1, max_n + 1):
        for subset in itertools.combinations(names, r):
            combos.append({f"use_{name}": True for name in subset})
    if samples is not None and len(combos) > samples:
        # Always include no-filter baseline plus a deterministic sample of active filters.
        base = [combos[0]]
        rest = combos[1:]
        random.shuffle(rest)
        combos = base + rest[:max(0, samples - 1)]
    return combos


FILTER_ERROR_POLICY = "fail_closed"  # fail_closed is safer for production; switch to "pass_open" only for debugging.
_FILTER_WARN_COUNT = 0


def _safe_passes_filters(df: pd.DataFrame, idx: int, filters: dict, filter_params: Optional[dict] = None) -> bool:
    global _FILTER_WARN_COUNT
    if not filters:
        return True
    try:
        return bool(passes_all_filters(df, idx, filters, filter_params=filter_params))
    except Exception as e:
        _FILTER_WARN_COUNT += 1
        if _FILTER_WARN_COUNT <= 5:
            print(f"⚠️ filter evaluation error at idx={idx}: {e}. Policy={FILTER_ERROR_POLICY}")
        return False if FILTER_ERROR_POLICY == "fail_closed" else True


def _max_drawdown_pct_from_equity(equity: Sequence[float]) -> float:
    if not equity:
        return 0.0
    s = pd.Series(equity, dtype=float)
    peak = s.cummax().replace(0, np.nan)
    dd_pct = ((peak - s) / peak).max()
    return float(0.0 if pd.isna(dd_pct) else dd_pct)


# ============================================================
# 5) FINAL BACKTEST ENGINE V3
# ============================================================
def _get_past_indicator_value(df: pd.DataFrame, col: str, i: int) -> Optional[float]:
    """Return the latest available indicator value up to bar i only. Never looks into future bars."""
    if col not in df.columns:
        return None
    past = pd.Series(df[col].iloc[: i + 1]).replace([np.inf, -np.inf], np.nan).dropna()
    if past.empty:
        return None
    val = float(past.iloc[-1])
    return val if np.isfinite(val) and val > 0 else None


def _intrabar_stop_target(
    open_pos: dict,
    bar_open: float,
    bar_high: float,
    bar_low: float,
    friction: Optional[dict] = None,
    broker_spec: Optional[dict] = None,
) -> Tuple[Optional[float], Optional[str]]:
    """
    Conservative SL/TP detection for one OHLC bar.

    Important: this function returns the RAW OHLC-side price expected by
    _close_open_position(), not the already-executed close price.

    V3.4 change:
    - If the stop was moved by trailing logic, SL exits are labelled as
      Trailing_SL / Trailing_SL_Gap / Trailing_SL_same_bar for easier review.
    """
    broker_spec = broker_spec or XAUUSD_SPEC
    friction = friction or broker_spec
    direction = open_pos["direction"]
    sl = float(open_pos["stop_loss"])
    tp = float(open_pos["take_profit"])
    source = str(broker_spec.get("ohlc_price_source", friction.get("ohlc_price_source", "bid"))).lower()
    spread = _spread_price(friction, broker_spec)
    sl_label = "Trailing_SL" if bool(open_pos.get("stop_loss_is_trailing", False)) else "SL"

    raw_open, raw_high, raw_low = float(bar_open), float(bar_high), float(bar_low)

    if direction == "long":
        # Long positions close via sell. With bid OHLC, raw prices are already close-trigger prices.
        if source == "ask":
            trig_open, trig_high, trig_low = raw_open - spread, raw_high - spread, raw_low - spread
            raw_from_trigger = lambda trigger_price: float(trigger_price + spread)
        elif source == "mid":
            offset = spread / 2.0
            trig_open, trig_high, trig_low = raw_open - offset, raw_high - offset, raw_low - offset
            raw_from_trigger = lambda trigger_price: float(trigger_price + offset)
        else:
            trig_open, trig_high, trig_low = raw_open, raw_high, raw_low
            raw_from_trigger = lambda trigger_price: float(trigger_price)

        if trig_open <= sl:
            return raw_open, f"{sl_label}_Gap"
        if trig_open >= tp:
            return raw_open, "TP_Gap"

        hit_sl = trig_low <= sl
        hit_tp = trig_high >= tp
        if hit_sl and hit_tp:
            if SAME_BAR_EXIT_POLICY.upper() == "TP_FIRST":
                return raw_from_trigger(tp), "TP_same_bar"
            return raw_from_trigger(sl), f"{sl_label}_same_bar"
        if hit_sl:
            return raw_from_trigger(sl), sl_label
        if hit_tp:
            return raw_from_trigger(tp), "TP"
        return None, None

    # Short positions close via buy. With bid OHLC, trigger is approximate ask = bid + spread,
    # but _close_open_position() expects bid raw and will add spread once.
    if source == "bid":
        trig_open, trig_high, trig_low = raw_open + spread, raw_high + spread, raw_low + spread
        raw_from_trigger = lambda trigger_price: float(trigger_price - spread)
    elif source == "mid":
        offset = spread / 2.0
        trig_open, trig_high, trig_low = raw_open + offset, raw_high + offset, raw_low + offset
        raw_from_trigger = lambda trigger_price: float(trigger_price - offset)
    else:
        trig_open, trig_high, trig_low = raw_open, raw_high, raw_low
        raw_from_trigger = lambda trigger_price: float(trigger_price)

    if trig_open >= sl:
        return raw_open, f"{sl_label}_Gap"
    if trig_open <= tp:
        return raw_open, "TP_Gap"

    hit_sl = trig_high >= sl
    hit_tp = trig_low <= tp
    if hit_sl and hit_tp:
        if SAME_BAR_EXIT_POLICY.upper() == "TP_FIRST":
            return raw_from_trigger(tp), "TP_same_bar"
        return raw_from_trigger(sl), f"{sl_label}_same_bar"
    if hit_sl:
        return raw_from_trigger(sl), sl_label
    if hit_tp:
        return raw_from_trigger(tp), "TP"
    return None, None


def _mark_to_market_equity(
    cash: float,
    open_pos: Optional[dict],
    mark_price_raw: float,
    broker_spec: dict,
    friction: dict,
    timestamp=None,
    bars_held: int = 0,
) -> float:
    """Equity = realized cash + conservative liquidation value of any open position."""
    if open_pos is None:
        return float(cash)
    direction = open_pos["direction"]
    exit_side = "sell" if direction == "long" else "buy"
    exit_exec, _ = _apply_execution_price(mark_price_raw, exit_side, friction, broker_spec, timestamp=timestamp)
    gross = _gross_pnl(open_pos["entry_price"], exit_exec, open_pos["lot"], direction, broker_spec)
    exit_commission = _commission_per_side(open_pos["lot"], friction)
    swap_cash = _swap_cash(open_pos["lot"], bars_held, friction)
    return float(cash + gross - exit_commission + swap_cash)


def _append_equity(equity: List[float], cash: float, open_pos: Optional[dict], mark_price_raw: float, broker_spec: dict, friction: dict, timestamp=None, bars_held: int = 0):
    equity.append(_mark_to_market_equity(cash, open_pos, mark_price_raw, broker_spec, friction, timestamp=timestamp, bars_held=bars_held))


def _close_open_position(open_pos: dict, exit_raw: float, exit_reason: str, i: int, ts, cash: float, friction: dict, broker_spec: dict) -> Tuple[float, dict]:
    direction = open_pos["direction"]
    exit_side = "sell" if direction == "long" else "buy"
    exit_exec, _ = _apply_execution_price(exit_raw, exit_side, friction, broker_spec, timestamp=ts)
    bars_held = max(0, int(i - open_pos["entry_idx"]))
    exit_commission = _commission_per_side(open_pos["lot"], friction)
    swap_cash = _swap_cash(open_pos["lot"], bars_held, friction)
    gross = _gross_pnl(open_pos["entry_price"], exit_exec, open_pos["lot"], direction, broker_spec)
    trade_pnl = gross - open_pos["entry_commission"] - exit_commission + swap_cash
    cash = float(cash + gross - exit_commission + swap_cash)
    trade = {
        "entry": open_pos["entry_price"],
        "exit": exit_exec,
        "entry_raw": open_pos["entry_raw"],
        "exit_raw": float(exit_raw),
        "entry_time": open_pos["entry_time"],
        "exit_time": ts,
        "signal_time": open_pos["signal_time"],
        "entry_idx": open_pos["entry_idx"],
        "exit_idx": i,
        "lot": open_pos["lot"],
        "direction": direction,
        "gross_pnl": gross,
        "entry_commission": open_pos["entry_commission"],
        "exit_commission": exit_commission,
        "swap_cash": swap_cash,
        "pnl": trade_pnl,
        "bars": bars_held,
        "reason": exit_reason,
        "stop_loss": open_pos["stop_loss"],
        "take_profit": open_pos["take_profit"],
    }
    return cash, trade


def run_backtest(
    config: dict,
    df: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL,
    verbose: bool = False,
    filter_params: Optional[dict] = None,
    broker_spec: dict = XAUUSD_SPEC,
    execution_mode: str = EXECUTION_MODE,
):
    """
    Final V3 backtest engine.
    Key protections:
    - next-bar execution support
    - no future ATR fallback; skips entries before indicator warm-up
    - SL/TP evaluated with OHLC high/low, including the entry bar
    - mark-to-market equity curve for realistic drawdown
    - explicit contract-size PnL and broker-cost units
    """
    df = _ensure_indicator_columns(df, config.get("params", {}))
    if len(df) < 5:
        return [], float(initial_capital), [float(initial_capital)]

    cash = float(initial_capital)
    open_pos = None
    trades: List[dict] = []
    equity: List[float] = []

    long_fn = config["entry"]["long_condition"]
    short_fn = config["entry"]["short_condition"]
    try:
        long_sig = pd.Series(long_fn(df), index=df.index).fillna(False).astype(bool)
    except Exception as e:
        if verbose:
            print("long signal error:", e)
        long_sig = pd.Series(False, index=df.index)
    try:
        short_sig = pd.Series(short_fn(df), index=df.index).fillna(False).astype(bool)
    except Exception as e:
        if verbose:
            print("short signal error:", e)
        short_sig = pd.Series(False, index=df.index)

    if DEBUG_SIGNAL_COUNTS and (str(config.get("id", "")).endswith("_0") or config.get("_debug_first", False)):
        print(
            f"[BT-DEBUG] {config.get('id')}: "
            f"long={int(long_sig.sum())}, short={int(short_sig.sum())}, "
            f"both={int((long_sig & short_sig).sum())}"
        )

    params = _adapt_params(config.get("params", {}))
    exit_cfg = config.get("exit", {}) or {}
    sizing_cfg = config.get("sizing", {}) or {}
    friction = {**broker_spec, **(config.get("friction", {}) or {})}
    filters = config.get("filters", {}) or {}
    max_dd_pct = exit_cfg.get("max_drawdown_pct")

    atr_period = int(exit_cfg.get("atr_period") or params.get("ATR", {}).get("period", DEFAULTS["ATR"]["period"]))
    atr_col = f"atr_{atr_period}"
    if atr_col not in df.columns:
        df[atr_col] = _atr(df["high"], df["low"], df["close"], atr_period)

    # Fast, no-lookahead ATR access. ffill only carries known past ATR values forward.
    atr_values = (
        pd.Series(df[atr_col], index=df.index)
        .replace([np.inf, -np.inf], np.nan)
        .ffill()
        .to_numpy(dtype=float)
    )

    start_i = 1 if execution_mode == "next_bar_open" else 0
    expected_equity_len = max(len(df) - start_i, 1)
    last_i = start_i - 1

    for i in range(start_i, len(df)):
        last_i = i
        row = df.iloc[i]
        ts = row.name
        signal_i = i - 1 if execution_mode == "next_bar_open" else i
        signal_i = max(0, signal_i)
        signal_ts = df.index[signal_i]
        exec_price_base = float(row["open"] if execution_mode == "next_bar_open" and "open" in df.columns else row["close"])
        bar_high = float(row["high"])
        bar_low = float(row["low"])
        bar_close = float(row["close"])
        current_atr = float(atr_values[i]) if i < len(atr_values) and np.isfinite(atr_values[i]) and atr_values[i] > 0 else None

        # ----------------------- manage open position -----------------------
        if open_pos is not None:
            direction = open_pos["direction"]
            bars_held = max(0, i - open_pos["entry_idx"])
            exit_price_raw = None
            exit_reason = None

            # Reversal/time exits are based on the completed signal bar and exit at current executable price.
            use_reversal = bool(exit_cfg.get("use_indicator_exit", exit_cfg.get("exit_on_reversal", False)))
            if use_reversal:
                opposite = bool(short_sig.iloc[signal_i]) if direction == "long" else bool(long_sig.iloc[signal_i])
                if opposite:
                    exit_price_raw, exit_reason = exec_price_base, "Reversal"

            max_hold = int(exit_cfg.get("max_holding_bars", 0) or 0)
            if exit_price_raw is None and max_hold > 0 and bars_held >= max_hold:
                exit_price_raw, exit_reason = exec_price_base, "Time"

            # Conservative intrabar SL/TP with gap/open checks.
            if exit_price_raw is None:
                exit_price_raw, exit_reason = _intrabar_stop_target(open_pos, exec_price_base, bar_high, bar_low, friction=friction, broker_spec=broker_spec)

            if exit_price_raw is not None:
                cash, trade = _close_open_position(open_pos, exit_price_raw, exit_reason, i, ts, cash, friction, broker_spec)
                trades.append(trade)
                open_pos = None
                _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
                if max_dd_pct is not None and _max_drawdown_pct_from_equity(equity) >= float(max_dd_pct):
                    break
                if cash <= 0:
                    break
                continue

            # Update trailing stop after the bar if no exit occurred.
            trail_type = str(exit_cfg.get("trailing_type", "none") or "none").lower()
            if trail_type != "none":
                old_sl = float(open_pos["stop_loss"])
                new_sl = old_sl
                if trail_type == "atr" and current_atr is not None:
                    mult = float(exit_cfg.get("trail_multiplier", 1.5))
                    dist = current_atr * mult
                    new_sl = bar_close - dist if direction == "long" else bar_close + dist
                elif trail_type == "percent":
                    pct = float(exit_cfg.get("trail_percent", 0.5)) / 100.0
                    dist = bar_close * pct
                    new_sl = bar_close - dist if direction == "long" else bar_close + dist
                elif trail_type == "step":
                    step_price = _to_price_units(float(exit_cfg.get("trail_step_pips", 50)), broker_spec, {"cost_value_mode": "points"})
                    min_step = max(step_price, float(broker_spec.get("point", 0.01)))
                    if direction == "long":
                        favourable = bar_close - open_pos["entry_price"]
                        steps = math.floor(max(0.0, favourable) / min_step)
                        if steps > 0:
                            new_sl = open_pos["entry_price"] + (steps - 1) * step_price
                    else:
                        favourable = open_pos["entry_price"] - bar_close
                        steps = math.floor(max(0.0, favourable) / min_step)
                        if steps > 0:
                            new_sl = open_pos["entry_price"] - (steps - 1) * step_price

                if direction == "long" and new_sl > old_sl:
                    open_pos["stop_loss"] = float(new_sl)
                    open_pos["stop_loss_is_trailing"] = True
                elif direction == "short" and new_sl < old_sl:
                    open_pos["stop_loss"] = float(new_sl)
                    open_pos["stop_loss_is_trailing"] = True

            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=bars_held)
            if max_dd_pct is not None and _max_drawdown_pct_from_equity(equity) >= float(max_dd_pct):
                break
            if equity[-1] <= 0:
                break
            continue

        # ----------------------- open new position -----------------------
        if current_atr is None:
            # Indicator warm-up not complete. Do not fall back to future ATR.
            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            continue

        if not _safe_passes_filters(df, signal_i, filters, filter_params=filter_params):
            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            continue

        go_long = bool(long_sig.iloc[signal_i])
        go_short = bool(short_sig.iloc[signal_i])
        if go_long and go_short:
            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            continue
        if not go_long and not go_short:
            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            continue

        direction = "long" if go_long else "short"
        entry_raw = exec_price_base
        entry_side = "buy" if direction == "long" else "sell"
        entry_exec, _ = _apply_execution_price(entry_raw, entry_side, friction, broker_spec, timestamp=ts)
        atr_mult = float(exit_cfg.get("atr_multiplier", params.get("ATR", {}).get("multiplier", 2.0)))

        if exit_cfg.get("sl_type", "atr") == "atr":
            stop_loss = entry_exec - current_atr * atr_mult if direction == "long" else entry_exec + current_atr * atr_mult
        else:
            stop_loss = _calculate_structure_stop_safe(df, signal_i, direction, window=int(exit_cfg.get("structure_window", 20)))

        if exit_cfg.get("tp_type", "rr") == "rr":
            rr = float(exit_cfg.get("risk_reward_ratio", 2.0))
            risk = abs(entry_exec - stop_loss)
            take_profit = entry_exec + risk * rr if direction == "long" else entry_exec - risk * rr
        else:
            take_profit = _calculate_fib_target_safe(
                entry_exec,
                df,
                signal_i,
                direction,
                fib_levels=exit_cfg.get("fib_levels", params.get("Fibonacci", {}).get("levels", [1.618])),
            )

        if not _valid_stop_target(entry_exec, stop_loss, take_profit, direction):
            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            continue

        lot = _calculate_lot(cash, entry_exec, stop_loss, sizing_cfg, atr_value=current_atr, spec=broker_spec)
        if lot <= 0 or not np.isfinite(float(lot)):
            # Risk-based sizing may skip trades when the raw lot is below broker min_lot.
            _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            continue
        entry_commission = _commission_per_side(lot, friction)
        cash -= entry_commission
        if cash <= 0:
            _append_equity(equity, cash, None, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
            break

        open_pos = {
            "entry_idx": i,
            "entry_time": ts,
            "signal_time": signal_ts,
            "entry_raw": entry_raw,
            "entry_price": entry_exec,
            "entry_commission": entry_commission,
            "lot": lot,
            "direction": direction,
            "stop_loss": float(stop_loss),
            "take_profit": float(take_profit),
            "stop_loss_is_trailing": False,
        }

        # Entry-bar SL/TP check: critical when execution is next-bar-open.
        if execution_mode == "next_bar_open":
            exit_price_raw, exit_reason = _intrabar_stop_target(open_pos, exec_price_base, bar_high, bar_low, friction=friction, broker_spec=broker_spec)
            if exit_price_raw is not None:
                cash, trade = _close_open_position(open_pos, exit_price_raw, f"EntryBar_{exit_reason}", i, ts, cash, friction, broker_spec)
                trades.append(trade)
                open_pos = None
                _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
                if max_dd_pct is not None and _max_drawdown_pct_from_equity(equity) >= float(max_dd_pct):
                    break
                continue

        _append_equity(equity, cash, open_pos, bar_close, broker_spec, friction, timestamp=ts, bars_held=0)
        if max_dd_pct is not None and _max_drawdown_pct_from_equity(equity) >= float(max_dd_pct):
            break

    # End-of-test close at the last processed bar. If max-DD/capital stop ended the test early,
    # do not look ahead to the final dataset close. Then pad equity flat so CAGR/drawdown use the
    # intended test horizon instead of an artificially short curve.
    if open_pos is not None and len(df) > 0:
        exit_i = min(max(last_i, 0), len(df) - 1)
        ts = df.index[exit_i]
        final_close = float(df["close"].iloc[exit_i])
        reason = "StoppedEarly" if exit_i < len(df) - 1 else "EndOfData"
        cash, trade = _close_open_position(open_pos, final_close, reason, exit_i, ts, cash, friction, broker_spec)
        trades.append(trade)
        open_pos = None
        _append_equity(equity, cash, open_pos, final_close, broker_spec, friction, timestamp=ts, bars_held=0)

    if not equity:
        equity = [float(initial_capital)]
    if len(equity) < expected_equity_len:
        equity.extend([float(equity[-1])] * (expected_equity_len - len(equity)))
    return trades, float(cash), equity


# ============================================================
# 6) METRICS / CLEANING
# ============================================================
def compute_strategy_metrics(trades, equity_curve=None, initial_capital: float = INITIAL_CAPITAL, bars_per_year: int = 6048):
    eq = pd.Series(equity_curve if equity_curve is not None and len(equity_curve) else [initial_capital], dtype=float)
    if eq.empty:
        eq = pd.Series([initial_capital], dtype=float)
    final_equity = float(eq.iloc[-1])
    net_profit = final_equity - float(initial_capital)

    n_bars = max(len(eq), 1)
    years = n_bars / float(bars_per_year) if bars_per_year else 0.0
    cagr = ((final_equity / initial_capital) ** (1 / years) - 1) if years > 0 and final_equity > 0 else 0.0

    peak = eq.cummax()
    dd_abs = float((peak - eq).max())
    dd_pct = float(((peak - eq) / peak.replace(0, np.nan)).max()) if len(eq) else 0.0
    if pd.isna(dd_pct):
        dd_pct = 0.0

    returns = eq.pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    if len(returns) > 1 and returns.std() > 0:
        vol = float(returns.std() * np.sqrt(bars_per_year))
        sharpe = float(returns.mean() / returns.std() * np.sqrt(bars_per_year))
        downside = returns[returns < 0]
        sortino = float(returns.mean() / downside.std() * np.sqrt(bars_per_year)) if len(downside) > 1 and downside.std() > 0 else 0.0
    else:
        vol, sharpe, sortino = 0.0, 0.0, 0.0

    if not trades:
        return {
            "Net Profit": 0.0, "CAGR": 0.0, "Max Drawdown": 0.0, "Max Drawdown %": 0.0,
            "Volatility": 0.0, "Sharpe Ratio": 0.0, "Sortino Ratio": 0.0,
            "Win Rate": 0.0, "Profit Factor": 0.0, "# Trades": 0,
            "Avg Trade Duration": 0.0, "Expectancy": 0.0, "Recovery Factor": 0.0,
            "Max Consecutive Losses": 0, "Stopped Early": False, "Strategy Score": 0.0,
        }

    tdf = pd.DataFrame(trades)
    stopped_early = bool("reason" in tdf.columns and tdf["reason"].astype(str).str.contains("StoppedEarly", case=False, na=False).any())
    wins = tdf[tdf["pnl"] > 0]
    losses = tdf[tdf["pnl"] < 0]
    win_rate = float(len(wins) / len(tdf)) if len(tdf) else 0.0
    gross_profit = float(wins["pnl"].sum()) if len(wins) else 0.0
    gross_loss = abs(float(losses["pnl"].sum())) if len(losses) else 0.0
    profit_factor_raw = np.inf if gross_loss == 0 and gross_profit > 0 else (gross_profit / gross_loss if gross_loss > 0 else 0.0)
    profit_factor_for_score = min(float(profit_factor_raw if np.isfinite(profit_factor_raw) else 5.0), 5.0)
    expectancy = float(tdf["pnl"].mean())
    avg_duration = float(tdf["bars"].mean()) if "bars" in tdf.columns else 0.0

    loss_flags = (tdf["pnl"] < 0).astype(int)
    groups = (loss_flags != loss_flags.shift()).cumsum()
    max_consec = int(loss_flags.groupby(groups).sum().max()) if len(loss_flags) else 0
    recovery = float(net_profit / dd_abs) if dd_abs > 0 else 0.0

    # Score emphasizes OOS-like stability: return + risk control + trade quality.
    score = (
        min(max(sharpe, -5), 5) * 0.30 +
        min(max(sortino, -5), 5) * 0.15 +
        min(profit_factor_for_score, 5) * 0.20 +
        win_rate * 0.10 +
        min(max(cagr, -1), 3) * 0.15 -
        min(dd_pct, 1.0) * 0.10
    )

    return {
        "Net Profit": float(net_profit),
        "CAGR": float(cagr),
        "Max Drawdown": float(dd_abs),
        "Max Drawdown %": float(dd_pct),
        "Volatility": float(vol),
        "Sharpe Ratio": float(sharpe),
        "Sortino Ratio": float(sortino),
        "Win Rate": float(win_rate),
        "Profit Factor": float(profit_factor_raw if np.isfinite(profit_factor_raw) else 999.0),
        "# Trades": int(len(tdf)),
        "Avg Trade Duration": avg_duration,
        "Expectancy": expectancy,
        "Recovery Factor": recovery,
        "Max Consecutive Losses": max_consec,
        "Stopped Early": bool(stopped_early),
        "Strategy Score": float(score),
    }


def clean_strategies(df: pd.DataFrame, capital: float = INITIAL_CAPITAL, min_trades: int = 30, max_mdd_pct: float = 0.50, win_rate_cap: float = 0.995, allow_stopped_early: bool = True):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out = out[out["# Trades"] >= int(min_trades)]
    if "Max Drawdown %" in out.columns:
        out = out[out["Max Drawdown %"] <= float(max_mdd_pct)]
    else:
        out = out[out["Max Drawdown"] <= float(max_mdd_pct) * float(capital)]
    out = out[out["Win Rate"] <= float(win_rate_cap)]
    if not allow_stopped_early and "Stopped Early" in out.columns:
        out = out[out["Stopped Early"] == False]
    return out.reset_index(drop=True)


def _csv_safe(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    drop_cols = [c for c in ["equity_curve", "trades", "config"] if c in df.columns]
    return df.drop(columns=drop_cols)


# ============================================================
# 7) CONFIG GENERATOR
# ============================================================
def _indicator_combos(cp: Dict[str, Dict], min_n: int = 2, max_n: Optional[int] = None) -> List[Tuple[str, ...]]:
    indicators = [k for k in cp.keys() if k not in SKIP_INDICATORS]
    if DROP_ATR_FROM_ENTRIES:
        indicators = [k for k in indicators if k != "ATR"]
    max_n = min(max_n or len(indicators), len(indicators))
    combos = []
    for r in range(min_n, max_n + 1):
        combos.extend(itertools.combinations(indicators, r))
    return combos


def _param_samples_for_combo(combined_params: Dict[str, Dict[str, list]], indicators: Sequence[str], max_samples: Optional[int]) -> Iterable[dict]:
    pools = []
    order = []
    fixed = {}
    for ind in indicators:
        grid = combined_params.get(ind, {}) or {}
        for key, vals in grid.items():
            if vals is None:
                continue
            vals = list(vals) if isinstance(vals, (list, tuple, set, np.ndarray)) else [vals]
            if ind == "Fibonacci" and key == "levels":
                fixed.setdefault(ind, {})[key] = vals
                continue
            if len(vals) == 0:
                continue
            pools.append(vals)
            order.append((ind, key))

    atr_periods = list((combined_params.get("ATR", {}) or {}).get("period", [DEFAULTS["ATR"]["period"]]))
    if not pools:
        params = copy.deepcopy(fixed)
        params.setdefault("ATR", {})
        params["ATR"].setdefault("period", random.choice(atr_periods))
        params["ATR"].setdefault("ma_period", 50)
        yield _adapt_params(params)
        return

    def build(draw):
        params = copy.deepcopy(fixed)
        for (ind, key), val in zip(order, draw):
            params.setdefault(ind, {})[key] = val
        # ATR is used by stops/sizing even when it is not part of entry combo.
        # Do not overwrite ATR period if the ATR entry indicator supplied it.
        params.setdefault("ATR", {})
        params["ATR"].setdefault("period", random.choice(atr_periods))
        params["ATR"].setdefault("ma_period", 50)
        return _adapt_params(params)

    if max_samples is None:
        for draw in itertools.product(*pools):
            yield build(draw)
    else:
        seen = set()
        max_tries = max(int(max_samples) * 10, 50)
        tries = 0
        while len(seen) < int(max_samples) and tries < max_tries:
            draw = tuple(vals[random.randrange(len(vals))] for vals in pools)
            if draw not in seen:
                seen.add(draw)
                yield build(draw)
            tries += 1


def _exit_sizing_filter_samples(
    sl_tp_settings: List[dict],
    trailing_stop_settings: List[dict],
    time_based_settings: List[dict],
    indicator_exit_settings: List[dict],
    drawdown_limits: List[float],
    sizing_methods: List[dict],
    filter_combos: List[dict],
    settings_samples: Optional[int],
) -> List[dict]:
    all_parts = []
    base_sl = sl_tp_settings or [{}]
    base_trail = trailing_stop_settings or [{"trailing_type": "none"}]
    base_time = time_based_settings or [{}]
    base_indicator_exit = indicator_exit_settings or [{"use_indicator_exit": False}]
    base_dd = drawdown_limits or [None]
    base_sizing = sizing_methods or [{"sizing_method": "fixed", "fixed_lot": 0.1}]
    base_filters = filter_combos or [{}]

    if settings_samples is None:
        iterator = itertools.product(base_sl, base_trail, base_time, base_indicator_exit, base_dd, base_sizing, base_filters)
        for sl, tr, tm, ie, dd, sz, flt in iterator:
            exit_cfg = {**sl, **tr, **tm, **ie}
            if dd is not None:
                exit_cfg["max_drawdown_pct"] = float(dd)
            all_parts.append({"exit": exit_cfg, "sizing": sz, "filters": flt})
        return all_parts

    for _ in range(max(1, int(settings_samples))):
        sl = random.choice(base_sl)
        tr = random.choice(base_trail)
        tm = random.choice(base_time)
        ie = random.choice(base_indicator_exit)
        dd = random.choice(base_dd)
        sz = random.choice(base_sizing)
        flt = random.choice(base_filters)
        exit_cfg = {**sl, **tr, **tm, **ie}
        if dd is not None:
            exit_cfg["max_drawdown_pct"] = float(dd)
        all_parts.append({"exit": exit_cfg, "sizing": sz, "filters": flt})
    return all_parts


def generate_all_strategy_configs_stream(
    *,
    combined_params: Dict[str, Dict],
    entry_functions: Any,
    short_entry_functions: Any,
    sl_tp_settings: List[dict],
    trailing_stop_settings: List[dict],
    time_based_settings: List[dict],
    indicator_exit_settings: List[dict],
    drawdown_limits: List[float],
    sizing_methods: List[dict],
    friction_settings: Any,
    strategy_batch_id: str,
    max_params_per_combo: Optional[int],
    settings_samples: Optional[int],
    batch_size: int,
    total_configs_cap: Optional[int],
    max_filters: int,
    filter_samples: Optional[int],
    skip_indicators: set = frozenset(),
) -> Iterable[List[dict]]:
    combos = _indicator_combos(combined_params, min_n=2, max_n=7)
    focused = globals().get("FOCUSED_INDICATOR_COMBOS", None)
    if focused:
        allowed = {frozenset([str(x) for x in combo]) for combo in focused}
        before_focus = len(combos)
        combos = [combo for combo in combos if frozenset(combo) in allowed]
        print(f"🎯 Focused indicator combos enabled: {len(combos)}/{before_focus} combos → {combos}")
    friction = friction_settings[0] if isinstance(friction_settings, list) and friction_settings else (friction_settings or {})
    friction = {**XAUUSD_SPEC, **friction}
    filter_combos = _build_filter_combinations(max_filters=max_filters, samples=filter_samples)

    sid = 0
    produced = 0
    bucket: List[dict] = []
    print(f"📊 Generating configs | combos={len(combos)} | filters={len(filter_combos)} | cap={total_configs_cap}")

    for combo in tqdm(combos, desc="🔄 Indicator Combos", leave=True):
        long_fn, short_fn, strat_name = _resolve_entry_pair(
            entry_functions,
            short_entry_functions,
            combo,
            entry_logic={"mode": ENTRY_MODE, "min_confirmations": MIN_CONFIRMATIONS},
            skip_indicators=skip_indicators,
        )
        if long_fn is None or short_fn is None:
            continue

        es_samples = _exit_sizing_filter_samples(
            sl_tp_settings,
            trailing_stop_settings,
            time_based_settings,
            indicator_exit_settings,
            drawdown_limits,
            sizing_methods,
            filter_combos,
            settings_samples,
        )

        for params in _param_samples_for_combo(combined_params, combo, max_params_per_combo):
            atr_period = params.get("ATR", {}).get("period", DEFAULTS["ATR"]["period"])
            for es in es_samples:
                cfg = {
                    "id": f"{strategy_batch_id}_{sid}",
                    "params": params,
                    "entry_indicators": tuple(combo),
                    "entry_strategy_name": strat_name,
                    "entry_logic": {"mode": ENTRY_MODE, "min_confirmations": MIN_CONFIRMATIONS},
                    "exit": {**es["exit"], "atr_period": atr_period},
                    "sizing": {**XAUUSD_SPEC, **es["sizing"]},
                    "friction": friction,
                    "filters": es["filters"],
                }
                bucket.append(cfg)
                sid += 1
                produced += 1
                if len(bucket) >= batch_size:
                    yield bucket
                    bucket = []
                    gc.collect()
                if total_configs_cap is not None and produced >= int(total_configs_cap):
                    if bucket:
                        yield bucket
                    print(f"📦 Config generation capped at {produced}")
                    return

    if bucket:
        yield bucket
    print(f"📦 Config generation done | produced={produced}")


# ============================================================
# 8) EVALUATION RUNNERS
# ============================================================

def _compact_config_signature(cfg: dict) -> str:
    payload = {
        "indicators": list(cfg.get("entry", {}).get("indicators", cfg.get("entry_indicators", []))),
        "entry_logic": cfg.get("entry_logic", {}),
        "params": cfg.get("params", {}),
        "exit": cfg.get("exit", {}),
        "sizing": cfg.get("sizing", {}),
        "filters": cfg.get("filters", {}),
    }
    try:
        return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()
    except Exception:
        return hashlib.md5(repr(payload).encode()).hexdigest()


def _with_timeframe_friction(cfg: dict, tf_name: str) -> dict:
    """Inject timeframe-specific bars_per_day for swap and holding-cost calculations."""
    cfg2 = copy.deepcopy(cfg)
    fr = {**XAUUSD_SPEC, **(cfg2.get("friction", {}) or {})}
    fr["bars_per_day"] = BARS_PER_DAY_BY_TF.get(tf_name, fr.get("bars_per_day", 96))
    cfg2["friction"] = fr
    sz = {**XAUUSD_SPEC, **(cfg2.get("sizing", {}) or {})}
    sz.setdefault("skip_if_lot_below_min", SKIP_RISK_LOT_BELOW_MIN)
    cfg2["sizing"] = sz
    return cfg2


def _evaluate_live_configs(live_cfgs: List[dict], df: pd.DataFrame, tf_name: str, phase: str = "FULL", window_id: Optional[int] = None) -> pd.DataFrame:
    records = []
    bars_per_year = BARS_PER_YEAR_BY_TF.get(tf_name, 6048)
    for cfg_live in live_cfgs:
        cfg_run = _with_timeframe_friction(cfg_live, tf_name)
        trades, final_cap, equity = run_backtest(cfg_run, df, initial_capital=INITIAL_CAPITAL, broker_spec=XAUUSD_SPEC)
        m = compute_strategy_metrics(trades, equity_curve=equity, initial_capital=INITIAL_CAPITAL, bars_per_year=bars_per_year)
        m.update({
            "Strategy ID": cfg_run.get("id", "N/A"),
            "Strategy Name": cfg_run.get("entry", {}).get("strategy_name"),
            "Indicators": ",".join(cfg_run.get("entry", {}).get("indicators", [])),
            "Config Signature": _compact_config_signature(cfg_run),
            "Timeframe": tf_name,
            "Phase": phase,
            "Window": window_id,
            "Final Capital": final_cap,
            "RUN_MODE": RUN_MODE,
            "equity_curve": np.array(equity, dtype=float),
            "trades": trades,
        })
        records.append(m)
    return pd.DataFrame(records)


def _split_walk_forward_windows(df: pd.DataFrame, train_window: int, test_window: int, step_size: int, timeframe: str):
    planned = plan_walk_forward_windows(
        _standardize_ohlcv(df),
        timeframe=timeframe,
        train_window=int(train_window),
        test_window=int(test_window),
        step_size=int(step_size),
    )
    return [
        {
            "window": w.window,
            "train_start": w.train_start,
            "train_end": w.train_end,
            "test_start": w.test_start,
            "test_end": w.test_end,
        }
        for w in planned
    ]


def _aggregate_walkforward_oos(
    test_rows: pd.DataFrame,
    min_profitable_window_ratio: float = 0.60,
    min_windows: int = 3,
    min_oos_trades: int = 30,
    max_oos_dd: Optional[float] = None,
    min_oos_sharpe: Optional[float] = None,
    min_oos_profit_factor: Optional[float] = None,
) -> pd.DataFrame:
    """Aggregate TEST rows by strategy so final ranking is based on true OOS consistency.

    V3.9: OOS Pass Rule now includes net profit, worst drawdown, median Sharpe,
    and capped average profit factor. Aggregate fields overwrite the single-window
    fields so quality gates act on the OOS aggregate, not one lucky window.
    """
    if test_rows is None or test_rows.empty:
        return pd.DataFrame()

    wf_gate = QUALITY_GATES.get("WALK_FORWARD", {})
    max_oos_dd = float(max_oos_dd if max_oos_dd is not None else wf_gate.get("max_dd_pct", 0.20))
    min_oos_sharpe = float(min_oos_sharpe if min_oos_sharpe is not None else wf_gate.get("min_sharpe", 0.30))
    min_oos_profit_factor = float(min_oos_profit_factor if min_oos_profit_factor is not None else wf_gate.get("min_profit_factor", 1.15))

    rows = []
    group_cols = ["Strategy ID", "Timeframe"]
    for (sid, tf), g in test_rows.groupby(group_cols, dropna=False):
        g = g.copy()
        windows = int(g["Window"].nunique()) if "Window" in g.columns else len(g)
        total_trades = int(pd.to_numeric(g.get("# Trades", 0), errors="coerce").fillna(0).sum()) if "# Trades" in g.columns else 0
        profitable_windows = float((pd.to_numeric(g["Net Profit"], errors="coerce") > 0).mean()) if "Net Profit" in g.columns and len(g) else 0.0
        worst_dd_pct = float(pd.to_numeric(g["Max Drawdown %"], errors="coerce").max()) if "Max Drawdown %" in g.columns else np.nan
        avg_score = float(pd.to_numeric(g.get("Strategy Score", 0), errors="coerce").fillna(0).mean()) if "Strategy Score" in g.columns else 0.0
        med_score = float(pd.to_numeric(g.get("Strategy Score", 0), errors="coerce").fillna(0).median()) if "Strategy Score" in g.columns else 0.0
        avg_cagr = float(pd.to_numeric(g.get("CAGR", 0), errors="coerce").fillna(0).mean()) if "CAGR" in g.columns else 0.0
        med_cagr = float(pd.to_numeric(g.get("CAGR", 0), errors="coerce").fillna(0).median()) if "CAGR" in g.columns else 0.0
        avg_sharpe = float(pd.to_numeric(g.get("Sharpe Ratio", 0), errors="coerce").fillna(0).mean()) if "Sharpe Ratio" in g.columns else 0.0
        med_sharpe = float(pd.to_numeric(g.get("Sharpe Ratio", 0), errors="coerce").fillna(0).median()) if "Sharpe Ratio" in g.columns else 0.0
        total_net = float(pd.to_numeric(g.get("Net Profit", 0), errors="coerce").fillna(0).sum()) if "Net Profit" in g.columns else 0.0
        if "Profit Factor" in g.columns:
            pf_series = pd.to_numeric(g["Profit Factor"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
            avg_pf = float(pf_series.replace(999.0, 5.0).clip(upper=5.0).mean())
        else:
            avg_pf = 0.0

        finite_worst_dd = worst_dd_pct if np.isfinite(worst_dd_pct) else 1.0
        oos_pass = bool(
            windows >= int(min_windows)
            and profitable_windows >= float(min_profitable_window_ratio)
            and total_trades >= int(min_oos_trades)
            and total_net > 0
            and finite_worst_dd <= max_oos_dd
            and med_sharpe >= min_oos_sharpe
            and avg_pf >= min_oos_profit_factor
        )

        # Robust OOS score: consistency first, then return quality, with worst-window DD penalty.
        oos_score = (
            med_score * 0.25
            + avg_score * 0.10
            + min(max(med_sharpe, -5), 5) * 0.20
            + profitable_windows * 0.20
            + min(max(med_cagr, -1), 3) * 0.10
            + min(avg_pf, 5) * 0.15
            - min(finite_worst_dd, 1.0) * 0.20
        )
        first = g.sort_values(["Strategy Score", "CAGR"], ascending=False).iloc[0].to_dict()
        first.update({
            "Phase": "OOS_AGG",
            "Window": "ALL",
            "OOS Windows": windows,
            "OOS Profitable Window %": profitable_windows,
            "OOS Total Trades": total_trades,
            "OOS Total Net Profit": total_net,
            "OOS Avg CAGR": avg_cagr,
            "OOS Median CAGR": med_cagr,
            "OOS Avg Sharpe": avg_sharpe,
            "OOS Median Sharpe": med_sharpe,
            "OOS Worst Drawdown %": worst_dd_pct,
            "OOS Avg Profit Factor Capped": avg_pf,
            "OOS Min Windows Required": int(min_windows),
            "OOS Min Trades Required": int(min_oos_trades),
            "OOS Max DD Required": max_oos_dd,
            "OOS Min Sharpe Required": min_oos_sharpe,
            "OOS Min PF Required": min_oos_profit_factor,
            "Strategy Score": float(oos_score),
            "CAGR": med_cagr,
            "Sharpe Ratio": med_sharpe,
            "Profit Factor": avg_pf,
            "Net Profit": total_net,
            "# Trades": total_trades,
            "Max Drawdown %": worst_dd_pct,
            "OOS Pass Rule": oos_pass,
        })
        # Aggregated rows should not carry one specific equity/trade sample as if it represented the full OOS path.
        first["equity_curve"] = np.array([], dtype=float)
        first["trades"] = []
        rows.append(first)
    return pd.DataFrame(rows)



def _evaluate_wf_chunk(live_cfgs: List[dict], df: pd.DataFrame, tf_name: str) -> pd.DataFrame:
    wf_cfg = WF_SETTINGS_BY_TF.get(tf_name, {
        "train_window": int(CFG.get("wf_train_window", 3000)),
        "test_window": int(CFG.get("wf_test_window", 500)),
        "step_size": int(CFG.get("wf_step_size", 500)),
        "top_n_train": int(CFG.get("wf_top_n_train", 3)),
    })
    try:
        windows = _split_walk_forward_windows(
            df,
            train_window=int(wf_cfg["train_window"]),
            test_window=int(wf_cfg["test_window"]),
            step_size=int(wf_cfg["step_size"]),
            timeframe=tf_name,
        )
    except UnsafeEvaluationError as e:
        print(f"⚠️ {tf_name}: walk-forward skipped because a safe plan cannot be formed: {e}")
        return pd.DataFrame()

    all_tests = []
    top_n = int(wf_cfg.get("top_n_train", CFG.get("wf_top_n_train", 3)))
    for w in windows:
        train_df = df.iloc[w["train_start"]:w["train_end"]].copy()
        test_df = df.iloc[w["test_start"]:w["test_end"]].copy()
        train_eval = _evaluate_live_configs(live_cfgs, train_df, tf_name, phase="TRAIN", window_id=w["window"])
        if train_eval.empty:
            continue
        # Keep only strategies with at least a few trades in training to reduce lucky no-trade rows.
        train_eval = train_eval[train_eval["# Trades"] >= max(1, int(MIN_TRADES // 2))]
        if train_eval.empty:
            continue
        train_eval = train_eval.sort_values(["Strategy Score", "CAGR"], ascending=False).head(top_n)
        top_ids = set(train_eval["Strategy ID"].tolist())
        top_cfgs = [cfg for cfg in live_cfgs if cfg.get("id") in top_ids]
        test_eval = _evaluate_live_configs(top_cfgs, test_df, tf_name, phase="TEST", window_id=w["window"])
        if not test_eval.empty:
            test_eval = test_eval.merge(
                train_eval[["Strategy ID", "Strategy Score", "CAGR", "# Trades"]].rename(columns={
                    "Strategy Score": "Train Strategy Score",
                    "CAGR": "Train CAGR",
                    "# Trades": "Train # Trades",
                }),
                on="Strategy ID",
                how="left",
            )
            all_tests.append(test_eval)

    test_rows = pd.concat(all_tests, ignore_index=True) if all_tests else pd.DataFrame()
    if test_rows.empty:
        return pd.DataFrame()
    agg = _aggregate_walkforward_oos(
        test_rows,
        min_profitable_window_ratio=0.60,
        min_windows=min(3, len(windows)),
        min_oos_trades=max(10, MIN_TRADES),
    )
    # Keep raw TEST rows on disk for audit.
    return agg


def smoke_test_entries(tf_name: str, df: pd.DataFrame):
    print(f"\n🔬 Smoke test on {tf_name}")
    for ind in ["EMA", "MACD", "RSI", "BollingerBands", "ATR", "Fibonacci", "Stochastic"]:
        if ind in SKIP_INDICATORS or ind not in entry_functions or ind not in short_entry_functions:
            continue
        try:
            test_params = _adapt_params({ind: {}})
            L = call_entry_fn(entry_functions[ind], ind, df, test_params)
            S = call_entry_fn(short_entry_functions[ind], ind, df, test_params)
            print(f"  • {ind}: ✅ long={int(L.sum())} / short={int(S.sum())}")
        except Exception as e:
            print(f"  • {ind}: ❌ {e}")


def run_timeframe_pipeline(tf_name: str, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    global PROFILE_RUNTIME_GUARD_HIT
    tf_dir = os.path.join(OUTPUT_ROOT, f"{RUN_ID}_{tf_name}")
    os.makedirs(tf_dir, exist_ok=True)
    df = _standardize_ohlcv(df)
    smoke_test_entries(tf_name, df)

    cleaned_frames, soft_frames = [], []
    gen = generate_all_strategy_configs_stream(
        combined_params=combined_params,
        entry_functions=entry_functions,
        short_entry_functions=short_entry_functions,
        sl_tp_settings=sl_tp_settings,
        trailing_stop_settings=trailing_stop_settings,
        time_based_settings=time_based_settings,
        indicator_exit_settings=indicator_exit_settings,
        drawdown_limits=drawdown_limits,
        sizing_methods=sizing_methods,
        friction_settings=ACTIVE_FRICTION_SETTINGS,
        strategy_batch_id=f"{RUN_ID}_{tf_name}",
        max_params_per_combo=MAX_PARAMS_PER_COMBO,
        settings_samples=SETTINGS_SAMPLES,
        batch_size=BATCH_SIZE,
        total_configs_cap=TOTAL_CONFIGS_CAP,
        max_filters=MAX_FILTERS,
        filter_samples=FILTER_SAMPLES,
        skip_indicators=SKIP_INDICATORS,
    )

    batch_start = 0
    batch_count = 0
    for cfg_chunk in tqdm(gen, desc=f"{tf_name} Batches"):
        chk_name = f"batch_{batch_start}_{batch_start + len(cfg_chunk) - 1}"
        cfg_path = os.path.join(tf_dir, f"{chk_name}_CFGS.pkl")
        raw_path = os.path.join(tf_dir, f"{chk_name}_EVAL_RAW.pkl")
        clean_path = os.path.join(tf_dir, f"{chk_name}_CLEAN.pkl")
        soft_path = os.path.join(tf_dir, f"{chk_name}_SOFT.pkl")

        if RESUME_EXISTING_BATCHES and os.path.exists(raw_path) and os.path.exists(clean_path):
            try:
                with open(clean_path, "rb") as f:
                    prev_clean = pickle.load(f)
                if isinstance(prev_clean, pd.DataFrame) and not prev_clean.empty:
                    cleaned_frames.append(prev_clean)
                elif os.path.exists(soft_path):
                    with open(soft_path, "rb") as f:
                        prev_soft = pickle.load(f)
                    if isinstance(prev_soft, pd.DataFrame) and not prev_soft.empty:
                        soft_frames.append(prev_soft)
                print(f"⏭️ {tf_name} {chk_name}: found existing checkpoint; skipped re-evaluation")
                batch_start += len(cfg_chunk)
                batch_count += 1
                if MAX_BATCHES_PER_TF is not None and batch_count >= int(MAX_BATCHES_PER_TF):
                    print(f"⏹️ {tf_name}: reached max_batches_per_tf={MAX_BATCHES_PER_TF}; stopping this timeframe safely.")
                    break
                if _runtime_limit_reached():
                    PROFILE_RUNTIME_GUARD_HIT = True
                    print(f"⏹️ Runtime guard reached after checkpoint skip; stopping safely. Re-run with RUN_ID_OVERRIDE='{RUN_ID}' to resume.")
                    break
                continue
            except Exception as e:
                print(f"⚠️ Could not load existing checkpoint for {chk_name}; will re-evaluate. Error: {e}")

        with open(cfg_path, "wb") as f:
            pickle.dump(cfg_chunk, f)

        live_cfgs = []
        for saved_cfg in cfg_chunk:
            try:
                live_cfgs.append(rebind_config_from_pickle(saved_cfg, entry_functions, short_entry_functions, skip_indicators=SKIP_INDICATORS))
            except Exception as e:
                print(f"⚠️ rebind error: {e}")

        if RUN_MODE == "WALK_FORWARD":
            eval_df = _evaluate_wf_chunk(live_cfgs, df, tf_name)
        else:
            eval_df = _evaluate_live_configs(live_cfgs, df, tf_name, phase=f"{RUN_MODE}_FULL_SAMPLE")

        if isinstance(eval_df, pd.DataFrame) and not eval_df.empty:
            eval_df["Config Batch Path"] = cfg_path
            eval_df["Config Batch Name"] = chk_name

        with open(raw_path, "wb") as f:
            pickle.dump(eval_df, f)

        cleaned_df = clean_strategies(eval_df, capital=INITIAL_CAPITAL, min_trades=MIN_TRADES, max_mdd_pct=0.50, win_rate_cap=0.995, allow_stopped_early=ALLOW_STOPPED_EARLY_IN_CLEAN)
        if RUN_MODE == "WALK_FORWARD" and not cleaned_df.empty and "OOS Pass Rule" in cleaned_df.columns:
            cleaned_df = cleaned_df[cleaned_df["OOS Pass Rule"] == True].reset_index(drop=True)
        cleaned_df = _apply_quality_gate(cleaned_df, RUN_MODE, verbose=True, label=f"{tf_name} {chk_name}")
        if cleaned_df.empty and not eval_df.empty and SOFT_KEEP_TOP_N > 0:
            top = eval_df.sort_values(["Strategy Score", "CAGR"], ascending=False).head(SOFT_KEEP_TOP_N).copy()
            soft_frames.append(top)
            with open(os.path.join(tf_dir, f"{chk_name}_SOFT.pkl"), "wb") as f:
                pickle.dump(top, f)
        if not cleaned_df.empty:
            cleaned_frames.append(cleaned_df)

        with open(clean_path, "wb") as f:
            pickle.dump(cleaned_df, f)
        print(f"💾 {tf_name} {chk_name}: raw={len(eval_df)} | clean={len(cleaned_df)}")

        batch_start += len(cfg_chunk)
        batch_count += 1
        del cfg_chunk, live_cfgs, eval_df, cleaned_df
        gc.collect()

        if MAX_BATCHES_PER_TF is not None and batch_count >= int(MAX_BATCHES_PER_TF):
            print(f"⏹️ {tf_name}: reached max_batches_per_tf={MAX_BATCHES_PER_TF}; stopping this timeframe safely.")
            break
        if _runtime_limit_reached():
            PROFILE_RUNTIME_GUARD_HIT = True
            elapsed_min = (time.time() - PIPELINE_START_TIME) / 60.0
            print(f"⏹️ Runtime guard reached ({elapsed_min:.1f} min >= {MAX_RUNTIME_MINUTES} min). Stopping safely after saved batch.")
            print(f"   To resume this same run, set RUN_ID_OVERRIDE = '{RUN_ID}' and re-run Cell 18.")
            break

    tf_clean = pd.concat(cleaned_frames, ignore_index=True) if cleaned_frames else pd.DataFrame()
    tf_soft = pd.concat(soft_frames, ignore_index=True) if soft_frames else pd.DataFrame()

    if not tf_clean.empty:
        tf_clean = tf_clean.sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
        tf_clean = deduplicate_ranked_results(tf_clean, corr_threshold=0.98)
        _csv_safe(tf_clean).head(100).to_csv(os.path.join(tf_dir, f"top_clean_{RUN_ID}_{tf_name}.csv"), index=False)
    if not tf_soft.empty:
        tf_soft = tf_soft.sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
        _csv_safe(tf_soft).head(100).to_csv(os.path.join(tf_dir, f"top_soft_{RUN_ID}_{tf_name}.csv"), index=False)

    return tf_clean, tf_soft



def deduplicate_ranked_results(df: pd.DataFrame, corr_threshold: float = 0.98) -> pd.DataFrame:
    """Remove exact config duplicates and highly correlated equity-curve duplicates within each timeframe."""
    if df is None or df.empty:
        return pd.DataFrame()
    work = df.copy().sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
    if "Config Signature" in work.columns:
        work = work.drop_duplicates(subset=["Timeframe", "Config Signature"], keep="first").reset_index(drop=True)

    if "equity_curve" not in work.columns:
        return work

    kept = []
    by_tf = work.groupby("Timeframe", dropna=False, sort=False)
    for _, g in by_tf:
        curves = []
        for _, row in g.iterrows():
            curve = row.get("equity_curve")
            if not isinstance(curve, (list, tuple, np.ndarray, pd.Series)) or len(curve) < 5:
                kept.append(row)
                continue
            arr = np.asarray(curve, dtype=float)
            arr = arr[np.isfinite(arr)]
            if len(arr) < 5 or np.nanstd(arr) == 0:
                kept.append(row)
                curves.append(arr)
                continue
            duplicate = False
            for prev in curves:
                n = min(len(arr), len(prev))
                if n < 5 or np.nanstd(prev[-n:]) == 0 or np.nanstd(arr[-n:]) == 0:
                    continue
                corr = np.corrcoef(arr[-n:], prev[-n:])[0, 1]
                if np.isfinite(corr) and corr >= corr_threshold:
                    duplicate = True
                    break
            if not duplicate:
                kept.append(row)
                curves.append(arr)
    return pd.DataFrame(kept).reset_index(drop=True) if kept else pd.DataFrame()


def compare_python_mt5_indicators(mt5_csv_path: str, python_df: pd.DataFrame, time_col: str = "time", params: Optional[dict] = None) -> pd.DataFrame:
    """
    Optional pre-EA validation helper.
    Export indicator values from MT5 to CSV, then compare columns with Python values by timestamp.
    Expected MT5 columns can include rsi, atr_14, macd_histogram, bb_upper, bb_lower, stoch_k, stoch_d.
    """
    mt5 = pd.read_csv(mt5_csv_path)
    mt5.columns = [str(c).strip().lower() for c in mt5.columns]
    if time_col.lower() in mt5.columns:
        mt5[time_col.lower()] = pd.to_datetime(mt5[time_col.lower()], errors="coerce")
        mt5 = mt5.dropna(subset=[time_col.lower()]).set_index(time_col.lower())
    py = _ensure_indicator_columns(_standardize_ohlcv(python_df), params or {})
    joined = mt5.join(py, how="inner", rsuffix="_py")
    rows = []
    for c in ["rsi", "atr_14", "macd_histogram", "bb_upper", "bb_lower", "stoch_k", "stoch_d"]:
        py_col = f"{c}_py" if f"{c}_py" in joined.columns else c if c in py.columns else None
        mt_col = c if c in mt5.columns else None
        if mt_col and py_col and mt_col in joined.columns and py_col in joined.columns:
            diff = (joined[py_col] - joined[mt_col]).abs().replace([np.inf, -np.inf], np.nan).dropna()
            if len(diff):
                rows.append({"indicator": c, "rows": len(diff), "mean_abs_diff": diff.mean(), "max_abs_diff": diff.max()})
    return pd.DataFrame(rows)




def _limit_rows_for_runtime(tf_name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Limit rows for runtime-safe modes. Uses the most recent bars only."""
    max_rows = MAX_DATA_ROWS_BY_TF.get(tf_name, MAX_DATA_ROWS)
    if max_rows is not None and max_rows > 0 and len(df) > max_rows:
        print(f"⏱️ Runtime-safe data cap for {tf_name}: using last {max_rows:,} of {len(df):,} rows")
        return df.tail(int(max_rows)).copy()
    return df.copy()


def _runtime_limit_reached() -> bool:
    if MAX_RUNTIME_MINUTES is None:
        return False
    elapsed_min = (time.time() - PIPELINE_START_TIME) / 60.0
    return elapsed_min >= float(MAX_RUNTIME_MINUTES)

# ============================================================
# 9) PREP DATA + RUN ALL TIMEFRAMES
# ============================================================

# ============================================================
# 12) MAIN PIPELINE RUNNER / AUTO SAMPLE → FOCUSED WALK-FORWARD
# ============================================================

def _runtime_cfg_from_profile(profile: dict, run_mode_label: str):
    """Apply a mode/profile dict to the global runtime variables used by the existing engine."""
    global CFG, ACTIVE_TIMEFRAMES, BATCH_SIZE, MAX_PARAMS_PER_COMBO, SETTINGS_SAMPLES
    global TOTAL_CONFIGS_CAP, MAX_FILTERS, FILTER_SAMPLES, MIN_TRADES, DROP_ATR_FROM_ENTRIES
    global MAX_DATA_ROWS, MAX_DATA_ROWS_BY_TF, MAX_RUNTIME_MINUTES, MAX_BATCHES_PER_TF
    global ALLOW_STOPPED_EARLY_IN_CLEAN, FOCUSED_INDICATOR_COMBOS, RUN_MODE
    global WF_SETTINGS_BY_TF

    RUN_MODE = run_mode_label
    CFG = copy.deepcopy(profile)
    ACTIVE_TIMEFRAMES = list(globals().get("ACTIVE_TIMEFRAMES_OVERRIDE", CFG.get("timeframes", [])))
    BATCH_SIZE = int(CFG["batch_size"])
    MAX_PARAMS_PER_COMBO = CFG.get("max_params_per_combo")
    SETTINGS_SAMPLES = CFG.get("settings_samples")
    TOTAL_CONFIGS_CAP = CFG.get("total_configs_cap")
    MAX_FILTERS = int(CFG.get("max_filters", 0))
    FILTER_SAMPLES = CFG.get("filter_samples")
    MIN_TRADES = int(CFG.get("min_trades", 30))
    DROP_ATR_FROM_ENTRIES = bool(CFG.get("drop_atr_from_entries", False))
    MAX_DATA_ROWS = CFG.get("max_data_rows")
    MAX_DATA_ROWS_BY_TF = CFG.get("max_data_rows_by_tf", {}) or {}
    MAX_RUNTIME_MINUTES = CFG.get("max_runtime_minutes")
    MAX_BATCHES_PER_TF = CFG.get("max_batches_per_tf")
    ALLOW_STOPPED_EARLY_IN_CLEAN = (RUN_MODE == "SMOKE")

    # Focused WFA can override calendar windows to keep Colab runs practical.
    if CFG.get("wf_settings_by_tf"):
        wf_base = copy.deepcopy(WF_SETTINGS_BY_TF)
        for k, v in CFG["wf_settings_by_tf"].items():
            wf_base[k] = v
        WF_SETTINGS_BY_TF = wf_base


def _split_sample_forward_raw(tf_name: str, raw_df: pd.DataFrame, sample_ratio: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Chronologically split capped raw OHLCV into sample-search and forward-holdout segments."""
    raw_df = _standardize_ohlcv(raw_df)
    split = split_sample_holdout(
        raw_df,
        timeframe=tf_name,
        sample_ratio=float(sample_ratio),
        min_sample_rows=100,
        min_holdout_rows=1,
    )
    sample_df = split.sample
    forward_df = split.holdout
    print(
        f"🔒 Holdout split {tf_name}: sample={len(sample_df):,} rows "
        f"({sample_df.index.min()} → {sample_df.index.max()}) | "
        f"forward={len(forward_df):,} rows ({forward_df.index.min()} → {forward_df.index.max()})"
    )
    return sample_df, forward_df


def _prepare_precomp_for_active_timeframes():
    """Load available TF data, apply runtime row caps, split sample/holdout when needed, and precompute indicators."""
    global ACTIVE_TIMEFRAMES, PRECOMP, ACTIVE_TF, df_tf, AUTO_FORWARD_RAW_BY_TF
    TIMEFRAME_DATA = {
        "M15": globals().get("DF_M15"),
        "M30": globals().get("DF_M30"),
        "H1": globals().get("DF_H1"),
        "H4": globals().get("DF_H4"),
    }
    TIMEFRAME_DATA = {k: v for k, v in TIMEFRAME_DATA.items() if v is not None}
    missing_tfs = [tf for tf in ACTIVE_TIMEFRAMES if tf not in TIMEFRAME_DATA and tf not in AUTO_FORWARD_RAW_BY_TF]
    if missing_tfs:
        print(f"⚠️ Missing timeframe data and will skip: {missing_tfs}")
    ACTIVE_TIMEFRAMES = [tf for tf in ACTIVE_TIMEFRAMES if tf in TIMEFRAME_DATA or tf in AUTO_FORWARD_RAW_BY_TF]
    if not ACTIVE_TIMEFRAMES and TIMEFRAME_DATA:
        ACTIVE_TIMEFRAMES = list(TIMEFRAME_DATA.keys())
        print(f"ℹ️ No preferred timeframe found for RUN_MODE={RUN_MODE}; falling back to available data: {ACTIVE_TIMEFRAMES}")
    if not ACTIVE_TIMEFRAMES:
        raise RuntimeError("No active timeframe data available. Please run the data-loading cells first.")

    PRECOMP = {}
    for tf_name in ACTIVE_TIMEFRAMES:
        # Exact forward phase in AUTO mode uses the later holdout segment from Phase 1.
        if AUTO_HOLDOUT_ENABLED and RUN_MODE == "WALK_FORWARD" and tf_name in AUTO_FORWARD_RAW_BY_TF:
            raw_df = _standardize_ohlcv(AUTO_FORWARD_RAW_BY_TF[tf_name]).copy()
            print(f"🔒 Using forward holdout only for {tf_name}: {len(raw_df):,} rows")
        else:
            raw_df = _limit_rows_for_runtime(tf_name, _standardize_ohlcv(TIMEFRAME_DATA[tf_name]))
            # AUTO sample phase uses only the earlier segment and stores the later segment for exact forward.
            if AUTO_HOLDOUT_ENABLED and RUN_MODE == "SAMPLED_ALL_TF":
                raw_df, forward_raw = _split_sample_forward_raw(tf_name, raw_df, AUTO_SAMPLE_FORWARD_RATIO)
                AUTO_FORWARD_RAW_BY_TF[tf_name] = forward_raw

        try:
            PRECOMP[tf_name] = _standardize_ohlcv(precompute_indicators(raw_df.copy(), combined_params, typical_params=globals().get("typical_params")))
        except Exception as e:
            print(f"⚠️ precompute_indicators failed for {tf_name}; using raw OHLCV + on-demand indicators. Error: {e}")
            PRECOMP[tf_name] = raw_df

    ACTIVE_TF = ACTIVE_TIMEFRAMES[0]
    df_tf = PRECOMP[ACTIVE_TF]
    return PRECOMP



def _run_current_profile_and_export() -> Tuple[pd.DataFrame, pd.DataFrame, str, str]:
    """Run the currently applied profile across ACTIVE_TIMEFRAMES and export combined results."""
    global ALL_CLEAN, ALL_SOFT, PROFILE_RUNTIME_GUARD_HIT, LAST_PROFILE_COMPLETED
    PROFILE_RUNTIME_GUARD_HIT = False
    LAST_PROFILE_COMPLETED = True
    _prepare_precomp_for_active_timeframes()

    all_clean_frames, all_soft_frames = [], []
    for tf_name in ACTIVE_TIMEFRAMES:
        print(f"\n==============================")
        print(f"▶ Running timeframe: {tf_name}")
        print(f"==============================")
        tf_clean, tf_soft = run_timeframe_pipeline(tf_name, PRECOMP[tf_name])
        if not tf_clean.empty:
            all_clean_frames.append(tf_clean)
        if not tf_soft.empty:
            all_soft_frames.append(tf_soft)
        if PROFILE_RUNTIME_GUARD_HIT:
            LAST_PROFILE_COMPLETED = False
            print("⏹️ Profile runtime guard hit; stopping remaining timeframes for this phase.")
            break

    LAST_PROFILE_COMPLETED = not PROFILE_RUNTIME_GUARD_HIT
    ALL_CLEAN = pd.concat(all_clean_frames, ignore_index=True) if all_clean_frames else pd.DataFrame()
    ALL_SOFT = pd.concat(all_soft_frames, ignore_index=True) if all_soft_frames else pd.DataFrame()

    combined_clean_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_top_strategies_{RUN_ID}.csv")
    combined_soft_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_top_strategies_{RUN_ID}_SOFT.csv")

    if not ALL_CLEAN.empty:
        ALL_CLEAN = _apply_quality_gate(ALL_CLEAN, RUN_MODE, verbose=True, label="combined")
        ALL_CLEAN = ALL_CLEAN.sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
        ALL_CLEAN = deduplicate_ranked_results(ALL_CLEAN, corr_threshold=0.98)
        _csv_safe(ALL_CLEAN).head(200).to_csv(combined_clean_csv, index=False)
        print(f"\n✅ Exported combined CLEAN strategies → {combined_clean_csv}")
    elif not ALL_SOFT.empty:
        ALL_SOFT = ALL_SOFT.sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
        ALL_SOFT = deduplicate_ranked_results(ALL_SOFT, corr_threshold=0.98)
        _csv_safe(ALL_SOFT).head(200).to_csv(combined_soft_csv, index=False)
        print("\n⚠️ No strategy passed strict filters. Exported SOFT top instead.")
        print(f"📤 {combined_soft_csv}")
    else:
        print("\n⚠️ No strategies produced usable results. Check *_EVAL_RAW.pkl files and smoke-test output.")

    if not ALL_CLEAN.empty:
        display(_csv_safe(ALL_CLEAN).head(20))
    elif not ALL_SOFT.empty:
        display(_csv_safe(ALL_SOFT).head(20))
    return ALL_CLEAN, ALL_SOFT, combined_clean_csv, combined_soft_csv


def _parse_indicator_family(indicators_value) -> Tuple[str, ...]:
    if isinstance(indicators_value, str):
        parts = [x.strip() for x in indicators_value.split(",") if x.strip()]
    elif isinstance(indicators_value, (list, tuple, set)):
        parts = [str(x).strip() for x in indicators_value if str(x).strip()]
    else:
        parts = []
    return tuple(sorted(parts))


def _select_auto_candidates(sample_clean: pd.DataFrame, top_n: int = 8, max_per_tf_family: int = 1) -> pd.DataFrame:
    """Choose diversified candidates from sampled CLEAN results for focused WFA."""
    if sample_clean is None or sample_clean.empty:
        return pd.DataFrame()
    work = sample_clean.copy()
    if "Stopped Early" in work.columns:
        work = work[work["Stopped Early"] == False]
    # Re-apply the sampled gate explicitly because candidates become the expensive WFA input.
    work = _apply_quality_gate(work, "SAMPLED_ALL_TF", verbose=True, label="auto-candidate-pool")
    if work.empty:
        return pd.DataFrame()
    work["Family"] = work["Indicators"].apply(_parse_indicator_family)
    work = work.sort_values(["Strategy Score", "Profit Factor", "Max Drawdown %"], ascending=[False, False, True]).reset_index(drop=True)
    # Diversify: avoid sending many near-identical configs from the same timeframe/family into WFA.
    picked = []
    counts = {}
    for _, row in work.iterrows():
        key = (row.get("Timeframe"), row.get("Family"))
        if counts.get(key, 0) >= int(max_per_tf_family):
            continue
        picked.append(row)
        counts[key] = counts.get(key, 0) + 1
        if len(picked) >= int(top_n):
            break
    return pd.DataFrame(picked).reset_index(drop=True) if picked else pd.DataFrame()


def _candidate_focus(candidate_df: pd.DataFrame) -> Tuple[List[Tuple[str, ...]], List[str]]:
    if candidate_df is None or candidate_df.empty:
        return [], []
    combos = []
    for fam in candidate_df["Indicators"].apply(_parse_indicator_family).tolist():
        if fam and fam not in combos:
            combos.append(fam)
    tfs = [tf for tf in candidate_df["Timeframe"].dropna().astype(str).drop_duplicates().tolist() if tf]
    return combos, tfs




def _find_saved_config_by_id(strategy_id: str, tf_name: str, sample_run_id: str, cfg_batch_path: Optional[str] = None) -> Tuple[Optional[dict], Optional[str]]:
    """Locate one exact sampled config in saved CFGS.pkl checkpoints.

    V3.9 prefers the direct Config Batch Path recorded during sample evaluation.
    This avoids slow Drive-wide globbing and prevents accidental matches from older runs.
    """
    strategy_id = str(strategy_id)
    tf_name = str(tf_name)
    cfg_files = []
    if cfg_batch_path is not None and str(cfg_batch_path).strip() and str(cfg_batch_path).lower() not in {"nan", "none"}:
        direct_path = str(cfg_batch_path).strip()
        if os.path.exists(direct_path):
            cfg_files.append(direct_path)
        else:
            print(f"⚠️ Direct config path not found for {strategy_id}: {direct_path}")
    run_glob = sorted(glob.glob(os.path.join(OUTPUT_ROOT, f"{sample_run_id}_{tf_name}", "*_CFGS.pkl")))
    for p in run_glob:
        if p not in cfg_files:
            cfg_files.append(p)
    if not cfg_files:
        # Last-resort fallback only. Prefer direct path / run folder above.
        cfg_files = sorted(glob.glob(os.path.join(OUTPUT_ROOT, f"**/*{tf_name}*_CFGS.pkl"), recursive=True))
    for cfg_path in cfg_files:
        try:
            with open(cfg_path, "rb") as f:
                cfg_list = pickle.load(f)
        except Exception:
            continue
        if not isinstance(cfg_list, list):
            continue
        for cfg in cfg_list:
            if isinstance(cfg, dict) and str(cfg.get("id")) == strategy_id:
                return copy.deepcopy(cfg), cfg_path
    return None, None



def _load_exact_candidate_configs(candidate_df: pd.DataFrame, sample_run_id: str) -> Tuple[Dict[str, List[dict]], pd.DataFrame]:
    """Load exact sampled configs. Preserves indicators, params, exits, sizing, filters, and timeframe."""
    if candidate_df is None or candidate_df.empty:
        return {}, pd.DataFrame()
    cfgs_by_tf: Dict[str, List[dict]] = {}
    loaded_rows = []
    for idx, row in candidate_df.reset_index(drop=True).iterrows():
        sid = str(row.get("Strategy ID"))
        tf = str(row.get("Timeframe"))
        cfg, cfg_path = _find_saved_config_by_id(sid, tf, sample_run_id, row.get("Config Batch Path"))
        if cfg is None:
            print(f"⚠️ Exact config not found for candidate {sid} ({tf}); skipping")
            continue
        sample_cfg = copy.deepcopy(cfg)
        cfg = copy.deepcopy(cfg)
        cfg["id"] = sid
        cfg["sample_strategy_id"] = sid
        cfg["sample_config_path"] = cfg_path
        cfg["sample_rank"] = int(idx + 1)
        cfg["sample_config_fingerprint"] = assert_exact_forward_config_identity(sample_cfg, cfg)
        cfgs_by_tf.setdefault(tf, []).append(cfg)
        r = row.to_dict()
        r["Loaded Config Path"] = cfg_path
        r["Exact Config Loaded"] = True
        r["Sample Config Fingerprint"] = cfg["sample_config_fingerprint"]
        loaded_rows.append(r)
    return cfgs_by_tf, pd.DataFrame(loaded_rows)


def _evaluate_exact_candidate_wf_chunk(live_cfgs: List[dict], df: pd.DataFrame, tf_name: str) -> pd.DataFrame:
    """Forward/WFA test exact sampled configs without regenerating any new configs."""
    wf_cfg = WF_SETTINGS_BY_TF.get(tf_name, {
        "train_window": int(CFG.get("wf_train_window", 3000)),
        "test_window": int(CFG.get("wf_test_window", 500)),
        "step_size": int(CFG.get("wf_step_size", 500)),
        "top_n_train": int(CFG.get("wf_top_n_train", 3)),
    })
    try:
        windows = _split_walk_forward_windows(
            df,
            train_window=int(wf_cfg["train_window"]),
            test_window=int(wf_cfg["test_window"]),
            step_size=int(wf_cfg["step_size"]),
            timeframe=tf_name,
        )
    except UnsafeEvaluationError as e:
        print(f"⚠️ {tf_name}: exact candidate WFA skipped because a safe plan cannot be formed: {e}")
        return pd.DataFrame()

    all_tests = []
    for w in windows:
        test_df = df.iloc[w["test_start"]:w["test_end"]].copy()
        test_eval = _evaluate_live_configs(live_cfgs, test_df, tf_name, phase="EXACT_TEST", window_id=w["window"])
        if not test_eval.empty:
            all_tests.append(test_eval)
    test_rows = pd.concat(all_tests, ignore_index=True) if all_tests else pd.DataFrame()
    if test_rows.empty:
        return pd.DataFrame()
    agg = _aggregate_walkforward_oos(
        test_rows,
        min_profitable_window_ratio=0.60,
        min_windows=min(3, len(windows)),
        min_oos_trades=max(10, MIN_TRADES),
    )
    if not agg.empty:
        agg["Phase"] = "EXACT_OOS_AGG"
        agg["Exact Config Forward Test"] = True
    return agg


def _run_exact_candidate_forward(candidate_df: pd.DataFrame, sample_run_id: str, base_run_id: str) -> Tuple[pd.DataFrame, pd.DataFrame, str, str]:
    """Run exact candidate configs through rolling OOS windows and export CLEAN/SOFT outputs."""
    global RUN_ID, RUN_MODE, PIPELINE_START_TIME, ALL_CLEAN, ALL_SOFT

    exact_cfgs_by_tf, loaded_candidates = _load_exact_candidate_configs(candidate_df, sample_run_id)
    exact_candidate_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_exact_loaded_candidates_{base_run_id}.csv")
    if not loaded_candidates.empty:
        _csv_safe(loaded_candidates).to_csv(exact_candidate_csv, index=False)
        print(f"\n✅ Loaded exact candidate configs → {exact_candidate_csv}")
        display(_csv_safe(loaded_candidates).head(20))
    else:
        print("\n⚠️ No exact candidate config could be loaded. Exact forward test skipped.")
        return pd.DataFrame(), pd.DataFrame(), "", ""

    exact_tfs = list(exact_cfgs_by_tf.keys())
    wf_profile = copy.deepcopy(MODE_SETTINGS["AUTO_SAMPLE_THEN_WF"].get("focused_wf", {}))
    wf_profile["timeframes"] = exact_tfs
    wf_profile["total_configs_cap"] = None
    wf_profile["max_batches_per_tf"] = None
    wf_profile["batch_size"] = max(1, min(20, max(len(v) for v in exact_cfgs_by_tf.values())))
    wf_profile["max_filters"] = 0
    wf_profile["filter_samples"] = 1

    RUN_ID = f"{base_run_id}_EXACT_WF"
    PIPELINE_START_TIME = time.time()
    _runtime_cfg_from_profile(wf_profile, "WALK_FORWARD")
    RUN_ID = f"{base_run_id}_EXACT_WF"
    _prepare_precomp_for_active_timeframes()

    all_clean_frames, all_soft_frames = [], []
    for tf_name, saved_cfgs in exact_cfgs_by_tf.items():
        if tf_name not in PRECOMP:
            print(f"⚠️ Missing precomputed data for {tf_name}; skipping exact WFA")
            continue
        print(f"\n==============================")
        print(f"▶ Exact candidate Forward/WFA: {tf_name} | configs={len(saved_cfgs)}")
        print(f"==============================")
        tf_dir = os.path.join(OUTPUT_ROOT, f"{RUN_ID}_{tf_name}")
        os.makedirs(tf_dir, exist_ok=True)
        cfg_path = os.path.join(tf_dir, "exact_candidates_CFGS.pkl")
        raw_path = os.path.join(tf_dir, "exact_candidates_EVAL_RAW.pkl")
        clean_path = os.path.join(tf_dir, "exact_candidates_CLEAN.pkl")
        soft_path = os.path.join(tf_dir, "exact_candidates_SOFT.pkl")

        with open(cfg_path, "wb") as f:
            pickle.dump(saved_cfgs, f)

        live_cfgs = []
        for saved_cfg in saved_cfgs:
            try:
                live = rebind_config_from_pickle(saved_cfg, entry_functions, short_entry_functions, skip_indicators=SKIP_INDICATORS)
                live = _with_timeframe_friction(live, tf_name)
                live_cfgs.append(live)
            except Exception as e:
                print(f"⚠️ exact candidate rebind error for {saved_cfg.get('id')}: {e}")
        if not live_cfgs:
            continue

        eval_df = _evaluate_exact_candidate_wf_chunk(live_cfgs, PRECOMP[tf_name], tf_name)
        with open(raw_path, "wb") as f:
            pickle.dump(eval_df, f)

        cleaned_df = clean_strategies(eval_df, capital=INITIAL_CAPITAL, min_trades=MIN_TRADES, max_mdd_pct=0.50, win_rate_cap=0.995, allow_stopped_early=False)
        if not cleaned_df.empty and "OOS Pass Rule" in cleaned_df.columns:
            cleaned_df = cleaned_df[cleaned_df["OOS Pass Rule"] == True].reset_index(drop=True)
        cleaned_df = _apply_quality_gate(cleaned_df, "WALK_FORWARD", verbose=True, label=f"exact {tf_name}")

        if cleaned_df.empty and eval_df is not None and not eval_df.empty and SOFT_KEEP_TOP_N > 0:
            soft = eval_df.sort_values(["Strategy Score", "CAGR"], ascending=False).head(SOFT_KEEP_TOP_N).copy()
            soft["Exact Config Forward Test"] = True
            all_soft_frames.append(soft)
            with open(soft_path, "wb") as f:
                pickle.dump(soft, f)
        if not cleaned_df.empty:
            cleaned_df["Exact Config Forward Test"] = True
            all_clean_frames.append(cleaned_df)

        with open(clean_path, "wb") as f:
            pickle.dump(cleaned_df, f)
        print(f"💾 Exact WFA {tf_name}: raw={len(eval_df) if eval_df is not None else 0} | clean={len(cleaned_df)}")

    ALL_CLEAN = pd.concat(all_clean_frames, ignore_index=True) if all_clean_frames else pd.DataFrame()
    ALL_SOFT = pd.concat(all_soft_frames, ignore_index=True) if all_soft_frames else pd.DataFrame()

    combined_clean_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_top_strategies_{RUN_ID}.csv")
    combined_soft_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_top_strategies_{RUN_ID}_SOFT.csv")
    if not ALL_CLEAN.empty:
        sort_cols = [c for c in ["OOS Pass Rule", "OOS Total Net Profit", "OOS Profitable Window %", "OOS Worst Drawdown %", "OOS Median Sharpe", "OOS Total Trades", "Strategy Score"] if c in ALL_CLEAN.columns]
        if sort_cols:
            ascending = [False if c not in {"OOS Worst Drawdown %"} else True for c in sort_cols]
            ALL_CLEAN = ALL_CLEAN.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
        else:
            ALL_CLEAN = ALL_CLEAN.sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
        ALL_CLEAN = deduplicate_ranked_results(ALL_CLEAN, corr_threshold=0.98)
        _csv_safe(ALL_CLEAN).head(200).to_csv(combined_clean_csv, index=False)
        print(f"\n✅ Exported EXACT CONFIG Forward CLEAN strategies → {combined_clean_csv}")
        display(_csv_safe(ALL_CLEAN).head(20))
    elif not ALL_SOFT.empty:
        ALL_SOFT = ALL_SOFT.sort_values(["Strategy Score", "CAGR"], ascending=False).reset_index(drop=True)
        ALL_SOFT = deduplicate_ranked_results(ALL_SOFT, corr_threshold=0.98)
        _csv_safe(ALL_SOFT).head(200).to_csv(combined_soft_csv, index=False)
        print("\n⚠️ No exact candidate passed strict forward filters. Exported SOFT top instead.")
        print(f"📤 {combined_soft_csv}")
        display(_csv_safe(ALL_SOFT).head(20))
    else:
        print("\n⚠️ Exact candidate forward produced no usable rows.")
    return ALL_CLEAN, ALL_SOFT, combined_clean_csv, combined_soft_csv

# Keep a stable base run id for multi-phase auto orchestration.
BASE_RUN_MODE = RUN_MODE
BASE_RUN_ID = RUN_ID

if RUN_MODE == "AUTO_SAMPLE_THEN_WF":
    print("\n🤖 AUTO_SAMPLE_THEN_WF enabled")
    print("   Phase 1: sample all available timeframes with quality gates")
    print("   Phase 2: select best exact configs across all timeframes")
    print("   Phase 3: forward/Walk-Forward test the exact sampled configs only")

    # -------------------------
    # Phase 1: all-TF sample using chronological sample/forward split
    # -------------------------
    AUTO_HOLDOUT_ENABLED = True
    AUTO_FORWARD_RAW_BY_TF = {}
    AUTO_SAMPLE_FORWARD_RATIO = float(MODE_SETTINGS["AUTO_SAMPLE_THEN_WF"].get("sample_forward_split_ratio", 0.70))
    SAMPLE_RUN_ID = f"{BASE_RUN_ID}_SAMPLE"
    RUN_ID = SAMPLE_RUN_ID
    FOCUSED_INDICATOR_COMBOS = None
    PIPELINE_START_TIME = time.time()
    _runtime_cfg_from_profile(MODE_SETTINGS["AUTO_SAMPLE_THEN_WF"], "SAMPLED_ALL_TF")
    sample_clean, sample_soft, sample_clean_csv, sample_soft_csv = _run_current_profile_and_export()
    sample_phase_complete = bool(LAST_PROFILE_COMPLETED)
    if not sample_phase_complete:
        print("\n⚠️ Sample phase stopped by runtime guard before completing all requested timeframes.")
        print("   Exact Forward/WFA will be skipped to avoid selecting from a partial sample universe.")
        print(f"   Resume sample first with RUN_ID_OVERRIDE = '{SAMPLE_RUN_ID}' if you want to continue this run.")

    # -------------------------
    # Phase 2: exact candidate selection
    # -------------------------
    candidates = _select_auto_candidates(
        sample_clean,
        top_n=int(MODE_SETTINGS["AUTO_SAMPLE_THEN_WF"].get("candidate_top_n", 8)),
        max_per_tf_family=int(MODE_SETTINGS["AUTO_SAMPLE_THEN_WF"].get("candidate_max_per_tf_family", 1)),
    )
    candidate_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_auto_candidates_{BASE_RUN_ID}.csv")
    if not candidates.empty:
        _csv_safe(candidates).to_csv(candidate_csv, index=False)
        print(f"\n✅ Auto-selected EXACT sampled configs for Forward/WFA → {candidate_csv}")
        display(_csv_safe(candidates).head(20))
    else:
        print("\n⚠️ No sampled CLEAN candidate passed the auto-candidate gate. Exact Forward/WFA skipped.")
        print("   Try SAMPLED_ALL_TF with more coverage or relax the sampled quality gate slightly.")

    # -------------------------
    # Phase 3: exact-config Forward / WFA
    # -------------------------
    if not candidates.empty and sample_phase_complete:
        focus_combos, focus_tfs = _candidate_focus(candidates)
        print(f"\n🎯 Candidate families for context only: {focus_combos}")
        print(f"🎯 Candidate timeframes: {focus_tfs}")
        print("🎯 Forward/WFA will use exact candidate configs loaded from the SAMPLE CFGS.pkl checkpoints.")
        wf_clean, wf_soft, wf_clean_csv, wf_soft_csv = _run_exact_candidate_forward(
            candidates,
            sample_run_id=SAMPLE_RUN_ID,
            base_run_id=BASE_RUN_ID,
        )
    elif not candidates.empty and not sample_phase_complete:
        print("\n⏸️ Exact Forward/WFA skipped because sample phase is incomplete. This is intentional in V3.9.")
else:
    # Normal one-phase execution for SMOKE / SAMPLED / SAMPLED_H1 / WALK_FORWARD / etc.
    _runtime_cfg_from_profile(CFG, RUN_MODE)
    _run_current_profile_and_export()


🚀 Starting XAUUSD pipeline | run_mode=AUTO_SAMPLE_THEN_WF | run_id=AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE
📌 Active timeframes: ['M30', 'H1', 'H4', 'M15']
📌 Runtime guard: max_runtime_minutes=150 | max_batches_per_tf=30 | resume=True
📌 Batch/config: batch_size=50 | cap=1500 | max_params_per_combo=12 | settings_samples=2 | max_filters=1
📌 Data row cap: max_data_rows=None | by_tf={'M15': 12000, 'M30': 12000, 'H1': 8000, 'H4': 5000}
📌 Execution mode: next_bar_open | exit policy: SL_FIRST

🤖 AUTO_SAMPLE_THEN_WF enabled
   Phase 1: sample all available timeframes with quality gates
   Phase 2: select best exact configs across all timeframes
   Phase 3: forward/Walk-Forward test the exact sampled configs only
⏱️ Runtime-safe data cap for M30: using last 12,000 of 30,197 rows
🔒 Holdout split M30: sample=8,400 rows (2024-07-17 01:30:00 → 2025-04-03 02:00:00) | forward=3,600 rows (2025-04-03 02:30:00 → 2025-07-23 17:00:00)
⏱️ Runtime-safe data cap for H1: using last 8,000 of 15,109 rows
🔒 Hold

M30 Batches: 0it [00:00, ?it/s]

📊 Generating configs | combos=120 | filters=2 | cap=1500



M30 Batches: 1it [00:54, 54.20s/it]
🔄 Indicator Combos:   2%|▎         | 3/120 [00:54<35:15, 18.08s/it]

🧹 Quality gate M30 batch_0_49: 8 → 0 | PF>=1.2, DD<=15%, Sharpe>=0.5, NetProfit>=0.0
💾 M30 batch_0_49: raw=50 | clean=0


...

,Net Profit,CAGR,Max Drawdown,Max Drawdown %,Volatility,Sharpe Ratio,Sortino Ratio,Win Rate,Profit Factor,# Trades,...,Indicators,Config Signature,Timeframe,Phase,Window,Final Capital,RUN_MODE,Config Batch Path,Config Batch Name,Family
0,123.578809,0.189026,39.234249,0.036842,0.083856,2.106805,0.529619,0.593750,1.978279,32,...,"EMA,BollingerBands",222596313d266f45a1c325b2dbf861dd,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1123.578809,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(BollingerBands, EMA)"
1,267.808394,0.302722,76.305752,0.067228,0.151698,1.819492,1.430359,0.409639,1.497120,83,...,"EMA,ATR",ed9a4cf6c7a6fc95faafe46ff881b2bb,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1267.808394,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(ATR, EMA)"
2,305.484753,0.345941,110.394954,0.098408,0.178199,1.756413,1.425431,0.421569,1.475317,102,...,"EMA,ATR,Fibonacci",5233f46cc7462d1a05c329b920323af8,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1305.484753,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_800_849,"(ATR, EMA, Fibonacci)"
3,110.316875,0.168232,60.816250,0.056393,0.100210,1.601920,0.425428,0.677419,2.101817,31,...,"EMA,BollingerBands",589869ccf3e5e3657c2dba4bb024888d,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1110.316875,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(BollingerBands, EMA)"
4,121.522431,0.406107,95.113124,0.086198,0.200793,1.797959,0.764308,0.387097,1.496498,31,...,"ATR,Stochastic,Fibonacci",0c2ba0bae91b57113f2f0150539d07fb,M15,SAMPLED_ALL_TF_FULL_SAMPLE,None,1121.522431,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_1300_1349,"(ATR, Fibonacci, Stochastic)"
5,170.718236,0.192035,51.721336,0.050939,0.113780,1.600975,1.002266,0.392157,1.569666,51,...,"EMA,ATR",cfc5cd09bcb876a24659acfc947a0875,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1170.718236,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(ATR, EMA)"
6,225.292338,0.254128,105.227530,0.093132,0.163625,1.465771,1.099347,0.428571,1.440321,84,...,"EMA,ATR,Fibonacci",5515a02d96ad305e3eceb0395f1d214b,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1225.292338,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_800_849,"(ATR, EMA, Fibonacci)"
7,124.581575,0.190603,68.005733,0.062843,0.108537,1.661707,0.540487,0.470588,1.573709,51,...,"EMA,BollingerBands,Fibonacci",d5483820339eedc754a43b4780678301,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1124.581575,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_750_799,"(BollingerBands, EMA, Fibonacci)"
8,27.276667,0.030447,32.677917,0.030830,0.046784,0.664444,0.191550,0.514286,1.325072,35,...,"MACD,Stochastic",7a02528eaf97a68c3f02af43d76f9a8d,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1027.276667,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_300_349,"(MACD, Stochastic)"
9,55.072667,0.082917,91.700977,0.085712,0.135949,0.654005,0.283924,0.488372,1.200812,43,...,"RSI,BollingerBands,ATR",45ed88705d8a24be58d25a502252b015,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1055.072667,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_950_999,"(ATR, BollingerBands, RSI)"



🎯 Candidate families for context only: [('BollingerBands', 'EMA'), ('ATR', 'EMA'), ('ATR', 'EMA', 'Fibonacci'), ('ATR', 'Fibonacci', 'Stochastic'), ('BollingerBands', 'EMA', 'Fibonacci'), ('MACD', 'Stochastic'), ('ATR', 'BollingerBands', 'RSI')]
🎯 Candidate timeframes: ['M30', 'H1', 'M15']
🎯 Forward/WFA will use exact candidate configs loaded from the SAMPLE CFGS.pkl checkpoints.

✅ Loaded exact candidate configs → /content/drive/MyDrive/XAUUSD_fulltest_batches/XAUUSD_exact_loaded_candidates_AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE.csv


,Net Profit,CAGR,Max Drawdown,Max Drawdown %,Volatility,Sharpe Ratio,Sortino Ratio,Win Rate,Profit Factor,# Trades,...,Timeframe,Phase,Window,Final Capital,RUN_MODE,Config Batch Path,Config Batch Name,Family,Loaded Config Path,Exact Config Loaded
0,123.578809,0.189026,39.234249,0.036842,0.083856,2.106805,0.529619,0.593750,1.978279,32,...,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1123.578809,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(BollingerBands, EMA)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
1,267.808394,0.302722,76.305752,0.067228,0.151698,1.819492,1.430359,0.409639,1.497120,83,...,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1267.808394,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(ATR, EMA)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
2,305.484753,0.345941,110.394954,0.098408,0.178199,1.756413,1.425431,0.421569,1.475317,102,...,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1305.484753,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_800_849,"(ATR, EMA, Fibonacci)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
3,110.316875,0.168232,60.816250,0.056393,0.100210,1.601920,0.425428,0.677419,2.101817,31,...,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1110.316875,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(BollingerBands, EMA)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
4,121.522431,0.406107,95.113124,0.086198,0.200793,1.797959,0.764308,0.387097,1.496498,31,...,M15,SAMPLED_ALL_TF_FULL_SAMPLE,None,1121.522431,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_1300_1349,"(ATR, Fibonacci, Stochastic)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
5,170.718236,0.192035,51.721336,0.050939,0.113780,1.600975,1.002266,0.392157,1.569666,51,...,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1170.718236,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_50_99,"(ATR, EMA)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
6,225.292338,0.254128,105.227530,0.093132,0.163625,1.465771,1.099347,0.428571,1.440321,84,...,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1225.292338,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_800_849,"(ATR, EMA, Fibonacci)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
7,124.581575,0.190603,68.005733,0.062843,0.108537,1.661707,0.540487,0.470588,1.573709,51,...,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1124.581575,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_750_799,"(BollingerBands, EMA, Fibonacci)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
8,27.276667,0.030447,32.677917,0.030830,0.046784,0.664444,0.191550,0.514286,1.325072,35,...,H1,SAMPLED_ALL_TF_FULL_SAMPLE,None,1027.276667,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_300_349,"(MACD, Stochastic)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True
9,55.072667,0.082917,91.700977,0.085712,0.135949,0.654005,0.283924,0.488372,1.200812,43,...,M30,SAMPLED_ALL_TF_FULL_SAMPLE,None,1055.072667,SAMPLED_ALL_TF,/content/drive/MyDrive/XAUUSD_fulltest_batches...,batch_950_999,"(ATR, BollingerBands, RSI)",/content/drive/MyDrive/XAUUSD_fulltest_batches...,True


🔒 Using forward holdout only for M30: 3,600 rows
🔒 Using forward holdout only for H1: 2,400 rows
🔒 Using forward holdout only for M15: 3,600 rows

▶ Exact candidate Forward/WFA: M30 | configs=4
💾 Exact WFA M30: raw=4 | clean=0

▶ Exact candidate Forward/WFA: H1 | configs=5
💾 Exact WFA H1: raw=5 | clean=0

▶ Exact candidate Forward/WFA: M15 | configs=1
⚠️ M15: exact candidate WFA skipped because a safe plan cannot be formed
💾 Exact WFA M15: raw=1 | clean=0

⚠️ No exact candidate passed strict forward filters. Exported SOFT top instead.
📤 /content/drive/MyDrive/XAUUSD_fulltest_batches/XAUUSD_top_strategies_AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE_EXACT_WF_SOFT.csv


,Net Profit,CAGR,Max Drawdown,Max Drawdown %,Volatility,Sharpe Ratio,Sortino Ratio,Win Rate,Profit Factor,# Trades,...,OOS Median Sharpe,OOS Worst Drawdown %,OOS Avg Profit Factor Capped,OOS Min Windows Required,OOS Min Trades Required,OOS Max DD Required,OOS Min Sharpe Required,OOS Min PF Required,OOS Pass Rule,Exact Config Forward Test
0,46.652210,1.040289,51.326927,0.050395,0.244595,3.045097,1.861604,0.428571,1.669758,7,...,3.045097,0.050395,1.669758,1.0,30.0,0.2,0.3,1.15,False,True
1,42.010643,0.903287,54.909576,0.053913,0.237517,2.835018,1.995044,0.428571,1.565443,7,...,2.835018,0.053913,1.565443,1.0,30.0,0.2,0.3,1.15,False,True
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,1.0,30.0,0.2,0.3,1.15,False,True
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,1.0,30.0,0.2,0.3,1.15,False,True
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,1.0,30.0,0.2,0.3,1.15,False,True
5,-57.782138,-0.338191,80.382893,0.080042,0.243080,-1.577075,-0.664080,0.450000,0.631648,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
6,-26.918729,-0.245297,28.550992,0.031864,0.121727,-2.215229,-0.814500,0.333333,0.314658,5,...,-2.215229,0.031864,0.314658,2.0,30.0,0.2,0.3,1.15,False,True
7,-44.553542,-0.353548,21.001458,0.035097,0.062321,-5.441082,-0.849740,0.500000,0.250695,4,...,-5.441082,0.035097,0.250695,2.0,30.0,0.2,0.3,1.15,False,True
8,-37.180581,-0.323272,20.091054,0.020091,0.077257,-5.872114,-0.594793,0.000000,0.000000,5,...,-5.872114,0.020091,0.000000,2.0,30.0,0.2,0.3,1.15,False,True
9,-48.667083,-0.401160,24.901667,0.028090,0.078065,-6.776950,-0.900160,0.000000,0.032238,5,...,-6.776950,0.028090,0.032238,2.0,30.0,0.2,0.3,1.15,False,True


In [ ]:
# ============================================================
# INSPECT & RE-RUN STRATEGY FROM THE LATEST PIPELINE RUN — V3.9
# ============================================================
import os, pickle, glob
import pandas as pd
import numpy as np

# Selection controls:
# - Set INSPECT_RANK = 1 to inspect Rank 2, etc.
# - Set INSPECT_STRATEGY_ID = "..." to inspect a specific strategy.
# - AUTO_SELECT_RELIABLE=True picks the first row with enough trades and not stopped early.
INSPECT_RANK = int(globals().get("INSPECT_RANK", 0))
INSPECT_STRATEGY_ID = globals().get("INSPECT_STRATEGY_ID", None)
AUTO_SELECT_RELIABLE = bool(globals().get("AUTO_SELECT_RELIABLE", True))
MIN_INSPECT_TRADES = int(globals().get("MIN_INSPECT_TRADES", globals().get("MIN_RELIABLE_TRADES_FOR_REVIEW", 30)))

# Prefer CLEAN export; if unavailable, use SOFT export.
clean_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_top_strategies_{RUN_ID}.csv")
soft_csv = os.path.join(OUTPUT_ROOT, f"XAUUSD_top_strategies_{RUN_ID}_SOFT.csv")

if os.path.exists(clean_csv):
    top_csv_path = clean_csv
    source_type = "CLEAN"
elif os.path.exists(soft_csv):
    top_csv_path = soft_csv
    source_type = "SOFT"
else:
    candidates = sorted(glob.glob(os.path.join(OUTPUT_ROOT, f"**/*{RUN_ID}*.csv"), recursive=True))
    if not candidates:
        raise FileNotFoundError("No exported strategy CSV found. Run the pipeline cell first.")
    top_csv_path = candidates[0]
    source_type = "FALLBACK"

print(f"📂 Loading {source_type} top strategies from: {top_csv_path}")
top_df = pd.read_csv(top_csv_path)
if top_df.empty:
    raise RuntimeError("Top strategy CSV is empty.")

sort_cols = [c for c in ["Strategy Score", "CAGR", "Net Profit"] if c in top_df.columns]
top_df = top_df.sort_values(sort_cols, ascending=False).reset_index(drop=True) if sort_cols else top_df.reset_index(drop=True)

print("\n🏆 Top 10 strategies:")
display(top_df.head(10))

# -------------------- choose row --------------------
selected_idx = max(0, min(INSPECT_RANK, len(top_df) - 1))
if INSPECT_STRATEGY_ID:
    matches = top_df.index[top_df["Strategy ID"].astype(str) == str(INSPECT_STRATEGY_ID)].tolist()
    if not matches:
        raise RuntimeError(f"INSPECT_STRATEGY_ID not found in top_df: {INSPECT_STRATEGY_ID}")
    selected_idx = matches[0]
elif AUTO_SELECT_RELIABLE:
    reliable = top_df.copy()
    if "# Trades" in reliable.columns:
        reliable = reliable[reliable["# Trades"] >= MIN_INSPECT_TRADES]
    if "Stopped Early" in reliable.columns:
        reliable = reliable[reliable["Stopped Early"].astype(str).str.lower().isin(["false", "0", "nan", "none"])]
    if not reliable.empty:
        selected_idx = int(reliable.index[0])
        if selected_idx != INSPECT_RANK:
            print(f"\n🔎 AUTO_SELECT_RELIABLE selected Rank {selected_idx + 1} instead of Rank {INSPECT_RANK + 1} because it has >= {MIN_INSPECT_TRADES} trades and did not stop early.")
    else:
        print(f"\n⚠️ No row met AUTO_SELECT_RELIABLE criteria. Falling back to Rank {selected_idx + 1}.")

best_row = top_df.iloc[selected_idx]
best_id = best_row["Strategy ID"]
best_tf = best_row["Timeframe"] if "Timeframe" in top_df.columns else ACTIVE_TF
print(f"\n⭐ Selected Strategy ID: {best_id}")
print(f"🏅 Selected Rank      : {selected_idx + 1}")
print(f"🕒 Timeframe          : {best_tf}")
if "Phase" in top_df.columns and str(best_row.get("Phase")) == "OOS_AGG":
    print("ℹ️ OOS aggregate row detected. The re-run below is a full-period sanity check for this config, not the same as the aggregated OOS score.")

# Locate config in the timeframe batch folders.
tf_dir = os.path.join(OUTPUT_ROOT, f"{RUN_ID}_{best_tf}")
cfg_files = sorted(glob.glob(os.path.join(tf_dir, "*_CFGS.pkl")))
if not cfg_files:
    cfg_files = sorted(glob.glob(os.path.join(OUTPUT_ROOT, f"**/*{best_tf}*_CFGS.pkl"), recursive=True))

best_cfg = None
best_cfg_batch = None
for full_path in cfg_files:
    with open(full_path, "rb") as f:
        cfg_list = pickle.load(f)
    for cfg in cfg_list:
        if cfg.get("id") == best_id:
            best_cfg = cfg
            best_cfg_batch = full_path
            break
    if best_cfg is not None:
        break

if best_cfg is None:
    raise RuntimeError(f"Could not find config for Strategy ID: {best_id}")

print(f"\n🔍 Found config in: {best_cfg_batch}")

best_cfg_live = rebind_config_from_pickle(
    best_cfg,
    entry_functions,
    short_entry_functions,
    skip_indicators=SKIP_INDICATORS,
)
# Important: saved CFG files contain raw/default friction. Inject timeframe-specific friction before print and re-run.
best_cfg_live = _with_timeframe_friction(best_cfg_live, str(best_tf))

entry_meta = best_cfg_live.get("entry", {})
print("\n📋 Strategy details:")
print(f"  • Indicators    : {entry_meta.get('indicators')}")
print(f"  • Strategy Name : {entry_meta.get('strategy_name')}")
print(f"  • Entry Logic   : {best_cfg_live.get('entry_logic')}")
print(f"  • Params        : {best_cfg_live.get('params')}")
print(f"  • Exit          : {best_cfg_live.get('exit')}")
print(f"  • Sizing        : {best_cfg_live.get('sizing')}")
print(f"  • Friction      : {best_cfg_live.get('friction')}")
print(f"  • Filters       : {best_cfg_live.get('filters')}")

# Re-run on that timeframe using the final patched engine.
df_best = PRECOMP[best_tf]
bars_per_year = BARS_PER_YEAR_BY_TF.get(best_tf, 6048)
trades, final_cap, equity = run_backtest(
    best_cfg_live,
    df_best,
    initial_capital=INITIAL_CAPITAL,
    broker_spec=XAUUSD_SPEC,
    verbose=False,
)
metrics = compute_strategy_metrics(
    trades,
    equity_curve=equity,
    initial_capital=INITIAL_CAPITAL,
    bars_per_year=bars_per_year,
)

print("\n🔁 Re-run result with final engine")
print("  ---------------------------")
print(f"  Strategy ID      : {best_id}")
print(f"  Timeframe        : {best_tf}")
print(f"  Strategy Name    : {entry_meta.get('strategy_name')}")
print("  ---------------------------")
print(f"  Net Profit       : {metrics['Net Profit']:.2f}")
print(f"  CAGR             : {metrics['CAGR']:.4%}")
print(f"  # Trades         : {metrics['# Trades']}")
print(f"  Win Rate         : {metrics['Win Rate']:.2%}")
print(f"  Profit Factor    : {metrics['Profit Factor']:.2f}")
print(f"  Max Drawdown     : {metrics['Max Drawdown']:.2f}")
print(f"  Max Drawdown %   : {metrics['Max Drawdown %']:.2%}")
print(f"  Sharpe Ratio     : {metrics['Sharpe Ratio']:.2f}")
print(f"  Sortino Ratio    : {metrics['Sortino Ratio']:.2f}")
print(f"  Expectancy       : {metrics['Expectancy']:.2f}")
print(f"  Stopped Early    : {bool(metrics.get('Stopped Early', False))}")
print(f"  Strategy Score   : {metrics['Strategy Score']:.2f}")
print("  ---------------------------")
print(f"  Final Capital    : {final_cap:.2f}")
print(f"  Equity points    : {len(equity)}")

# Validation warnings: prevent low-trade/stopped-early rows from being mistaken for final strategies.
warnings = []
best_run_mode = str(best_row.get("RUN_MODE", RUN_MODE)).upper()
best_phase = str(best_row.get("Phase", ""))
if best_run_mode == "SMOKE":
    warnings.append("This result is from SMOKE mode. Use it only to validate the pipeline, not for final strategy selection.")
if source_type == "SOFT":
    warnings.append("This row came from SOFT fallback, meaning it did not pass the strict CLEAN filters.")
if metrics.get("# Trades", 0) < MIN_INSPECT_TRADES:
    warnings.append(f"Too few trades: {metrics.get('# Trades', 0)} trades. Metrics are not statistically reliable yet.")
if bool(metrics.get("Stopped Early", False)):
    warnings.append("Strategy hit max-drawdown cutoff and stopped early in the re-run.")
if "QUALITY_GATES" in globals():
    gate = QUALITY_GATES.get(best_run_mode, QUALITY_GATES.get(RUN_MODE, {}))
    if gate:
        if metrics.get("Net Profit", 0) < gate.get("min_net_profit", -np.inf):
            warnings.append(f"Net Profit is below the quality gate: {metrics.get('Net Profit', 0):.2f} < {gate.get('min_net_profit')}.")
        if metrics.get("Profit Factor", 0) < gate.get("min_profit_factor", 0):
            warnings.append(f"Profit Factor is below the quality gate: {metrics.get('Profit Factor', 0):.2f} < {gate.get('min_profit_factor')}.")
        if metrics.get("Max Drawdown %", 1) > gate.get("max_dd_pct", 1):
            warnings.append(f"Max Drawdown % is above the quality gate: {metrics.get('Max Drawdown %', 0):.2%} > {gate.get('max_dd_pct'):.0%}.")
        if metrics.get("Sharpe Ratio", -999) < gate.get("min_sharpe", -999):
            warnings.append(f"Sharpe Ratio is below the quality gate: {metrics.get('Sharpe Ratio', 0):.2f} < {gate.get('min_sharpe')}.")
if best_phase in {"FULL", "FALLBACK_FULL"}:
    warnings.append("Legacy phase label detected. Re-run Cell 18 after the latest patch to get explicit phase labels such as SAMPLED_FULL_SAMPLE.")
friction_bpd = best_cfg_live.get("friction", {}).get("bars_per_day")
expected_bpd = BARS_PER_DAY_BY_TF.get(str(best_tf))
if expected_bpd is not None and int(friction_bpd) != int(expected_bpd):
    warnings.append(f"bars_per_day mismatch: config={friction_bpd}, expected={expected_bpd} for {best_tf}.")
if warnings:
    print("\n⚠️ Strategy validation warnings")
    for w in warnings:
        print(f"  - {w}")

if trades:
    trades_df = pd.DataFrame(trades)
    print("\n📑 Sample trades (first 20):")
    display(trades_df.head(20))
else:
    print("\n⚠️ No trades taken by this strategy in this re-run.")


📂 Loading SOFT top strategies from: /content/drive/MyDrive/XAUUSD_fulltest_batches/XAUUSD_top_strategies_AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE_EXACT_WF_SOFT.csv

🏆 Top 10 strategies:


,Net Profit,CAGR,Max Drawdown,Max Drawdown %,Volatility,Sharpe Ratio,Sortino Ratio,Win Rate,Profit Factor,# Trades,...,OOS Median Sharpe,OOS Worst Drawdown %,OOS Avg Profit Factor Capped,OOS Min Windows Required,OOS Min Trades Required,OOS Max DD Required,OOS Min Sharpe Required,OOS Min PF Required,OOS Pass Rule,Exact Config Forward Test
0,46.652210,1.040289,51.326927,0.050395,0.244595,3.045097,1.861604,0.428571,1.669758,7,...,3.045097,0.050395,1.669758,1.0,30.0,0.2,0.3,1.15,False,True
1,42.010643,0.903287,54.909576,0.053913,0.237517,2.835018,1.995044,0.428571,1.565443,7,...,2.835018,0.053913,1.565443,1.0,30.0,0.2,0.3,1.15,False,True
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,1.0,30.0,0.2,0.3,1.15,False,True
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,1.0,30.0,0.2,0.3,1.15,False,True
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,1.0,30.0,0.2,0.3,1.15,False,True
5,-57.782138,-0.338191,80.382893,0.080042,0.243080,-1.577075,-0.664080,0.450000,0.631648,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
6,-26.918729,-0.245297,28.550992,0.031864,0.121727,-2.215229,-0.814500,0.333333,0.314658,5,...,-2.215229,0.031864,0.314658,2.0,30.0,0.2,0.3,1.15,False,True
7,-44.553542,-0.353548,21.001458,0.035097,0.062321,-5.441082,-0.849740,0.500000,0.250695,4,...,-5.441082,0.035097,0.250695,2.0,30.0,0.2,0.3,1.15,False,True
8,-37.180581,-0.323272,20.091054,0.020091,0.077257,-5.872114,-0.594793,0.000000,0.000000,5,...,-5.872114,0.020091,0.000000,2.0,30.0,0.2,0.3,1.15,False,True
9,-48.667083,-0.401160,24.901667,0.028090,0.078065,-6.776950,-0.900160,0.000000,0.032238,5,...,-6.776950,0.028090,0.032238,2.0,30.0,0.2,0.3,1.15,False,True



⚠️ No row met AUTO_SELECT_RELIABLE criteria. Falling back to Rank 1.

⭐ Selected Strategy ID: AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE_SAMPLE_H1_831
🏅 Selected Rank      : 1
🕒 Timeframe          : H1

🔍 Found config in: /content/drive/MyDrive/XAUUSD_fulltest_batches/AUTO_SAMPLE_THEN_WF_20260504_1047_SAMPLE_EXACT_WF_H1/exact_candidates_CFGS.pkl

📋 Strategy details:
  • Indicators    : ('EMA', 'ATR', 'Fibonacci')
  • Strategy Name : AND_ATR_EMA_Fibonacci
  • Entry Logic   : {'mode': 'AND', 'min_confirmations': None}
  • Params        : {'Fibonacci': {'levels': [23.6, 38.2, 50, 61.8, 78.6]}, 'EMA': {'fast': 5, 'slow': 190}, 'ATR': {'period': 20, 'multiplier': 2.0, 'ma_period': 50}, 'MACD': {'signal': 9, 'fast': 12, 'slow': 26}, 'RSI': {'oversold': 30, 'overbought': 70, 'period': 14}, 'BollingerBands': {'mult': 2.0, 'period': 20}, 'Stochastic': {'d': 3, 'smooth': 3, 'oversold': 20, 'k': 14, 'overbought': 80}}
  • Exit          : {'sl_type': 'atr', 'tp_type': 'fib', 'atr_multiplier': 2.5, 

,entry,exit,entry_raw,exit_raw,entry_time,exit_time,signal_time,entry_idx,exit_idx,lot,direction,gross_pnl,entry_commission,exit_commission,swap_cash,pnl,bars,reason,stop_loss,take_profit
0,2912.99,2910.762550,2911.39,2910.862550,2025-03-05 17:00:00,2025-03-06 09:00:00,2025-03-05 16:00:00,130,145,0.01,long,-2.227450,0.035,0.035,-0.006250,-2.303700,15,Trailing_SL,2910.862550,3011.02462
1,2916.83,2902.424450,2915.23,2902.524450,2025-03-06 17:00:00,2025-03-07 02:00:00,2025-03-06 16:00:00,153,161,0.01,long,-14.405550,0.035,0.035,-0.003333,-14.478883,8,Trailing_SL,2902.524450,2979.23626
2,2911.13,2901.519000,2909.53,2901.619000,2025-03-07 23:00:00,2025-03-10 10:00:00,2025-03-07 22:00:00,182,193,0.01,long,-9.611000,0.035,0.035,-0.004583,-9.685583,11,Trailing_SL,2901.619000,2974.26436
3,2903.44,2899.082800,2903.54,2897.482800,2025-03-10 16:00:00,2025-03-11 04:00:00,2025-03-10 15:00:00,199,210,0.01,short,4.357200,0.035,0.035,-0.004583,4.282617,11,Trailing_SL,2898.982800,2841.38970
4,3042.50,3030.948600,3040.90,3031.048600,2025-03-20 15:00:00,2025-03-21 05:00:00,2025-03-20 14:00:00,382,395,0.01,long,-11.551400,0.035,0.035,-0.005417,-11.626817,13,Trailing_SL,3031.048600,3109.01598
5,3023.24,3013.496300,3021.64,3013.596300,2025-03-24 00:00:00,2025-03-24 16:00:00,2025-03-21 22:00:00,413,429,0.01,long,-9.743700,0.035,0.035,-0.006667,-9.820367,16,Trailing_SL,3013.596300,3117.19726
6,3020.20,3039.107600,3018.60,3039.207600,2025-03-26 20:00:00,2025-03-27 15:00:00,2025-03-26 19:00:00,479,497,0.01,long,18.907600,0.035,0.035,-0.007500,18.830100,18,Trailing_SL,3039.207600,3074.53244
7,3123.99,3115.473850,3122.39,3115.573850,2025-04-02 16:00:00,2025-04-02 23:00:00,2025-04-02 15:00:00,589,596,0.01,long,-8.516150,0.035,0.035,-0.002917,-8.589067,7,Trailing_SL,3115.573850,3203.43380
8,3142.62,3146.607400,3141.02,3146.707400,2025-04-03 01:00:00,2025-04-03 03:00:00,2025-04-02 23:00:00,597,599,0.01,long,3.987400,0.035,0.035,-0.000833,3.916567,2,Trailing_SL,3146.707400,3220.41344
9,3130.52,3099.553850,3128.92,3099.653850,2025-04-03 18:00:00,2025-04-04 05:00:00,2025-04-03 17:00:00,614,624,0.01,long,-30.966150,0.035,0.035,-0.004167,-31.040317,10,Trailing_SL,3099.653850,3314.53514
